In [ ]:
# -*- coding: utf-8 -*-
"""experiment_rag_v2

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1vYowV_H0PHjKktn2fqphP-dmEnegB_z-

# 🔧 PART 0: SETUP（環境設定）

System Version: **v2**

History:
- v1: 基礎 RAG（Dense + BM25 + RRF → LLM）
- v2: metadata 分層、priority weighting、Pre-LLM Gate

### 0.1 安裝套件


In [ ]:
!pip install pandas==2.2.2 numpy==2.0.2 pypdf langchain-text-splitters sentence-transformers faiss-cpu rank_bm25 google-generativeai huggingface_hub --upgrade

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.7/333.7 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 34.4 MB/s eta 0:00:00


In [ ]:
!pip install groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.7/139.7 kB 7.0 MB/s eta 0:00:00


### 0.2 Mount Google Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### 0.3 設定路徑與 API Key


In [ ]:
# --- 路徑 ---
INDEX_DIR = "/content/drive/MyDrive/AML/index_v2/v2/"
EXPERIMENT_ROOT_DIR = "/content/drive/MyDrive/AML/experiments/runs"

In [ ]:
# --- Embedding 模型（fallback，正常情況從 metadata.json 讀取）---
EMBEDDING_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

In [ ]:
# --- 檢索參數 ---
DEFAULT_TOP_K = 5
DEFAULT_RRF_K = 60

In [ ]:
# --- 功能開關 ---
DEFAULT_USE_PRIORITY_WEIGHTING = True
DEFAULT_ENABLE_GATE = True   # Gate 預設關閉，實驗時手動開啟

In [ ]:
# --- API Key ---
from google.colab import userdata
GROQ_API_KEY = userdata.get("colab-rag-llama3-dev")
GEMINI_API_KEY = userdata.get("aml-redflag-auditable-rag")
# HF_TOKEN = userdata.get('rag-demo-qwen-test')

In [ ]:
import os

# --- 驗證 ---
assert os.path.exists(INDEX_DIR), f"❌ 索引路徑不存在: {INDEX_DIR}"
print(f"✅ Config 載入完成")
print(f"   索引路徑: {INDEX_DIR}")
print(f"   Priority Weighting: {DEFAULT_USE_PRIORITY_WEIGHTING}")
print(f"   Gate: {DEFAULT_ENABLE_GATE}")

✅ Config 載入完成
   索引路徑: /content/drive/MyDrive/AML/index_v2/v2/
   Priority Weighting: True
   Gate: True


###0.4 LLM 模型選擇

In [ ]:
print("\n選擇 LLM 模型：")
print("  [1] Llama-3.3-70B (Groq) 🚀 — 最強推理，適合最終產出")
print("  [2] Gemini Flash — 平衡型")
print("  [3] Gemini Flash Lite — 快速輕量")
print("  [4] Llama-3.1-8B (Groq) ⚡ — 檢核工具：測試資料檢索精準度")
print("  [5] Mixtral-8x7b (Groq) 🧠 — 中堅實力：測試邏輯中轉")

MODEL_CHOICE = input("輸入 1/2/3/4/5: ").strip() or "1"

LLM_CONFIG = {
    "1": {
        "provider": "groq",
        "llm_model_name": "llama-3.3-70b-versatile",
        "api_key": GROQ_API_KEY,
    },
    "2": {
        "provider": "gemini",
        "llm_model_name": "gemini-2.0-flash",
        "api_key": GEMINI_API_KEY,
    },
    "3": {
        "provider": "gemini",
        "llm_model_name": "gemini-2.0-flash-lite",
        "api_key": GEMINI_API_KEY,
    },
    "4": {
        "provider": "groq",
        "llm_model_name": "llama-3.1-8b-instant",
        "api_key": GROQ_API_KEY,
    },
    "5": {
        "provider": "groq",
        "llm_model_name": "mixtral-8x7b-32768",
        "api_key": GROQ_API_KEY,
    },
}


選擇 LLM 模型：
  [1] Llama-3.3-70B (Groq) 🚀 — 最強推理，適合最終產出
  [2] Gemini Flash — 平衡型
  [3] Gemini Flash Lite — 快速輕量
  [4] Llama-3.1-8B (Groq) ⚡ — 檢核工具：測試資料檢索精準度
  [5] Mixtral-8x7b (Groq) 🧠 — 中堅實力：測試邏輯中轉
輸入 1/2/3/4/5: 4


In [ ]:
SELECTED_CONFIG = LLM_CONFIG[MODEL_CHOICE]
print(f"✅ 已選擇：{SELECTED_CONFIG['llm_model_name']}")

✅ 已選擇：llama-3.1-8b-instant


### 0.5 共用 Imports

In [ ]:
import json
import re
import pickle
import numpy as np
import faiss
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Any, Optional, Set
from enum import Enum
from dataclasses import dataclass, field

from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
import jieba
import google.generativeai as genai
from groq import Groq

print("✅ 所有 imports 完成")

/usr/local/lib/python3.12/dist-packages/jieba/__init__.py:44: SyntaxWarning: invalid escape sequence '\.'
  re_han_default = re.compile("([\u4E00-\u9FD5a-zA-Z0-9+#&\._%\-]+)", re.U)
/usr/local/lib/python3.12/dist-packages/jieba/__init__.py:46: SyntaxWarning: invalid escape sequence '\s'
  re_skip_default = re.compile("(\r\n|\s)", re.U)
/usr/local/lib/python3.12/dist-packages/jieba/finalseg/__init__.py:78: SyntaxWarning: invalid escape sequence '\.'
  re_skip = re.compile("([a-zA-Z0-9]+(?:\.\d+)?%?)")


✅ 所有 imports 完成


/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


### 0.6 📊 實驗記錄器

In [ ]:
"""### 0.6 ✏️ 實驗設定（每次實驗前改這格）"""

# ============================================
# ✏️ 每次實驗前只改這格
# ============================================

EXPERIMENT_VERSION = "v4.0"   # ← 版本號
EXPERIMENT_TYPE = "baseline"    # ← baseline / fix / tuning / ab_test / debug
EXPERIMENT_NOTE = ""    # ← 自由備註（會寫進目錄名和 metadata）
ENABLE_LOGGING = True   # ← 是否記錄到 Google Drive

# EXPERIMENT_VERSION = "v2.0"   # ← 版本號
# EXPERIMENT_TYPE = "baseline"    # ← baseline / fix / tuning / ab_test / debug
# EXPERIMENT_NOTE = ""    # ← 自由備註（會寫進目錄名和 metadata）
# ENABLE_LOGGING = True   # ← 是否記錄到 Google Drive

# 行為開關
USE_PRIORITY_WEIGHTING = True   # ← 是否啟用 priority weighting
ENABLE_GATE = True    # ← 是否啟用 Pre-LLM Gate

print(f"📋 實驗設定: {EXPERIMENT_VERSION} / {EXPERIMENT_TYPE}")
print(f"   Note: {EXPERIMENT_NOTE or '(none)'}")
print(f"   Priority={USE_PRIORITY_WEIGHTING} | Gate={ENABLE_GATE} | Log={ENABLE_LOGGING}")

📋 實驗設定: v4.0 / baseline
   Note: (none)
   Priority=True | Gate=True | Log=True


In [ ]:
"""### 0.7 📊 實驗記錄器"""

class ExperimentLogger:
    """RAG 實驗記錄器

    職責：把每次實驗的設定、結果、指標寫成 JSON 存到 Google Drive。
    不負責決定實驗參數（那是 Cell 0.6 的事）。
    """

    def __init__(
        self,
        base_dir: str = EXPERIMENT_ROOT_DIR,
        version: str = EXPERIMENT_VERSION,
        experiment_type: str = EXPERIMENT_TYPE,
        note: str = EXPERIMENT_NOTE,
    ):
        date_str = datetime.now().strftime("%Y%m%d")

        # 目錄名：v2.0_20250210_baseline_some_note
        exp_name = f"{version}_{date_str}_{experiment_type}"
        if note:
            exp_name += f"_{note}"

        self.exp_dir = Path(base_dir) / exp_name
        self.cases_dir = self.exp_dir / "cases"
        self.exp_dir.mkdir(parents=True, exist_ok=True)
        self.cases_dir.mkdir(exist_ok=True)

        self.metadata = {
            "experiment_id": exp_name,
            "version": version,
            "type": experiment_type,
            "note": note,
            "created_at": datetime.now().isoformat(),
            "status": "running",
        }

        print(f"📁 實驗目錄: {exp_name}")

    def log_config(self, config: Dict[str, Any]):
        """記錄配置"""
        with open(self.exp_dir / "config.json", "w", encoding="utf-8") as f:
            json.dump(config, f, indent=2, ensure_ascii=False)
        print(f"✅ 配置已保存")

    def log_case(
        self,
        case_id: str,
        scenario: str,
        expected: str,
        actual: str,
        passed: bool,
        retrieved_chunks: List[Dict] = None,
        llm_response: Dict = None,
        error: str = None,
    ):
        """記錄單個案例"""
        case_data = {
            "case_id": case_id,
            "scenario": scenario,
            "expected": expected,
            "actual": actual,
            "passed": passed,
            "timestamp": datetime.now().isoformat(),
        }
        if retrieved_chunks:
            case_data["retrieved_chunks"] = retrieved_chunks
        if llm_response:
            case_data["llm_response"] = llm_response
        if error:
            case_data["error"] = error

        status = "pass" if passed else "fail"
        with open(self.cases_dir / f"{case_id}_{expected}_{status}.json", "w", encoding="utf-8") as f:
            json.dump(case_data, f, indent=2, ensure_ascii=False)

        emoji = "✅" if passed else "❌"
        print(f"  {emoji} {case_id}: {expected} → {actual}")

    def log_batch_results(self, results: List[Dict[str, Any]]):
        """記錄批次結果"""
        with open(self.exp_dir / "results.json", "w", encoding="utf-8") as f:
            json.dump(results, f, indent=2, ensure_ascii=False)
        print(f"✅ 批次結果已保存")

    def log_metrics(self, metrics: Dict[str, Any]):
        """記錄指標"""
        with open(self.exp_dir / "metrics.json", "w", encoding="utf-8") as f:
            json.dump(metrics, f, indent=2, ensure_ascii=False)
        print(f"✅ 指標已保存")

    def finalize(self, summary: str = ""):
        """完成實驗"""
        self.metadata["status"] = "completed"
        self.metadata["completed_at"] = datetime.now().isoformat()
        if summary:
            self.metadata["summary"] = summary

        with open(self.exp_dir / "metadata.json", "w", encoding="utf-8") as f:
            json.dump(self.metadata, f, indent=2, ensure_ascii=False)

        print(f"\n🎉 實驗完成: {self.exp_dir.name}")
        print(f"📊 {summary}")

print("✅ ExperimentLogger 已定義")

✅ ExperimentLogger 已定義


In [ ]:
# class ExperimentLogger:
#     """RAG 實驗記錄器"""

#     def __init__(
#         self,
#         base_dir: str = EXPERIMENT_ROOT_DIR,
#         version: str = "v2.0",
#         experiment_type: str = "baseline",
#         description: str = "",
#         config: Dict[str, Any] = None,
#         auto_describe: bool = True,
#     ):
#         """初始化實驗記錄器"""
#         date_str = datetime.now().strftime("%Y%m%d")

#         if not description and auto_describe and config:
#             description = self._auto_generate_description(config, experiment_type)

#         exp_name = f"{version}_{date_str}_{experiment_type}"
#         if description:
#             exp_name += f"_{description}"

#         self.exp_dir = Path(base_dir) / exp_name
#         self.cases_dir = self.exp_dir / "cases"

#         self.exp_dir.mkdir(parents=True, exist_ok=True)
#         self.cases_dir.mkdir(exist_ok=True)

#         self.metadata = {
#             "experiment_id": exp_name,
#             "version": version,
#             "type": experiment_type,
#             "description": description,
#             "created_at": datetime.now().isoformat(),
#             "status": "running",
#         }

#         print(f"📁 實驗目錄: {exp_name}")

#     def _auto_generate_description(self, config: Dict[str, Any], exp_type: str) -> str:
#         """自動從配置生成描述"""
#         model_map = {
#             "llama-3.3-70b-versatile": "L70b",
#             "llama-3.1-8b-instant": "L8b",
#             "gemini-2.0-flash": "GemF",
#             "gemini-2.0-flash-lite": "GemFL",
#             "mixtral-8x7b-32768": "Mix8x7",
#         }
#         raw_name = config.get("llm_model_name", "unknown")
#         model_tag = model_map.get(raw_name, "other")

#         if exp_type == "baseline":
#             return model_tag
#         elif exp_type == "fix":
#             if config.get("use_priority_weighting"):
#                 return f"{model_tag}_priority_enabled"
#             return f"{model_tag}_bug_fix"
#         elif exp_type == "tuning":
#             changed = [model_tag]
#             if config.get("k", 5) != 5:
#                 changed.append(f"k{config['k']}")
#             return "_".join(changed[:3])
#         elif exp_type == "ab_test":
#             return f"{model_tag}_gate_comparison"
#         elif exp_type == "debug":
#             if "target_case" in config:
#                 return f"{model_tag}_case_{config['target_case']}"
#             return f"{model_tag}_investigation"
#         return model_tag

#     def log_config(self, config: Dict[str, Any]):
#         """記錄配置"""
#         config_path = self.exp_dir / "config.json"
#         with open(config_path, "w", encoding="utf-8") as f:
#             json.dump(config, f, indent=2, ensure_ascii=False)
#         print(f"✅ 配置已保存")

#     def log_case(
#         self,
#         case_id: str,
#         scenario: str,
#         expected: str,
#         actual: str,
#         passed: bool,
#         retrieved_chunks: List[Dict] = None,
#         llm_response: Dict = None,
#         error: str = None,
#     ):
#         """記錄單個案例"""
#         case_data = {
#             "case_id": case_id,
#             "scenario": scenario,
#             "expected": expected,
#             "actual": actual,
#             "passed": passed,
#             "timestamp": datetime.now().isoformat(),
#         }

#         if retrieved_chunks:
#             case_data["retrieved_chunks"] = retrieved_chunks
#         if llm_response:
#             case_data["llm_response"] = llm_response
#         if error:
#             case_data["error"] = error

#         status = "pass" if passed else "fail"
#         case_path = self.cases_dir / f"{case_id}_{expected}_{status}.json"

#         with open(case_path, "w", encoding="utf-8") as f:
#             json.dump(case_data, f, indent=2, ensure_ascii=False)

#         emoji = "✅" if passed else "❌"
#         print(f"  {emoji} {case_id}: {expected} → {actual}")

#     def log_batch_results(self, results: List[Dict[str, Any]]):
#         """記錄批次結果"""
#         results_path = self.exp_dir / "results.json"
#         with open(results_path, "w", encoding="utf-8") as f:
#             json.dump(results, f, indent=2, ensure_ascii=False)
#         print(f"✅ 批次結果已保存")

#     def log_metrics(self, metrics: Dict[str, Any]):
#         """記錄指標"""
#         metrics_path = self.exp_dir / "metrics.json"
#         with open(metrics_path, "w", encoding="utf-8") as f:
#             json.dump(metrics, f, indent=2, ensure_ascii=False)
#         print(f"✅ 指標已保存")

#     def finalize(self, summary: str = ""):
#         """完成實驗"""
#         self.metadata["status"] = "completed"
#         self.metadata["completed_at"] = datetime.now().isoformat()
#         if summary:
#             self.metadata["summary"] = summary

#         meta_path = self.exp_dir / "metadata.json"
#         with open(meta_path, "w", encoding="utf-8") as f:
#             json.dump(self.metadata, f, indent=2, ensure_ascii=False)

#         print(f"\n🎉 實驗完成: {self.exp_dir.name}")
#         print(f"📊 {summary}")

# print("✅ ExperimentLogger 已定義")

# 📋 PART 1: Pre-LLM Gate（知識範圍守門）

在 LLM 分析前進行 deterministic 判斷：
- 主題是否在知識範圍內？
- 是否有明確知識缺口？
- 證據是否足夠？

> Gate 是可開關的（`enable_gate` 參數），預設關閉。


In [ ]:
class KnowledgeManifest:
    """知識清單：定義系統知識範圍"""

    def __init__(self):
        self.covered_topics: Dict[str, str] = {
            "virtual_assets": "虛擬資產/加密貨幣相關紅旗",
            "cash_structuring": "現金拆分/門檻規避",
            "rapid_movement": "快速流轉/過路帳戶",
            "third_party": "第三人代辦/人頭帳戶",
            "cross_border": "跨境高風險地區",
            "identity_mismatch": "與身分/商業模式不符",
            "shell_company": "空殼公司/受益人不透明",
        }

        self.not_covered_topics: Dict[str, str] = {
            "TBML": "貿易型洗錢（Trade-Based Money Laundering）",
            "sanctions": "制裁名單篩選",
            "tax_evasion": "稅務逃漏",
        }

        self.required_evidence: Dict[str, List[str]] = {
            "TBML": ["invoice", "customs", "shipping", "goods_flow"],
        }

    def is_topic_covered(self, topic: str) -> bool:
        return topic.lower() in [t.lower() for t in self.covered_topics.keys()]

    def is_topic_explicitly_not_covered(self, topic: str) -> bool:
        return topic.upper() in [t.upper() for t in self.not_covered_topics.keys()]

    def get_required_evidence(self, topic: str) -> List[str]:
        return self.required_evidence.get(topic.upper(), [])

In [ ]:
class TopicDetector:
    """主題偵測器：rule-based 關鍵字匹配"""

    def __init__(self):
        self.topic_keywords: Dict[str, List[str]] = {
            "TBML": [
                "貿易型洗錢", "TBML", "報關", "報關單", "發票",
                "貨物", "貨物流向", "進出口", "國際貿易",
                "信用狀", "L/C", "提單", "invoice",
                "trade-based", "trade based", "customs", "shipping",
            ],
            "virtual_assets": [
                "虛擬資產", "加密貨幣", "比特幣", "以太幣",
                "混幣", "錢包", "交易所", "OTC", "私下交易",
                "非託管", "隱私幣",
            ],
            "cash_structuring": [
                "拆分", "門檻", "小額", "多筆", "申報",
            ],
            "rapid_movement": [
                "快速流轉", "入帳即轉", "過路帳戶", "多對手方",
                "很快轉出", "立即轉出", "轉出", "進出頻繁",   # 新增口語化描述
            ],
            "third_party": [
                "第三人", "代辦", "人頭", "代為操作",
            ],
            "cross_border": [
                "跨境", "境外", "高風險地區", "匯往", "匯款",
            ],
            "identity_mismatch": [
                "與身分不符", "與職業不符", "與業務不符",
                "業務無關", "不符", "無關", "無法說明",     # new
            ],
            "shell_company": [
                "空殼公司", "受益人不明", "股權結構", "不透明",
            ],
        }

    def detect_topics(self, text: str) -> Set[str]:
        detected = set()
        text_lower = text.lower()

        for topic, keywords in self.topic_keywords.items():
            for keyword in keywords:
                if keyword.lower() in text_lower:
                    detected.add(topic)
                    break

        return detected

    def detect_evidence_fields(self, text: str) -> Set[str]:
        evidence_keywords = {
            "invoice": ["發票", "invoice", "單據"],
            "customs": ["報關單", "報關", "customs", "海關"],
            "shipping": ["提單", "運送", "shipping", "物流"],
            "goods_flow": ["貨物流向", "貨物", "商品", "goods"],
        }

        detected = set()
        text_lower = text.lower()

        for field_name, keywords in evidence_keywords.items():
            for keyword in keywords:
                if keyword.lower() in text_lower:
                    detected.add(field_name)
                    break

        return detected

    def detect_explicit_knowledge_gap(self, text: str) -> bool:
        """偵測 scenario 是否明確表示知識缺口"""
        gap_patterns = [
            r"教材沒有",
            r"沒有.*章節",
            r"沒有.*資料",
            r"手上沒有",
            r"缺乏.*文件",
        ]

        for pattern in gap_patterns:
            if re.search(pattern, text):
                return True

        return False

In [ ]:
class GateDecision(Enum):
    ALLOW = "ALLOW"
    REFUSE = "REFUSE"

In [ ]:
@dataclass
class GateResult:
    decision: GateDecision
    reason_code: str = ""
    reason_message: str = ""
    metadata: Dict[str, Any] = field(default_factory=dict)

    def to_refuse_response(self) -> Dict[str, Any]:
        return {
            "scenario_summary": self.reason_message,
            "assessment": "refuse",
            "identified_flags": [],
            "follow_up_questions": [],
            "disclaimer": "本分析僅供教育參考，不構成法律意見。",
            "_gate_metadata": {
                "decision": self.decision.value,
                "reason_code": self.reason_code,
                "detected_topics": self.metadata.get("detected_topics", []),
            },
        }

In [ ]:
# ============================================================
# 🔬 Semantic Scope Classifier（語意範圍分類器）
# ============================================================

class SemanticScopeClassifier:
    """
    用 embedding 距離判斷 query 是否在 AML domain 內。
    """

    def __init__(
        self,
        embedding_model,
        threshold: float = 0.35,
    ):
        self.embedding_model = embedding_model
        self.threshold = threshold

        self.anchor_sentences: list = [
            "客戶使用加密貨幣進行大額交易",
            "虛擬資產交易所發現可疑的混幣行為",
            "Suspicious cryptocurrency transactions involving mixing services",
            "客戶分批存入現金，每次金額剛好低於申報門檻",
            "Multiple cash deposits structured to avoid reporting thresholds",
            "資金入帳後立即轉出到其他帳戶",
            "帳戶收到大額款項後迅速流轉至多個對手方",
            "第三人代為開戶並操作帳戶交易",
            "帳戶由他人代辦，本人無法說明交易目的",
            "資金頻繁匯往高風險國家或地區",
            "Cross-border wire transfers to jurisdictions with weak AML controls",
            "交易模式與客戶申報的職業和收入明顯不符",
            "帳戶交易量遠超其商業規模所能解釋的範圍",
            "公司受益人結構不透明，無法確認最終控制人",
            "Shell company with no apparent business operations receiving large transfers",
        ]

        self._anchor_embeddings = None

    def _ensure_anchors_encoded(self):
        if self._anchor_embeddings is None:
            self._anchor_embeddings = self.embedding_model.encode(
                self.anchor_sentences,
                normalize_embeddings=True,
                show_progress_bar=False,
            )

    def compute_scope_similarity(self, query: str) -> tuple:
        self._ensure_anchors_encoded()

        query_embedding = self.embedding_model.encode(
            [query],
            normalize_embeddings=True,
            show_progress_bar=False,
        )

        similarities = np.dot(self._anchor_embeddings, query_embedding.T).flatten()
        max_idx = int(np.argmax(similarities))
        max_sim = float(similarities[max_idx])
        nearest_anchor = self.anchor_sentences[max_idx]

        return max_sim, nearest_anchor

    def is_in_scope(self, query: str, verbose: bool = False) -> bool:
        max_sim, nearest = self.compute_scope_similarity(query)

        if verbose:
            status = "IN-SCOPE" if max_sim > self.threshold else "OUT-OF-SCOPE"
            print(f"   🔬 Semantic Scope: {status} (sim={max_sim:.3f}, τ={self.threshold})")

        return max_sim > self.threshold

print("✅ SemanticScopeClassifier 已定義")

✅ SemanticScopeClassifier 已定義


In [ ]:
def pre_llm_gate(
    scenario: str,
    manifest: KnowledgeManifest = None,
    detector: TopicDetector = None,
    scope_classifier: "SemanticScopeClassifier" = None,  # ← 加這一行
    verbose: bool = False,
) -> GateResult:
    """
    Pre-LLM Gate（Balanced v3）

    設計哲學：
    - Gate 只負責「災難級拒絕」
    - 灰階情境一律放行，但留下明確風險標記
    - 證據不足 → 警告（warning），不是 veto
    """

    if manifest is None:
        manifest = KnowledgeManifest()
    if detector is None:
        detector = TopicDetector()

    detected_topics = detector.detect_topics(scenario)

    # ----------------------------
    # Rule 1: 使用者明確表示沒有資料
    # ----------------------------
    if detector.detect_explicit_knowledge_gap(scenario):
        uncovered = [
            t for t in detected_topics
            if manifest.is_topic_explicitly_not_covered(t)
        ]
        if uncovered:
            if verbose:
                print(f"🚫 Gate: 明確知識缺口 — {uncovered}")
            return GateResult(
                decision=GateDecision.REFUSE,
                reason_code="EXPLICIT_KNOWLEDGE_GAP",
                reason_message=f"情境明確表示缺乏 {', '.join(uncovered)} 相關教材，無法進行分析",
                metadata={
                    "detected_topics": list(detected_topics),
                    "uncovered_topics": uncovered,
                },
            )

    # ----------------------------
    # 主題分類
    # ----------------------------
    covered_topics = [
        t for t in detected_topics if manifest.is_topic_covered(t)
    ]
    explicitly_not_covered = [
        t for t in detected_topics if manifest.is_topic_explicitly_not_covered(t)
    ]

    # ----------------------------
    # Rule 2: 完全超出系統範圍（真的沒救）
    # ----------------------------
    if explicitly_not_covered and not covered_topics:
        if verbose:
            print(f"🚫 Gate: 主題完全超出範圍 — {explicitly_not_covered}")
        return GateResult(
            decision=GateDecision.REFUSE,
            reason_code="TOPIC_NOT_COVERED",
            reason_message=f"本系統未包含 {', '.join(explicitly_not_covered)} 相關教材",
            metadata={
                "detected_topics": list(detected_topics),
                "explicitly_not_covered": explicitly_not_covered,
            },
        )

    # ----------------------------
    # Rule 3: 證據不足 → 風險標記（不直接拒絕）
    # ----------------------------
    evidence_warnings = []
    present_evidence = detector.detect_evidence_fields(scenario)

    for topic in detected_topics:
        required = manifest.get_required_evidence(topic)
        if not required:
            continue

        missing = set(required) - present_evidence
        missing_ratio = len(missing) / len(required)

        if missing_ratio > 0:
            evidence_warnings.append({
                "topic": topic,
                "missing_evidence": list(missing),
                "missing_ratio": round(missing_ratio, 2),
                "severity": (
                    "high" if missing_ratio > 0.5
                    else "medium" if missing_ratio > 0.25
                    else "low"
                )
            })

    if verbose and evidence_warnings:
        print(f"⚠️ Gate Warning: 證據不足標記 — {evidence_warnings}")



    if verbose and evidence_warnings:
        print(f"⚠️ Gate Warning: 證據不足標記 — {evidence_warnings}")

    # ----------------------------
    # Rule 4 (NEW): Semantic Out-of-Scope Detection
    # ----------------------------
    if not detected_topics and scope_classifier is not None:
        is_in = scope_classifier.is_in_scope(scenario, verbose=verbose)

        if not is_in:
            sim, nearest = scope_classifier.compute_scope_similarity(scenario)
            if verbose:
                print(f"🚫 Gate: 語意超出範圍 (sim={sim:.3f})")
            return GateResult(
                decision=GateDecision.REFUSE,
                reason_code="SEMANTIC_OUT_OF_SCOPE",
                reason_message="此問題不在本系統的知識範圍內（AML 紅旗偵測）",
                metadata={
                    "detected_topics": [],
                    "semantic_similarity": round(sim, 4),
                    "nearest_anchor": nearest[:80],
                    "threshold": scope_classifier.threshold,
                    "gate_mode": "balanced_v3.1_semantic",
                },
            )

    # ----------------------------
    # ALLOW（附帶風險地圖）
    # ----------------------------
    if verbose:
        print(f"✅ Gate: ALLOW — Covered topics: {covered_topics}")

    return GateResult(
        decision=GateDecision.ALLOW,
        metadata={
            "detected_topics": list(detected_topics),
            "covered_topics": covered_topics,
            "explicitly_not_covered": explicitly_not_covered,
            "evidence_warnings": evidence_warnings,
            "gate_mode": "balanced_v3"
        },
    )


In [ ]:
# old version
# def pre_llm_gate(
#     scenario: str,
#     manifest: KnowledgeManifest = None,
#     detector: TopicDetector = None,
#     verbose: bool = False,
# ) -> GateResult:
#     """
#     Pre-LLM Gate：在 LLM 分析前進行 deterministic 判斷

#     Rules:
#         1. 明確知識缺口 → REFUSE
#         2. 主題完全超出範圍 → REFUSE
#         3. 證據不足（特定主題）→ REFUSE
#         4. 其他 → ALLOW

#     Args:
#         scenario: 情境描述
#         manifest: 知識清單（None 時使用預設）
#         detector: 主題偵測器（None 時使用預設）
#         verbose: 是否顯示詳細過程

#     Returns:
#         GateResult: 包含 decision, reason_code, reason_message, metadata
#     """
#     if manifest is None:
#         manifest = KnowledgeManifest()
#     if detector is None:
#         detector = TopicDetector()

#     # Rule 1: 明確知識缺口
#     if detector.detect_explicit_knowledge_gap(scenario):
#         detected_topics = detector.detect_topics(scenario)
#         uncovered = [
#             t for t in detected_topics
#             if manifest.is_topic_explicitly_not_covered(t)
#         ]

#         if uncovered:
#             if verbose:
#                 print(f"🚫 Gate: 明確知識缺口 — {uncovered}")

#             return GateResult(
#                 decision=GateDecision.REFUSE,
#                 reason_code="EXPLICIT_KNOWLEDGE_GAP",
#                 reason_message=f"情境明確表示缺乏 {', '.join(uncovered)} 相關教材，無法進行分析",
#                 metadata={
#                     "detected_topics": list(detected_topics),
#                     "uncovered_topics": uncovered,
#                 },
#             )

#     # Rule 2: 主題超出範圍
#     detected_topics = detector.detect_topics(scenario)
#     explicitly_not_covered = [
#         t for t in detected_topics
#         if manifest.is_topic_explicitly_not_covered(t)
#     ]
#     covered_topics_detected = [
#         t for t in detected_topics
#         if manifest.is_topic_covered(t)
#     ]

#     if explicitly_not_covered and not covered_topics_detected:
#         if verbose:
#             print(f"🚫 Gate: 主題超出範圍 — {explicitly_not_covered}")

#         return GateResult(
#             decision=GateDecision.REFUSE,
#             reason_code="TOPIC_NOT_COVERED",
#             reason_message=f"本系統未包含 {', '.join(explicitly_not_covered)} 相關教材",
#             metadata={
#                 "detected_topics": list(detected_topics),
#                 "explicitly_not_covered": explicitly_not_covered,
#             },
#         )

#     # Rule 3: 證據不足
#     for topic in detected_topics:
#         required = manifest.get_required_evidence(topic)
#         if required:
#             present_evidence = detector.detect_evidence_fields(scenario)
#             missing = set(required) - present_evidence

#             if len(missing) > len(required) / 2:
#                 if verbose:
#                     print(f"🚫 Gate: 證據不足 — {topic}")

#                 return GateResult(
#                     decision=GateDecision.REFUSE,
#                     reason_code="INSUFFICIENT_EVIDENCE",
#                     reason_message=f"{topic} 分析需要更多證據資訊",
#                     metadata={
#                         "topic": topic,
#                         "missing_evidence": list(missing),
#                     },
#                 )

#     if verbose:
#         print(f"✅ Gate: ALLOW — {detected_topics}")

#     return GateResult(
#         decision=GateDecision.ALLOW,
#         metadata={"detected_topics": list(detected_topics)},
#     )


# print("✅ PART 1: Pre-LLM Gate 模組已載入")

# 🔍 PART 2: 檢索模組 (Retrieval Pipeline)

職責：Read-Only
- 載入已建立的索引
- 提供 Dense Search（向量檢索）
- 提供 BM25 Search（關鍵字檢索）
- 提供 Hybrid Search（混合檢索）

執行時機：
- 每次查詢時
- 不需要重建索引，只需載入
"""

### 2.0 載入索引

In [ ]:
def load_all_indexes(index_dir: str) -> dict:
    """
    載入所有索引和資料

    Args:
        index_dir: 索引存放路徑

    Returns:
        dict: faiss_index, chunks, bm25_index, tokenized_corpus,
              embedding_model, metadata

    Raises:
        FileNotFoundError: 如果索引檔案不存在
        ValueError: 如果 FAISS 向量數與 chunks 數不一致
    """
    base_path = Path(index_dir)

    required_files = [
        "faiss_index.bin", "chunks.json",
        "bm25_index.pkl", "tokenized_corpus.pkl", "metadata.json",
    ]

    missing_files = [f for f in required_files if not (base_path / f).exists()]
    if missing_files:
        raise FileNotFoundError(
            f"缺少索引檔案: {missing_files}\n"
            f"請先執行 build_data notebook 建立索引。"
        )

    print(f"📂 載入索引從: {index_dir}")

    # 1. Metadata（先載入以獲取模型名稱）
    with open(base_path / "metadata.json", "r", encoding="utf-8") as f:
        metadata = json.load(f)
    print(f"   ✅ metadata.json (version: {metadata.get('version', 'unknown')})")
    print(f"      建立時間: {metadata.get('created_at', 'unknown')}")
    print(f"      Chunk Size: {metadata.get('config', {}).get('chunk_size', 'unknown')}")

    # 2. FAISS Index
    faiss_index = faiss.read_index(str(base_path / "faiss_index.bin"))
    print(f"   ✅ faiss_index.bin ({faiss_index.ntotal} vectors, dim={faiss_index.d})")

    # 3. Chunks
    with open(base_path / "chunks.json", "r", encoding="utf-8") as f:
        chunks = json.load(f)
    print(f"   ✅ chunks.json ({len(chunks)} chunks)")

    # 4. BM25 Index
    with open(base_path / "bm25_index.pkl", "rb") as f:
        bm25_index = pickle.load(f)
    print(f"   ✅ bm25_index.pkl")

    # 5. Tokenized Corpus
    with open(base_path / "tokenized_corpus.pkl", "rb") as f:
        tokenized_corpus = pickle.load(f)
    print(f"   ✅ tokenized_corpus.pkl")

    # 6. Embedding Model（從 metadata 讀取模型名稱）
    # [BUG FIX] 原本讀 metadata.get("embedding_model_name")
    # 但 build_data 寫入的 key 是 config.embedding_model
    embedding_model_name = metadata.get("config", {}).get(
        "embedding_model", EMBEDDING_MODEL_NAME
    )
    embedding_model = SentenceTransformer(embedding_model_name)
    print(f"   ✅ embedding_model ({embedding_model_name})")

    # 驗證一致性
    assert faiss_index.ntotal == len(chunks), (
        f"❌ 資料不一致：FAISS 有 {faiss_index.ntotal} 個向量，"
        f"但 chunks 有 {len(chunks)} 個"
    )

    return {
        "faiss_index": faiss_index,
        "chunks": chunks,
        "bm25_index": bm25_index,
        "tokenized_corpus": tokenized_corpus,
        "embedding_model": embedding_model,
        "metadata": metadata,
    }

In [ ]:
# === 執行載入 ===
print("=" * 60)
print("🔍 PART 2: 檢索模組")
print("=" * 60)

loaded = load_all_indexes(INDEX_DIR)

🔍 PART 2: 檢索模組
📂 載入索引從: /content/drive/MyDrive/AML/index_v2/v2/
   ✅ metadata.json (version: v2)
      建立時間: 2026-02-10T07:20:24.023073
      Chunk Size: 400
   ✅ faiss_index.bin (226 vectors, dim=384)
   ✅ chunks.json (226 chunks)
   ✅ bm25_index.pkl
   ✅ tokenized_corpus.pkl


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

   ✅ embedding_model (sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2)


In [ ]:
import os

print(f"正在檢查目錄: {INDEX_DIR}")
if os.path.exists(INDEX_DIR):
    files = os.listdir(INDEX_DIR)
    if files:
        print("找到以下檔案：")
        for f in files:
            print(f"- {f}")
    else:
        print("目錄為空。")
else:
    print("目錄不存在。請確認 Google Drive 已正確掛載，且路徑正確。")

正在檢查目錄: /content/drive/MyDrive/AML/index_v2/v2/
找到以下檔案：
- faiss_index.bin
- chunks.json
- bm25_index.pkl
- tokenized_corpus.pkl
- metadata.json


In [ ]:
# 解包為模組級變數
faiss_index = loaded["faiss_index"]
chunks = loaded["chunks"]
bm25_index = loaded["bm25_index"]
tokenized_corpus = loaded["tokenized_corpus"]
embedding_model = loaded["embedding_model"]  # SentenceTransformer，用於向量檢索
index_metadata = loaded["metadata"]          # ← 改名避免跟其他 metadata 混淆

print(f"\n✅ 索引載入完成")
print(f"📦 索引版本: {index_metadata.get('version')} | chunks: {len(chunks)} | vectors: {faiss_index.ntotal}")

# ===== 🔬 初始化 SemanticScopeClassifier =====
scope_classifier = SemanticScopeClassifier(
    embedding_model=embedding_model,
    threshold=0.35,
)
print(f"✅ SemanticScopeClassifier 已初始化 (threshold={scope_classifier.threshold})")

# print(f"\n✅ 索引載入完成")
# print(f"📦 索引版本: {index_metadata.get('version')} | chunks: {len(chunks)} | vectors: {faiss_index.ntotal}")
# print("=" * 60)


✅ 索引載入完成
📦 索引版本: v2 | chunks: 226 | vectors: 226
✅ SemanticScopeClassifier 已初始化 (threshold=0.35)


### 2.1 Dense Search（向量搜尋）


In [ ]:
def dense_search(
    query: str,
    faiss_index: faiss.Index,
    chunks: list,
    embedding_model: SentenceTransformer,
    k: int = DEFAULT_TOP_K,
) -> list:
    """
    使用向量相似度進行搜尋

    Args:
        query: 查詢文字
        faiss_index: FAISS 向量索引
        chunks: chunk 列表
        embedding_model: SentenceTransformer 模型
        k: 返回的結果數量

    Returns:
        list[dict]: 每個 dict 包含 {"chunk": ChunkDict, "score": float}
    """
    query_embedding = embedding_model.encode(
        [query], normalize_embeddings=True
    )

    scores, indices = faiss_index.search(
        np.array(query_embedding, dtype="float32"), k
    )

    results = []
    for idx, score in zip(indices[0], scores[0]):
        if idx < len(chunks):  # 防禦性檢查
            results.append({
                "chunk": chunks[idx],
                "score": float(score),
            })

    return results

### 2.2 BM25 Search（關鍵字搜尋）


In [ ]:
def bm25_search(
    query: str,
    bm25_index,
    tokenized_corpus: list,
    chunks: list,
    k: int = DEFAULT_TOP_K,
) -> list:
    """
    使用 BM25 進行關鍵字搜尋

    Args:
        query: 查詢文字
        bm25_index: BM25Okapi 索引
        tokenized_corpus: 分詞後的語料
        chunks: chunk 列表
        k: 返回的結果數量

    Returns:
        list[dict]: 每個 dict 包含 {"chunk": ChunkDict, "score": float}
    """
    # 判斷查詢語言並分詞
    if any("\u4e00" <= char <= "\u9fff" for char in query):
        query_tokens = list(jieba.cut(query))
    else:
        query_tokens = query.lower().split()

    scores = bm25_index.get_scores(query_tokens)
    top_k_indices = np.argsort(scores)[::-1][:k]

    results = []
    for idx in top_k_indices:
        results.append({
            "chunk": chunks[idx],
            "score": float(scores[idx]),
        })

    return results

### 2.3 Hybrid Search（混合搜尋 + RRF）


In [ ]:
def hybrid_search(
    query: str,
    faiss_index: faiss.Index,
    chunks: list,
    embedding_model: SentenceTransformer,
    bm25_index,
    tokenized_corpus: list,
    k: int = DEFAULT_TOP_K,
    rrf_k: int = DEFAULT_RRF_K,
    use_priority_weighting: bool = DEFAULT_USE_PRIORITY_WEIGHTING,
) -> list:
    """
    混合搜尋：Dense + BM25，使用 RRF (Reciprocal Rank Fusion) 融合排名

    Args:
        query: 查詢文字
        faiss_index: FAISS 向量索引
        chunks: chunk 列表
        embedding_model: SentenceTransformer 模型
        bm25_index: BM25 索引
        tokenized_corpus: 分詞後的語料
        k: 返回的結果數量
        rrf_k: RRF 參數（通常為 60）
        use_priority_weighting: 是否使用 retrieval_priority 加權

    Returns:
        list[dict]: 每個 dict 包含 chunk, score, raw_rrf_score, priority_weight

    History:
        - v1: 基礎 RRF
        - v2: 加入 priority_weighting（用 metadata 的 retrieval_priority 加權）
    """
    # 取得兩種搜尋結果（多取以確保融合效果）
    dense_results = dense_search(
        query, faiss_index, chunks, embedding_model, k=k * 2
    )
    bm25_results = bm25_search(
        query, bm25_index, tokenized_corpus, chunks, k=k * 2
    )

    # 建立 chunk_id → 排名
    dense_ranks = {}
    for rank, r in enumerate(dense_results, start=1):
        chunk_id = r["chunk"]["chunk_id"]
        dense_ranks[chunk_id] = rank

    bm25_ranks = {}
    for rank, r in enumerate(bm25_results, start=1):
        chunk_id = r["chunk"]["chunk_id"]
        bm25_ranks[chunk_id] = rank

    # RRF 融合
    all_chunk_ids = set(dense_ranks.keys()) | set(bm25_ranks.keys())
    rrf_scores = {}

    for chunk_id in all_chunk_ids:
        score = 0
        if chunk_id in dense_ranks:
            score += 1 / (rrf_k + dense_ranks[chunk_id])
        if chunk_id in bm25_ranks:
            score += 1 / (rrf_k + bm25_ranks[chunk_id])
        rrf_scores[chunk_id] = score

    # 權重調整
    chunk_lookup = {c["chunk_id"]: c for c in chunks}

    if use_priority_weighting:
        weighted_scores = {}
        for chunk_id, rrf_score in rrf_scores.items():
            chunk = chunk_lookup[chunk_id]
            priority = chunk.get("retrieval_priority", 1.0)
            weighted_scores[chunk_id] = rrf_score * priority
    else:
        weighted_scores = rrf_scores

    # 排序取 top-k
    sorted_chunk_ids = sorted(
        weighted_scores.keys(),
        key=lambda x: weighted_scores[x],
        reverse=True,
    )[:k]

    # 組裝結果
    results = []
    for chunk_id in sorted_chunk_ids:
        results.append({
            "chunk": chunk_lookup[chunk_id],
            "score": weighted_scores[chunk_id],
            "raw_rrf_score": rrf_scores[chunk_id],
            "priority_weight": chunk_lookup[chunk_id].get("retrieval_priority", 1.0),
        })

    return results

### 2.4 便捷函數 retrieve_contexts()

取得上下文（供 PART 3 使用）

In [ ]:
def retrieve_contexts(
    query: str,
    faiss_index: faiss.Index,
    chunks: list,
    embedding_model: SentenceTransformer,
    bm25_index,
    tokenized_corpus: list,
    k: int = DEFAULT_TOP_K,
    method: str = "hybrid",
    use_priority_weighting: bool = DEFAULT_USE_PRIORITY_WEIGHTING,
) -> list:
    """
    檢索相關上下文，供 LLM 分析使用。
    這是 PART 2 提供給 PART 3 的主要介面。

    Args:
        query: 查詢文字（通常是情境描述）
        faiss_index: FAISS 向量索引
        chunks: chunk 列表
        embedding_model: SentenceTransformer 模型
        bm25_index: BM25 索引
        tokenized_corpus: 分詞後的語料
        k: 返回的結果數量
        method: 檢索方法 ("dense", "bm25", "hybrid")
        use_priority_weighting: 是否使用 retrieval_priority 加權

    Returns:
        list[dict]: 格式化的上下文列表
    """
    if method == "dense":
        search_results = dense_search(
            query, faiss_index, chunks, embedding_model, k
        )
    elif method == "bm25":
        search_results = bm25_search(
            query, bm25_index, tokenized_corpus, chunks, k
        )
    else:  # hybrid（預設）
        search_results = hybrid_search(
            query, faiss_index, chunks, embedding_model,
            bm25_index, tokenized_corpus, k,
            use_priority_weighting=use_priority_weighting,  # ← [BUG FIX] 原本漏傳
        )

    # 轉換為 ContextDict 格式
    contexts = []
    for r in search_results:
        chunk = r["chunk"]
        contexts.append({
            "chunk_id": chunk.get("chunk_id", ""),
            "source": chunk.get("source", "Unknown"),
            "page": chunk.get("page", 0),
            "text": chunk.get("text", ""),
            "score": r["score"],
            "doc_category": chunk.get("doc_category", "unknown"),
            "explanation_style": chunk.get("explanation_style", "neutral"),
        })

    return contexts

#🔍 PART 2: 檢索測試

### 2.5 【測試】快速驗證

In [ ]:
# 測試查詢  # 資料科學/實驗者 thinking
test_query = "什麼是洗錢？"

print(f"\n🧪 測試查詢: {test_query}")
print("-" * 40)

# Dense Search
print("\n📊 Dense Search 結果:")
dense_results = dense_search(
    test_query, faiss_index, chunks, embedding_model, k=3
)
for i, r in enumerate(dense_results, 1):
    print(f"   [{i}] {r['chunk']['source']} p.{r['chunk']['page']} "
          f"(score: {r['score']:.3f})")

# BM25 Search
print("\n📊 BM25 Search 結果:")
bm25_results = bm25_search(
    test_query, bm25_index, tokenized_corpus, chunks, k=3
)
for i, r in enumerate(bm25_results, 1):
    print(f"   [{i}] {r['chunk']['source']} p.{r['chunk']['page']} "
          f"(score: {r['score']:.3f})")

# Hybrid Search
print("\n📊 Hybrid Search 結果:")
hybrid_results = hybrid_search(
    test_query, faiss_index, chunks, embedding_model,
    bm25_index, tokenized_corpus, k=3
)
for i, r in enumerate(hybrid_results, 1):
    print(f"   [{i}] {r['chunk']['source']} p.{r['chunk']['page']} "
          f"(score: {r['score']:.3f})")

print(f"   [{i}] {r['chunk']['source']} p.{r['chunk']['page']} "
      f"(Score: {r['score']:.3f} | Raw: {r['raw_rrf_score']:.3f} | W: {r['priority_weight']})")
print("\n✅ PART 2 測試完成！")


🧪 測試查詢: 什麼是洗錢？
----------------------------------------

📊 Dense Search 結果:


NameError: name 'dense_search' is not defined

In [ ]:
# # 開發者thinking
# if __name__ == "__main__":
#     # 測試查詢
#     test_query = "什麼是洗錢？"

#     print(f"\n🧪 測試查詢: {test_query}")
#     print("-" * 40)

#     # Dense Search
#     print("\n📊 Dense Search 結果:")
#     dense_results = dense_search(
#         test_query, faiss_index, chunks, embedding_model, k=3
#     )
#     for i, r in enumerate(dense_results, 1):
#         print(f"   [{i}] {r['chunk']['source']} p.{r['chunk']['page']} "
#               f"(score: {r['score']:.3f})")

#     # BM25 Search
#     print("\n📊 BM25 Search 結果:")
#     bm25_results = bm25_search(
#         test_query, bm25_index, tokenized_corpus, chunks, k=3
#     )
#     for i, r in enumerate(bm25_results, 1):
#         print(f"   [{i}] {r['chunk']['source']} p.{r['chunk']['page']} "
#               f"(score: {r['score']:.3f})")

#     # Hybrid Search
#     print("\n📊 Hybrid Search 結果:")
#     hybrid_results = hybrid_search(
#         test_query, faiss_index, chunks, embedding_model,
#         bm25_index, tokenized_corpus, k=3
#     )
#     for i, r in enumerate(hybrid_results, 1):
#         print(f"   [{i}] {r['chunk']['source']} p.{r['chunk']['page']} "
#               f"(score: {r['score']:.3f})")

#     print("\n✅ PART 2 測試完成！")

這 12 題是測試「檢索」功能的——能不能找到正確的頁面？

In [ ]:
RETRIEVAL_TEST_CASES = [
    # Easy
    {"id": "E1", "query": "什麼是洗錢？", "expected_page": 2, "difficulty": "easy"},
    {"id": "E2", "query": "洗錢的三種態樣是什麼？", "expected_page": 2, "difficulty": "easy"},
    {"id": "E3", "query": "銀行現金交易多少錢以上要申報？", "expected_page": 8, "difficulty": "easy"},

    # Medium
    {"id": "M1", "query": "把大筆現金拆成小筆存入銀行會有什麼法律問題？", "expected_page": 8, "difficulty": "medium"},
    {"id": "M2", "query": "開公司戶頭要帶什麼文件？", "expected_page": 5, "difficulty": "medium"},
    {"id": "M3", "query": "銀行什麼情況下會拒絕讓我開戶？", "expected_page": 6, "difficulty": "medium"},
    {"id": "M4", "query": "為什麼金融機構要做客戶審查？", "expected_page": 4, "difficulty": "medium"},

    # Hard
    {"id": "H1", "query": "學生把帳戶賣給別人會怎樣？", "expected_page": 9, "difficulty": "hard"},
    {"id": "H2", "query": "出國可以帶多少現金？超過怎麼辦？", "expected_page": 7, "difficulty": "hard"},
    {"id": "H3", "query": "高風險地區的客戶開戶會被怎麼對待？", "expected_page": 4, "difficulty": "hard"},

    # Edge case（文件中沒有的）
    {"id": "X1", "query": "用比特幣洗錢會怎樣？", "expected_page": None, "difficulty": "edge"},
    {"id": "X2", "query": "洗錢防制法是哪一年制定的？", "expected_page": None, "difficulty": "edge"},
]

In [ ]:
def evaluate_retrieval(search_func, test_cases, k=3):
    """評估 retrieval 準確度"""
    results = {"recall@1": 0, "recall@3": 0, "details": []}
    valid_cases = [tc for tc in test_cases if tc["expected_page"] is not None]

    for tc in valid_cases:
        search_results = search_func(tc["query"])
        retrieved_pages = [r["chunk"]["page"] for r in search_results[:k]]

        hit_at_1 = retrieved_pages[0] == tc["expected_page"] if retrieved_pages else False
        hit_at_3 = tc["expected_page"] in retrieved_pages[:3]

        if hit_at_1:
            results["recall@1"] += 1
        if hit_at_3:
            results["recall@3"] += 1

        results["details"].append({
            "id": tc["id"],
            "query": tc["query"],
            "expected": tc["expected_page"],
            "got": retrieved_pages[0] if retrieved_pages else None,
            "hit": "✅" if hit_at_1 else "❌"
        })

    n = len(valid_cases)
    print(f"Recall@1: {results['recall@1']}/{n} ({results['recall@1']/n*100:.0f}%)")
    print(f"Recall@3: {results['recall@3']}/{n} ({results['recall@3']/n*100:.0f}%)")

    return results

In [ ]:
# 執行評估
search_func = lambda q: hybrid_search(q, faiss_index, chunks, embedding_model, bm25_index, tokenized_corpus)
evaluate_retrieval(search_func, RETRIEVAL_TEST_CASES)

### 2.6 Retrieval-Only 評估 📊

目的：隔離搜索品質，不經過 LLM
- 載入標註好的 query-relevance dataset
- 對每個 query 跑 dense / bm25 / hybrid search
- 計算 Precision@K, Recall@K, MRR
- 記錄 raw scores（為 Day 3 的分布圖做準備）

#### Config — 改這裡就好，下面全部自動

In [ ]:
# ⬇️ 每次換 test set，只改這一行 ⬇️
TEST_SET_NAME = "scenario_queries"

# # 合法的 test set 名稱（防打錯字）
# VALID_TEST_SETS = {
#     "basic_50",        # 50 個精確/概念/數據/術語 retrieval query
#     "scenario_20",     # 20 個場景描述 query
#     "list_15",         # 15 個列表型 query
#     "cross_chunk_10",  # 10 個跨 chunk query
# }

# assert TEST_SET_NAME in VALID_TEST_SETS, (
#     f"❌ 不認識的 test set: '{TEST_SET_NAME}'\n"
#     f"   合法選項: {sorted(VALID_TEST_SETS)}"
# )

# 路徑自動生成
EVAL_DIR = "/content/drive/MyDrive/AML/eval"
ANNOTATED_PATH = f"{EVAL_DIR}/{TEST_SET_NAME}_annotated.json"
EVAL_RESULT_PATH = f"{EVAL_DIR}/{TEST_SET_NAME}_eval_result.json"

os.makedirs(EVAL_DIR, exist_ok=True)

print(f"📋 Test Set: {TEST_SET_NAME}")
print(f"   標註檔: {ANNOTATED_PATH}")
print(f"   結果檔: {EVAL_RESULT_PATH}")

In [ ]:
# 準備 Query-Relevance 標註資料
# === 標註輔助：對每個 query 跑搜索，看 top-5 結果 ===

def annotate_helper(query: str, k: int = 5):
    """
    標註輔助工具：
    給一個 query，印出 dense / bm25 / hybrid 的 top-k 結果。
    你看完後，人工決定哪些 chunk_id 是 relevant 的。
    """
    print(f"=" * 70)
    print(f"🔍 Query: {query}")
    print(f"=" * 70)

    # Dense search
    dense_results = dense_search(
        query, faiss_index, chunks, embedding_model, k=k
    )
    print(f"\n--- Dense Search (top-{k}) ---")
    for i, r in enumerate(dense_results, 1):
        c = r["chunk"]
        print(f"  [{i}] {c['chunk_id']} (score={r['score']:.4f})")
        print(f"      {c['text'][:120]}...")
        print()

    # BM25 search
    bm25_results = bm25_search(
        query, bm25_index, tokenized_corpus, chunks, k=k
    )
    print(f"\n--- BM25 Search (top-{k}) ---")
    for i, r in enumerate(bm25_results, 1):
        c = r["chunk"]
        print(f"  [{i}] {c['chunk_id']} (score={r['score']:.4f})")
        print(f"      {c['text'][:120]}...")
        print()

    # Hybrid search
    hybrid_results = hybrid_search(
        query, faiss_index, chunks, embedding_model,
        bm25_index, tokenized_corpus, k=k
    )
    print(f"\n--- Hybrid Search (top-{k}) ---")
    for i, r in enumerate(hybrid_results, 1):
        c = r["chunk"]
        print(f"  [{i}] {c['chunk_id']} (score={r['score']:.4f})")
        print(f"      {c['text'][:120]}...")
        print()

#### 查詢

In [ ]:
# === 對第一批 query 做標註 ===
# 一次看一個，決定哪些 chunk 是 relevant

# annotate_helper("FATF 的全稱是什麼？")
annotate_helper("某人把一件有完整公開來歷的東西，換成一件來歷在結構上就無從查證的東西——這個轉換行為，本身說明了什麼？")

In [ ]:
# annotate_batch(annotated_queries, start=18, count=3)

#### 載入 / 建立標註資料

In [ ]:
# === query_relevance_dataset.json ===
# 這就是你的 ground truth
# strict relevance

scenario_queries = [
    {
        "query_id": "S01",
        "query": "一家汽車經銷商卻持續出口大量紡織品到海外，這種買賣項目和公司登記業務完全對不上的情況，是否值得關注？",
        "category": "場景描述",
        "relevant_chunks": [
            "fatf_tbm_laundering_red_flags.pdf_p6_c0"
            ],
        "expected_document": "fatf_tbm_laundering_red_flags.pdf",
        "lexical_overlap_risk": "high",
        "semantic_distance_level": 1,
        "semantic_shift_note": None
    },
    {
        "query_id": "S02",
        "query": "某間公司的組織架構異常複雜，中間層層疊了好幾家在監管寬鬆地區註冊的空殼法人，而且看不出實際業務邏輯，這種組織設計有什麼問題？",
        "category": "場景描述",
        "relevant_chunks": [
            "fatf_tbm_laundering_red_flags.pdf_p5_c0"
            ],
        "expected_document": "fatf_tbm_laundering_red_flags.pdf",
        "lexical_overlap_risk": "high",
        "semantic_distance_level": 1,
        "semantic_shift_note": None
    },
    {
        "query_id": "S03",
        "query": "賣方開出來的帳單上寫的錢和雙方本來談好的價錢差了一大截，連運過來的東西數量和品質也跟紙上寫的對不上，這種帳面和實物兜不攏的狀況，算是什麼問題？",
        "category": "去關鍵字/語義轉換",
        "relevant_chunks": [
            "fatf_tbm_laundering_red_flags.pdf_p7_c1"
            ],
        "expected_document": "fatf_tbm_laundering_red_flags.pdf",
        "lexical_overlap_risk": "medium",
        "semantic_distance_level": 2,
        "semantic_shift_note": "移除「發票」「合約」「進出口」「貨物」等詞。核心：文件與交付不一致。"
    },
    {
        "query_id": "S04",
        "query": "某貿易商的帳戶看起來只是資金的中繼站——每天進來大量匯款、快速轉走，日終餘額幾乎為零，而且找不到合理的商業解釋，這類帳戶有什麼特徵？",
        "category": "場景描述",
        "relevant_chunks": [
            "fatf_tbm_laundering_red_flags.pdf_p8_c1"
        ],
        "expected_document": "fatf_tbm_laundering_red_flags.pdf",
        "lexical_overlap_risk": "high",
        "semantic_distance_level": 1,
        "semantic_shift_note": None
    },
    {
        "query_id": "S05",
        "query": "某企業進口貨物的金額，和它對外匯出用於支付進口的銀行轉帳金額有很大落差，這種數字上的不一致意味著什麼？",
        "category": "場景描述",
        "relevant_chunks": [
            "fatf_tbm_laundering_red_flags.pdf_p7_c4"
        ],
        "expected_document": "fatf_tbm_laundering_red_flags.pdf",
        "lexical_overlap_risk": "high",
        "semantic_distance_level": 1,
        "semantic_shift_note": None
    },
    {
        "query_id": "S06",
        "query": "一家公司的註冊地址是一棟住宅大樓的信箱地址，而且沒有標明具體的單位號碼，也沒有任何商業或工業空間，這樣的登記地點為什麼引人懷疑？",
        "category": "場景描述",
        "relevant_chunks": [
            "fatf_tbm_laundering_red_flags.pdf_p5_c1"
        ],
        "expected_document": "fatf_tbm_laundering_red_flags.pdf",
        "lexical_overlap_risk": "high",
        "semantic_distance_level": 1,
        "semantic_shift_note": None
    },
    {
        "query_id": "S07",
        "query": "一段已經確認的約定，在即將兌現的節點上被替換成一個陌生的版本——這種「最後一刻的替換」在交易情境中，通常暗示了什麼？",
        "category": "去關鍵字/語義轉換",
        "relevant_chunks": [
            "fatf_tbm_laundering_red_flags.pdf_p8_c0"
        ],
        "expected_document": "fatf_tbm_laundering_red_flags.pdf",
        "lexical_overlap_risk": "low",
        "semantic_distance_level": 3,
        "semantic_shift_note": "將「付款流程被篡改」抽象化為「約定在節點被替換」，去金融術語。"
    },
    {
        "query_id": "S08",
        "query": "某人在數位貨幣交易平台開戶時存入了一大筆資金，金額與其個人背景和職業收入完全不相稱，而且第一天就開始大量進出交易，這種新用戶行為有什麼可疑之處？",
        "category": "場景描述",
        "relevant_chunks": [
            "fatf_virtual_assets_red_flags.pdf_p16_c3"
        ],
        "expected_document": "fatf_virtual_assets_red_flags.pdf",
        "lexical_overlap_risk": "high",
        "semantic_distance_level": 1,
        "semantic_shift_note": None
    },
    {
        "query_id": "S09",
        "query": "某人把一件有完整公開來歷的東西，換成一件來歷在結構上就無從查證的東西——這個轉換行為，本身說明了什麼？",
        "category": "去關鍵字/語義轉換",
        "relevant_chunks": [
            "fatf_virtual_assets_red_flags.pdf_p11_c6",
            "tw_aml_training_slides.pdf_p3_c0"
        ],
        "expected_document": "fatf_virtual_assets_red_flags.pdf",
        "lexical_overlap_risk": "low",
        "semantic_distance_level": 3,
        "semantic_shift_note": "保留「可追溯→不可追溯」結構，改為抽象物件隱喻。"
    },
    {
        "query_id": "S10",
        "query": "有人透過暗網關聯的 IP 位址或加密通訊工具登入數位資產服務平台進行交易，這種連線方式為什麼會被視為異常？",
        "category": "場景描述",
        "relevant_chunks": [
            "fatf_virtual_assets_red_flags.pdf_p12_c3"
        ],
        "expected_document": "fatf_virtual_assets_red_flags.pdf",
        "lexical_overlap_risk": "high",
        "semantic_distance_level": 1,
        "semantic_shift_note": None
    },
    {
        "query_id": "S11",
        "query": "客戶的資金來源直接關聯到已知的網路勒索事件地址，或是來自地下交易市場的錢包，這種資金流向為什麼需要特別警覺？",
        "category": "場景描述",
        "relevant_chunks": [
            "fatf_virtual_assets_red_flags.pdf__p17_c4",
            "fatf_virtual_assets_red_flags.pdf_p19_c5"
        ],
        "expected_document": "fatf_virtual_assets_red_flags.pdf",
        "lexical_overlap_risk": "high",
        "semantic_distance_level": 1,
        "semantic_shift_note": None
    },
    {
        "query_id": "S12",
        "query": "一位用戶註冊數位資產帳號時提交了經過修改的身分證照片，而且在被要求補充資金來源說明時拒絕回答，這在開戶審查階段代表什麼？",
        "category": "場景描述",
        "relevant_chunks": [
            "fatf_virtual_assets_red_flags.pdf_p14_c7",
            "fatf_virtual_assets_red_flags.pdf_p14_c8"
        ],
        "expected_document": "fatf_virtual_assets_red_flags.pdf",
        "lexical_overlap_risk": "high",
        "semantic_distance_level": 1,
        "semantic_shift_note": None
    },
    {
        "query_id": "S13",
        "query": "某客戶習慣把資金匯到一個對數位資產完全沒有監管規範的國家的交易平台，而且那個平台也沒有落實任何身分驗證措施，這和地理風險有什麼關係？",
        "category": "場景描述",
        "relevant_chunks": [
            "fatf_virtual_assets_red_flags.pdf_p19_c6"
        ],
        "expected_document": "fatf_virtual_assets_red_flags.pdf",
        "lexical_overlap_risk": "high",
        "semantic_distance_level": 1,
        "semantic_shift_note": None
    },
    {
        "query_id": "S14",
        "query": "某平台發現有大量看似互不相關的數位錢包，其實全部是從同一個網路位址操控的，這可能代表什麼手法？",
        "category": "場景描述",
        "relevant_chunks": [
            "fatf_virtual_assets_red_flags.pdf_p12_c4"
        ],
        "expected_document": "fatf_virtual_assets_red_flags.pdf",
        "lexical_overlap_risk": "high",
        "semantic_distance_level": 1,
        "semantic_shift_note": None
    },
    {
        "query_id": "S15",
        "query": "有人先把一批說不清楚哪裡來的財物，委託多人分批代為處理，途中幾度易手，最後以置產或添購名貴物件的方式讓那筆財物看起來出自正當來源——請問法規上用什麼框架來描述這整個過程？",
        "category": "去關鍵字/語義轉換",
        "relevant_chunks": [
            "tw_aml_training_slides.pdf_p2_c0"
        ],
        "expected_document": "tw_aml_training_slides.pdf",
        "lexical_overlap_risk": "medium",
        "semantic_distance_level": 2,
        "semantic_shift_note": "轉為記者敘事視角；全面換詞（財物、易手、正當來源），保留洗錢三階段結構。"
    },
    {
        "query_id": "S16",
        "query": "銀行在什麼情況下會直接拒絕讓客戶開戶或進行交易？例如發現客戶疑似使用假名或者拿偽造的證件來辦理，還有哪些類似的拒絕情形？",
        "category": "場景描述",
        "relevant_chunks": [
            "tw_aml_training_slides.pdf_p6_c0"
        ],
        "expected_document": "tw_aml_training_slides.pdf",
        "lexical_overlap_risk": "high",
        "semantic_distance_level": 1,
        "semantic_shift_note": None
    },
    {
        "query_id": "S17",
        "query": "如果有人在報紙上看到「高價收購銀行帳戶」的廣告，把自己的金融卡和存摺交給對方，即使只是為了賺取小利，這樣做在法律上會面臨什麼後果？",
        "category": "場景描述",
        "relevant_chunks": [
            "tw_aml_training_slides.pdf_p9_c0"
        ],
        "expected_document": "tw_aml_training_slides.pdf",
        "lexical_overlap_risk": "high",
        "semantic_distance_level": 1,
        "semantic_shift_note": None
    },
    {
        "query_id": "S18",
        "query": "主管交代員工把一大筆來路不明的現金拆成好幾份、每份都控制在法定申報金額以下分批存入銀行，這樣刻意規避通報機制的行為會觸犯什麼罪？",
        "category": "場景描述",
        "relevant_chunks": [
            "tw_aml_training_slides.pdf_p8_c0"
        ],
        "expected_document": "tw_aml_training_slides.pdf",
        "lexical_overlap_risk": "high",
        "semantic_distance_level": 1,
        "semantic_shift_note": None
    },
    {
        "query_id": "S19",
        "query": "出國旅遊時，除了各國貨幣現鈔和黃金之外，還有哪些可能被利用於資金非法移動的物品也需要在邊境向海關申報？如果沒有誠實申報會怎樣？",
        "category": "場景描述",
        "relevant_chunks": [
            "tw_aml_training_slides.pdf_p7_c0"
        ],
        "expected_document": "tw_aml_training_slides.pdf",
        "lexical_overlap_risk": "high",
        "semantic_distance_level": 1,
        "semantic_shift_note": None
    },
    {
        "query_id": "S20",
        "query": "要在銀行以公司名義開戶時，除了營業登記等基本文件之外，銀行還會要求提供哪些關於公司背後實際控制人的身分資訊？",
        "category": "場景描述",
        "relevant_chunks": [
            "tw_aml_training_slides.pdf_p5_c0"
        ],
        "expected_document": "tw_aml_training_slides.pdf",
        "lexical_overlap_risk": "high",
        "semantic_distance_level": 1,
        "semantic_shift_note": None
    }
]

# 存檔
with open(ANNOTATED_PATH, "w", encoding="utf-8") as f:
    json.dump(scenario_queries, f, ensure_ascii=False, indent=2)
print(f"✅ 已儲存 {len(scenario_queries)} 個標註到 {ANNOTATED_PATH}")

NameError: name 'ANNOTATED_PATH' is not defined

In [ ]:
def load_or_create_annotated(path: str, test_set_name: str) -> list:
    """
    如果標註檔已存在 → 載入
    如果不存在 → 提示你要先建立

    為什麼分開？
    - 標註是「人工 + 輔助」的過程，不是一鍵完成的
    - 這個函式只負責「讀取已經標好的資料」
    """
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        print(f"✅ 載入 {len(data)} 個已標註 query（從 {path}）")

        # 檢查有多少已完成標註（relevant_chunks 非空）
        done = sum(1 for q in data if q.get("relevant_chunks"))
        todo = len(data) - done
        if todo > 0:
            print(f"   ⚠️ 其中 {todo} 個尚未標註 relevant_chunks")
        return data
    else:
        print(f"⚠️ 標註檔不存在: {path}")
        print(f"   請先執行以下步驟：")
        print(f"   1. 準備你的 {test_set_name} 題目（統一格式，含 relevant_chunks: [] 欄位）")
        print(f"   2. 用 annotate_helper() 輔助標註")
        print(f"   3. 用 save_annotated() 存檔")
        return []


def save_annotated(data: list, path: str = None):
    """
    儲存標註資料

    Args:
        data: 標註好的 query list
        path: 預設使用 ANNOTATED_PATH（由 TEST_SET_NAME 決定）
    """
    if path is None:
        path = ANNOTATED_PATH
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    done = sum(1 for q in data if q.get("relevant_chunks"))
    print(f"✅ 已儲存 {len(data)} 個 query（{done} 個已標註）→ {path}")


# 載入
annotated_queries = load_or_create_annotated(ANNOTATED_PATH, TEST_SET_NAME)

In [ ]:
def load_or_create_annotated(path: str, test_set_name: str) -> list:
    """
    如果標註檔已存在 → 載入
    如果不存在 → 回傳空 list，提示你用 init_from_variable() 建立

    為什麼分開？
    - 標註是「人工 + 輔助」的過程，不是一鍵完成的
    - 這個函式只負責「讀取已經標好的資料」
    """
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        print(f"✅ 載入 {len(data)} 個已標註 query（從 {path}）")

        # 檢查有多少已完成標註（relevant_chunks 非空）
        done = sum(1 for q in data if q.get("relevant_chunks"))
        todo = len(data) - done
        if todo > 0:
            print(f"   ⚠️ 其中 {todo} 個尚未標註 relevant_chunks")
        return data
    else:
        print(f"⚠️ 標註檔不存在: {path}")
        print(f"   → 如果你已經有題目變數，請用 init_from_variable() 建立標註檔")
        print(f"   → 例如: annotated_queries = init_from_variable(scenario_queries)")
        return []


def init_from_variable(query_list: list) -> list:
    """
    首次建立標註檔：從你筆記本裡已有的題目變數，驗證格式後存到 Google Drive。

    這是「第一次用某個 test set」時的入口：
    1. 你在筆記本裡定義好 scenario_queries（或 basic_queries 等）
    2. 呼叫 init_from_variable(scenario_queries)
    3. 它會驗證格式、存檔、回傳可用的 annotated_queries

    用法：
        annotated_queries = init_from_variable(scenario_queries)

    之後重開 Colab，load_or_create_annotated() 就會自動從檔案載入。
    """
    # --- 格式驗證 ---
    required_keys = {"query_id", "query", "relevant_chunks"}
    for i, q in enumerate(query_list):
        missing = required_keys - set(q.keys())
        if missing:
            raise ValueError(
                f"❌ 第 {i} 筆（{q.get('query_id', '?')}）缺少欄位: {missing}\n"
                f"   每筆必須包含: {sorted(required_keys)}\n"
                f"   你的欄位: {sorted(q.keys())}"
            )
        if not isinstance(q["relevant_chunks"], list):
            raise ValueError(
                f"❌ 第 {i} 筆（{q['query_id']}）的 relevant_chunks 必須是 list，"
                f"收到: {type(q['relevant_chunks'])}"
            )

    # --- 存檔 ---
    save_annotated(query_list)

    # --- 統計 ---
    done = sum(1 for q in query_list if q.get("relevant_chunks"))
    print(f"\n📋 已建立標註檔: {TEST_SET_NAME}")
    print(f"   總題數: {len(query_list)}")
    print(f"   已標註: {done}")
    print(f"   待標註: {len(query_list) - done}")
    print(f"   存檔位置: {ANNOTATED_PATH}")
    print(f"\n   下一步: 用 annotate_helper() 開始標註 relevant_chunks")

    return query_list


def save_annotated(data: list, path: str = None):
    """
    儲存標註資料（標幾題就存一次，避免 Colab 斷線丟失）

    Args:
        data: 標註好的 query list
        path: 預設使用 ANNOTATED_PATH（由 TEST_SET_NAME 決定）
    """
    if path is None:
        path = ANNOTATED_PATH
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    done = sum(1 for q in data if q.get("relevant_chunks"))
    print(f"💾 已儲存 {len(data)} 個 query（{done} 個已標註）→ {path}")

In [ ]:
# === 首次建立標註檔 ===
# 你在前一個 cell 已經定義了 scenario_queries 變數
# init_from_variable() 會：
#   1. 驗證格式（確認有 query_id, query, relevant_chunks）
#   2. 存到 Google Drive（路徑由 TEST_SET_NAME 自動決定）
#   3. 回傳可用的 annotated_queries

annotated_queries = init_from_variable(scenario_queries)

#### 評估

In [ ]:
eval_result = run_eval_comparison(annotated_queries, k=5)
save_eval_result(eval_result)

In [ ]:
# === 看每題的明細 ===
for method_name in ["dense", "bm25", "hybrid_rrf"]:
    print(f"\n{'=' * 60}")
    print(f"📊 {method_name} — 逐題明細")
    print(f"{'=' * 60}")

    per_query = eval_result["results"][method_name]["per_query"]
    for pq in per_query:
        # MRR=1.0 代表 top-1 就命中，0.5 代表 top-2，0.0 代表完全沒命中
        hit = "✅" if pq["mrr"] > 0 else "❌"
        print(f"  {hit} [{pq['query_id']}] MRR={pq['mrr']:.2f} P@3={pq['precision@3']:.2f} R@5={pq['recall@5']:.2f}")
        print(f"     query: {pq['query'][:50]}...")

        if pq["mrr"] == 0:
            print(f"     ⚠️ relevant={pq['relevant_chunks']}")
            print(f"     ⚠️ 但 top-5 撈到: {pq['retrieved_top5']}")
        print()

#### 工具

標註輔助工具

In [ ]:
def annotate_helper(query: str, k: int = 5, show_text_length: int = 150):
    """
    標註輔助：印出一個 query 的 dense / bm25 / hybrid 搜索結果。
    你看完後，人工決定哪些 chunk_id 是 relevant。

    Args:
        query: 要標註的查詢
        k: 每種搜索方法顯示幾筆
        show_text_length: 每個 chunk 顯示多少字元（調大可以看更多內容）
    """
    print(f"{'=' * 70}")
    print(f"🔍 Query: {query}")
    print(f"{'=' * 70}")

    methods = [
        ("Dense", dense_search(query, faiss_index, chunks, embedding_model, k=k)),
        ("BM25", bm25_search(query, bm25_index, tokenized_corpus, chunks, k=k)),
        ("Hybrid", hybrid_search(query, faiss_index, chunks, embedding_model,
                                  bm25_index, tokenized_corpus, k=k)),
    ]

    # 收集所有出現過的 chunk_id（方便你複製貼上）
    seen_ids = []

    for method_name, results in methods:
        print(f"\n--- {method_name} Search (top-{k}) ---")
        for i, r in enumerate(results, 1):
            c = r["chunk"]
            cid = c["chunk_id"]
            if cid not in seen_ids:
                seen_ids.append(cid)
            text_preview = c["text"][:show_text_length].replace("\n", " ")
            print(f"  [{i}] {cid}")
            print(f"      score={r['score']:.4f} | {c.get('doc_category', '')} | p.{c['page']}")
            print(f"      {text_preview}...")
            print()

    # 印出所有出現過的 chunk_id，方便複製
    print(f"{'=' * 70}")
    print(f"📋 所有出現過的 chunk_id（共 {len(seen_ids)} 個，方便複製）：")
    for cid in seen_ids:
        print(f'    "{cid}",')
    print(f"{'=' * 70}")


def annotate_batch(queries: list, start: int = 0, count: int = 3, k: int = 5):
    """
    批次標註輔助：連續顯示多個 query 的搜索結果。

    Args:
        queries: 標註資料 list（需要有 "query" 欄位）
        start: 從第幾個開始（0-indexed）
        count: 一次顯示幾個
        k: 每個 query 每種方法顯示幾筆
    """
    end = min(start + count, len(queries))
    for i in range(start, end):
        q = queries[i]
        qid = q.get("query_id", f"#{i}")
        already_done = bool(q.get("relevant_chunks"))
        status = "✅ 已標註" if already_done else "⬜ 待標註"

        print(f"\n\n{'#' * 70}")
        print(f"# [{qid}] {status}")
        print(f"{'#' * 70}")

        annotate_helper(q["query"], k=k)

評估指標

In [ ]:
def precision_at_k(retrieved_ids: list, relevant_ids: set, k: int) -> float:
    """
    Precision@K: top-K 結果中，有多少比例是 relevant 的

    例：top-5 = [A, B, C, D, E]，relevant = {A, C}
    → Precision@5 = 2/5 = 0.4
    """
    top_k = retrieved_ids[:k]
    hits = sum(1 for rid in top_k if rid in relevant_ids)
    return hits / k if k > 0 else 0.0


def recall_at_k(retrieved_ids: list, relevant_ids: set, k: int) -> float:
    """
    Recall@K: 所有 relevant 中，top-K 撈到了幾個

    例：relevant = {A, C, F}，top-5 撈到 A, C
    → Recall@5 = 2/3 ≈ 0.67
    """
    if not relevant_ids:
        return 0.0
    top_k = retrieved_ids[:k]
    hits = sum(1 for rid in top_k if rid in relevant_ids)
    return hits / len(relevant_ids)


def mrr(retrieved_ids: list, relevant_ids: set) -> float:
    """
    MRR: 第一個 relevant 結果的排名倒數

    例：results = [X, A, Y, ...]，A 是 relevant
    → RR = 1/2 = 0.5
    """
    for i, rid in enumerate(retrieved_ids):
        if rid in relevant_ids:
            return 1.0 / (i + 1)
    return 0.

單方法評估

In [ ]:
def evaluate_single_method(
    queries: list,
    search_fn,
    method_name: str,
    k: int = 5,
    **search_kwargs,
) -> dict:
    """
    對一組標註資料，跑某種搜索方法，計算指標。
    只評估 relevant_chunks 非空的 query（跳過未標註的）。

    Args:
        queries: 標註好的 query list
        search_fn: 搜索函式（dense_search / bm25_search / hybrid_search）
        method_name: 方法名稱（用於報告）
        k: top-K
        **search_kwargs: 傳給 search_fn 的參數

    Returns:
        dict: 包含 aggregate metrics + per_query details
    """
    # 只評估已標註的
    evaluable = [q for q in queries if q.get("relevant_chunks")]
    if not evaluable:
        print(f"⚠️ 沒有任何已標註的 query，無法評估")
        return {"method": method_name, "error": "no_annotated_queries"}

    skipped = len(queries) - len(evaluable)
    if skipped > 0:
        print(f"  ℹ️ 跳過 {skipped} 個未標註 query，評估 {len(evaluable)} 個")

    all_p3, all_p5, all_r5, all_mrr_scores = [], [], [], []
    per_query = []

    for item in evaluable:
        query = item["query"]
        relevant_ids = set(item["relevant_chunks"])

        # 執行搜索
        results = search_fn(query=query, k=k, **search_kwargs)
        retrieved_ids = [r["chunk"]["chunk_id"] for r in results]

        # 計算指標
        p3 = precision_at_k(retrieved_ids, relevant_ids, 3)
        p5 = precision_at_k(retrieved_ids, relevant_ids, 5)
        r5 = recall_at_k(retrieved_ids, relevant_ids, 5)
        rr = mrr(retrieved_ids, relevant_ids)

        all_p3.append(p3)
        all_p5.append(p5)
        all_r5.append(r5)
        all_mrr_scores.append(rr)

        per_query.append({
            "query_id": item.get("query_id", ""),
            "query": query,
            "category": item.get("category", ""),
            "relevant_chunks": list(relevant_ids),
            "retrieved_top5": retrieved_ids[:5],
            "precision@3": round(p3, 4),
            "precision@5": round(p5, 4),
            "recall@5": round(r5, 4),
            "mrr": round(rr, 4),
            # raw scores — Day 3 畫分布圖用
            "raw_scores": [
                {"chunk_id": r["chunk"]["chunk_id"], "score": round(r["score"], 6)}
                for r in results[:k]
            ],
        })

    n = len(evaluable)
    return {
        "method": method_name,
        "num_evaluated": n,
        "num_skipped": skipped,
        "aggregate": {
            "mean_precision@3": round(sum(all_p3) / n, 4),
            "mean_precision@5": round(sum(all_p5) / n, 4),
            "mean_recall@5": round(sum(all_r5) / n, 4),
            "mean_mrr": round(sum(all_mrr_scores) / n, 4),
        },
        "per_query": per_query,
    }

三方法比較（一鍵執行）

In [ ]:
def run_eval_comparison(queries: list, k: int = 5) -> dict:
    """
    一鍵跑 dense / bm25 / hybrid 三種方法的比較。

    Args:
        queries: 已標註的 query list
        k: top-K

    Returns:
        dict: 完整評估結果（含 config, 三種方法的結果）
    """
    print(f"🔬 開始評估: {TEST_SET_NAME}")
    print(f"   查詢數: {len(queries)}")
    print(f"   k: {k}")
    print(f"{'=' * 55}")

    # Dense
    print(f"\n1/3 Dense Search...")
    dense_eval = evaluate_single_method(
        queries, dense_search, "dense", k=k,
        faiss_index=faiss_index, chunks=chunks,
        embedding_model=embedding_model,
    )

    # BM25
    print(f"\n2/3 BM25 Search...")
    bm25_eval = evaluate_single_method(
        queries, bm25_search, "bm25", k=k,
        bm25_index=bm25_index, tokenized_corpus=tokenized_corpus,
        chunks=chunks,
    )

    # Hybrid
    print(f"\n3/3 Hybrid Search (RRF)...")
    hybrid_eval = evaluate_single_method(
        queries, hybrid_search, "hybrid_rrf", k=k,
        faiss_index=faiss_index, chunks=chunks,
        embedding_model=embedding_model,
        bm25_index=bm25_index, tokenized_corpus=tokenized_corpus,
    )

    # 印出比較表
    print(f"\n{'=' * 55}")
    print(f"📊 {TEST_SET_NAME} 評估結果")
    print(f"{'=' * 55}")
    print(f"{'Method':<15} {'P@3':>8} {'P@5':>8} {'R@5':>8} {'MRR':>8}")
    print(f"{'-' * 55}")
    for ev in [dense_eval, bm25_eval, hybrid_eval]:
        if "error" in ev:
            print(f"{ev['method']:<15} {'N/A':>8}")
            continue
        agg = ev["aggregate"]
        print(f"{ev['method']:<15} "
              f"{agg['mean_precision@3']:>8.3f} "
              f"{agg['mean_precision@5']:>8.3f} "
              f"{agg['mean_recall@5']:>8.3f} "
              f"{agg['mean_mrr']:>8.3f}")

    # 組裝完整結果
    full_result = {
        "test_set": TEST_SET_NAME,
        "evaluated_at": datetime.now().isoformat(),
        "config": {
            "embedding_model": EMBEDDING_MODEL_NAME,
            "chunk_size": index_metadata.get("config", {}).get("chunk_size"),
            "total_chunks": len(chunks),
            "rrf_k": DEFAULT_RRF_K,
            "priority_weighting": DEFAULT_USE_PRIORITY_WEIGHTING,
            "k": k,
        },
        "results": {
            "dense": dense_eval,
            "bm25": bm25_eval,
            "hybrid_rrf": hybrid_eval,
        },
    }

    return full_result


def save_eval_result(result: dict, path: str = None):
    """儲存評估結果"""
    if path is None:
        path = EVAL_RESULT_PATH
    with open(path, "w", encoding="utf-8") as f:
        json.dump(result, f, ensure_ascii=False, indent=2)
    print(f"\n✅ 評估結果已儲存: {path}")

標註進度總覽

In [ ]:
def show_annotation_progress(queries: list):
    """印出標註進度，方便你知道還剩多少"""
    total = len(queries)
    done = sum(1 for q in queries if q.get("relevant_chunks"))
    todo = total - done

    print(f"📊 標註進度: {TEST_SET_NAME}")
    print(f"   總數: {total}")
    print(f"   已標: {done} ✅")
    print(f"   待標: {todo} ⬜")
    print(f"   進度: {done/total*100:.0f}%")

    if todo > 0:
        # 列出待標的前 5 個
        print(f"\n   待標的前 5 個：")
        count = 0
        for i, q in enumerate(queries):
            if not q.get("relevant_chunks"):
                qid = q.get("query_id", f"#{i}")
                print(f"   [{qid}] {q['query'][:60]}...")
                count += 1
                if count >= 5:
                    break


print(f"\n✅ PART 2.5 Retrieval-Only 評估模組載入完成")
print(f"   Test Set: {TEST_SET_NAME}")
print(f"   下一步: show_annotation_progress(annotated_queries)")


✅ PART 2.5 Retrieval-Only 評估模組載入完成


NameError: name 'TEST_SET_NAME' is not defined

In [ ]:
show_annotation_progress(annotated_queries)

In [ ]:
# # === query_relevance_dataset.json ===
# # 這就是你的 ground truth
# # strict relevance

# scenario_queries = [
#     {
#         "query_id": "S01",
#         "query": "一家汽車經銷商卻持續出口大量紡織品到海外，這種買賣項目和公司登記業務完全對不上的情況，是否值得關注？",
#         "category": "場景描述",
#         "relevant_chunks": [],
#         "expected_document": "fatf_tbm_laundering_red_flags.pdf",
#         "lexical_overlap_risk": "high",
#         "semantic_distance_level": 1,
#         "semantic_shift_note": None
#     },
#     {
#         "query_id": "S02",
#         "query": "某間公司的組織架構異常複雜，中間層層疊了好幾家在監管寬鬆地區註冊的空殼法人，而且看不出實際業務邏輯，這種組織設計有什麼問題？",
#         "category": "場景描述",
#         "relevant_chunks": [],
#         "expected_document": "fatf_tbm_laundering_red_flags.pdf",
#         "lexical_overlap_risk": "high",
#         "semantic_distance_level": 1,
#         "semantic_shift_note": None
#     },
#     {
#         "query_id": "S03",
#         "query": "賣方開出來的帳單上寫的錢和雙方本來談好的價錢差了一大截，連運過來的東西數量和品質也跟紙上寫的對不上，這種帳面和實物兜不攏的狀況，算是什麼問題？",
#         "category": "去關鍵字/語義轉換",
#         "relevant_chunks": [],
#         "expected_document": "fatf_tbm_laundering_red_flags.pdf",
#         "lexical_overlap_risk": "medium",
#         "semantic_distance_level": 2,
#         "semantic_shift_note": "移除「發票」「合約」「進出口」「貨物」等詞。核心：文件與交付不一致。"
#     },
#     {
#         "query_id": "S04",
#         "query": "某貿易商的帳戶看起來只是資金的中繼站——每天進來大量匯款、快速轉走，日終餘額幾乎為零，而且找不到合理的商業解釋，這類帳戶有什麼特徵？",
#         "category": "場景描述",
#         "relevant_chunks": [],
#         "expected_document": "fatf_tbm_laundering_red_flags.pdf",
#         "lexical_overlap_risk": "high",
#         "semantic_distance_level": 1,
#         "semantic_shift_note": None
#     },
#     {
#         "query_id": "S05",
#         "query": "某企業進口貨物的金額，和它對外匯出用於支付進口的銀行轉帳金額有很大落差，這種數字上的不一致意味著什麼？",
#         "category": "場景描述",
#         "relevant_chunks": [],
#         "expected_document": "fatf_tbm_laundering_red_flags.pdf",
#         "lexical_overlap_risk": "high",
#         "semantic_distance_level": 1,
#         "semantic_shift_note": None
#     },
#     {
#         "query_id": "S06",
#         "query": "一家公司的註冊地址是一棟住宅大樓的信箱地址，而且沒有標明具體的單位號碼，也沒有任何商業或工業空間，這樣的登記地點為什麼引人懷疑？",
#         "category": "場景描述",
#         "relevant_chunks": [],
#         "expected_document": "fatf_tbm_laundering_red_flags.pdf",
#         "lexical_overlap_risk": "high",
#         "semantic_distance_level": 1,
#         "semantic_shift_note": None
#     },
#     {
#         "query_id": "S07",
#         "query": "一段已經確認的約定，在即將兌現的節點上被替換成一個陌生的版本——這種「最後一刻的替換」在交易情境中，通常暗示了什麼？",
#         "category": "去關鍵字/語義轉換",
#         "relevant_chunks": [],
#         "expected_document": "fatf_tbm_laundering_red_flags.pdf",
#         "lexical_overlap_risk": "low",
#         "semantic_distance_level": 3,
#         "semantic_shift_note": "將「付款流程被篡改」抽象化為「約定在節點被替換」，去金融術語。"
#     },
#     {
#         "query_id": "S08",
#         "query": "某人在數位貨幣交易平台開戶時存入了一大筆資金，金額與其個人背景和職業收入完全不相稱，而且第一天就開始大量進出交易，這種新用戶行為有什麼可疑之處？",
#         "category": "場景描述",
#         "relevant_chunks": [],
#         "expected_document": "fatf_virtual_assets_red_flags.pdf",
#         "lexical_overlap_risk": "high",
#         "semantic_distance_level": 1,
#         "semantic_shift_note": None
#     },
#     {
#         "query_id": "S09",
#         "query": "某人把一件有完整公開來歷的東西，換成一件來歷在結構上就無從查證的東西——這個轉換行為，本身說明了什麼？",
#         "category": "去關鍵字/語義轉換",
#         "relevant_chunks": [],
#         "expected_document": "fatf_virtual_assets_red_flags.pdf",
#         "lexical_overlap_risk": "low",
#         "semantic_distance_level": 3,
#         "semantic_shift_note": "保留「可追溯→不可追溯」結構，改為抽象物件隱喻。"
#     },
#     {
#         "query_id": "S10",
#         "query": "有人透過暗網關聯的 IP 位址或加密通訊工具登入數位資產服務平台進行交易，這種連線方式為什麼會被視為異常？",
#         "category": "場景描述",
#         "relevant_chunks": [],
#         "expected_document": "fatf_virtual_assets_red_flags.pdf",
#         "lexical_overlap_risk": "high",
#         "semantic_distance_level": 1,
#         "semantic_shift_note": None
#     },
#     {
#         "query_id": "S11",
#         "query": "客戶的資金來源直接關聯到已知的網路勒索事件地址，或是來自地下交易市場的錢包，這種資金流向為什麼需要特別警覺？",
#         "category": "場景描述",
#         "relevant_chunks": [],
#         "expected_document": "fatf_virtual_assets_red_flags.pdf",
#         "lexical_overlap_risk": "high",
#         "semantic_distance_level": 1,
#         "semantic_shift_note": None
#     },
#     {
#         "query_id": "S12",
#         "query": "一位用戶註冊數位資產帳號時提交了經過修改的身分證照片，而且在被要求補充資金來源說明時拒絕回答，這在開戶審查階段代表什麼？",
#         "category": "場景描述",
#         "relevant_chunks": [],
#         "expected_document": "fatf_virtual_assets_red_flags.pdf",
#         "lexical_overlap_risk": "high",
#         "semantic_distance_level": 1,
#         "semantic_shift_note": None
#     },
#     {
#         "query_id": "S13",
#         "query": "某客戶習慣把資金匯到一個對數位資產完全沒有監管規範的國家的交易平台，而且那個平台也沒有落實任何身分驗證措施，這和地理風險有什麼關係？",
#         "category": "場景描述",
#         "relevant_chunks": [],
#         "expected_document": "fatf_virtual_assets_red_flags.pdf",
#         "lexical_overlap_risk": "high",
#         "semantic_distance_level": 1,
#         "semantic_shift_note": None
#     },
#     {
#         "query_id": "S14",
#         "query": "某平台發現有大量看似互不相關的數位錢包，其實全部是從同一個網路位址操控的，這可能代表什麼手法？",
#         "category": "場景描述",
#         "relevant_chunks": [],
#         "expected_document": "fatf_virtual_assets_red_flags.pdf",
#         "lexical_overlap_risk": "high",
#         "semantic_distance_level": 1,
#         "semantic_shift_note": None
#     },
#     {
#         "query_id": "S15",
#         "query": "有人先把一批說不清楚哪裡來的財物，委託多人分批代為處理，途中幾度易手，最後以置產或添購名貴物件的方式讓那筆財物看起來出自正當來源——請問法規上用什麼框架來描述這整個過程？",
#         "category": "去關鍵字/語義轉換",
#         "relevant_chunks": [],
#         "expected_document": "tw_aml_training_slides.pdf",
#         "lexical_overlap_risk": "medium",
#         "semantic_distance_level": 2,
#         "semantic_shift_note": "轉為記者敘事視角；全面換詞（財物、易手、正當來源），保留洗錢三階段結構。"
#     },
#     {
#         "query_id": "S16",
#         "query": "銀行在什麼情況下會直接拒絕讓客戶開戶或進行交易？例如發現客戶疑似使用假名或者拿偽造的證件來辦理，還有哪些類似的拒絕情形？",
#         "category": "場景描述",
#         "relevant_chunks": [],
#         "expected_document": "tw_aml_training_slides.pdf",
#         "lexical_overlap_risk": "high",
#         "semantic_distance_level": 1,
#         "semantic_shift_note": None
#     },
#     {
#         "query_id": "S17",
#         "query": "如果有人在報紙上看到「高價收購銀行帳戶」的廣告，把自己的金融卡和存摺交給對方，即使只是為了賺取小利，這樣做在法律上會面臨什麼後果？",
#         "category": "場景描述",
#         "relevant_chunks": [],
#         "expected_document": "tw_aml_training_slides.pdf",
#         "lexical_overlap_risk": "high",
#         "semantic_distance_level": 1,
#         "semantic_shift_note": None
#     },
#     {
#         "query_id": "S18",
#         "query": "主管交代員工把一大筆來路不明的現金拆成好幾份、每份都控制在法定申報金額以下分批存入銀行，這樣刻意規避通報機制的行為會觸犯什麼罪？",
#         "category": "場景描述",
#         "relevant_chunks": [],
#         "expected_document": "tw_aml_training_slides.pdf",
#         "lexical_overlap_risk": "high",
#         "semantic_distance_level": 1,
#         "semantic_shift_note": None
#     },
#     {
#         "query_id": "S19",
#         "query": "出國旅遊時，除了各國貨幣現鈔和黃金之外，還有哪些可能被利用於資金非法移動的物品也需要在邊境向海關申報？如果沒有誠實申報會怎樣？",
#         "category": "場景描述",
#         "relevant_chunks": [],
#         "expected_document": "tw_aml_training_slides.pdf",
#         "lexical_overlap_risk": "high",
#         "semantic_distance_level": 1,
#         "semantic_shift_note": None
#     },
#     {
#         "query_id": "S20",
#         "query": "要在銀行以公司名義開戶時，除了營業登記等基本文件之外，銀行還會要求提供哪些關於公司背後實際控制人的身分資訊？",
#         "category": "場景描述",
#         "relevant_chunks": [],
#         "expected_document": "tw_aml_training_slides.pdf",
#         "lexical_overlap_risk": "high",
#         "semantic_distance_level": 1,
#         "semantic_shift_note": None
#     }
# ]

# # 存檔
# import json
# relevance_path = "/content/drive/MyDrive/AML/eval/scenario_queries.json"
# os.makedirs(os.path.dirname(relevance_path), exist_ok=True)

# with open(relevance_path, "w", encoding="utf-8") as f:
#     json.dump(scenario_queries, f, ensure_ascii=False, indent=2)

# print(f"✅ 已儲存 {len(scenario_queries)} 個標註到 {relevance_path}")

In [ ]:
# # === Retrieval Evaluation Metrics ===

# def precision_at_k(retrieved_ids: list, relevant_ids: set, k: int) -> float:
#     """
#     Precision@K: top-K 結果中，有多少比例是 relevant 的

#     例如：top-5 撈到 [A, B, C, D, E]，其中 A, C 是 relevant
#     → Precision@5 = 2/5 = 0.4
#     """
#     top_k = retrieved_ids[:k]
#     hits = sum(1 for rid in top_k if rid in relevant_ids)
#     return hits / k


# def recall_at_k(retrieved_ids: list, relevant_ids: set, k: int) -> float:
#     """
#     Recall@K: 所有 relevant chunks 中，有多少被 top-K 撈到

#     例如：relevant = {A, C, F}，top-5 撈到了 A 和 C
#     → Recall@5 = 2/3 = 0.67
#     """
#     if not relevant_ids:
#         return 0.0
#     top_k = retrieved_ids[:k]
#     hits = sum(1 for rid in top_k if rid in relevant_ids)
#     return hits / len(relevant_ids)


# def mrr(retrieved_ids: list, relevant_ids: set) -> float:
#     """
#     MRR (Mean Reciprocal Rank): 第一個 relevant 結果出現在第幾名的倒數

#     例如：top-5 = [X, A, Y, Z, W]，A 是 relevant
#     → RR = 1/2 = 0.5（因為 A 在第 2 名）

#     如果第 1 名就是 relevant → RR = 1.0（最好的情況）
#     """
#     for i, rid in enumerate(retrieved_ids):
#         if rid in relevant_ids:
#             return 1.0 / (i + 1)
#     return 0.0

In [ ]:
# # === 批次評估 ===

# def evaluate_retrieval(
#     query_relevance: list,
#     search_fn,            # dense_search / bm25_search / hybrid_search
#     search_fn_name: str,
#     k: int = 5,
#     **search_kwargs,       # 傳給 search_fn 的額外參數
# ) -> dict:
#     """
#     對一組標註資料，跑某種搜索方法，計算指標

#     Returns:
#         dict with per-query details and aggregate metrics
#     """
#     all_p_at_3 = []
#     all_p_at_5 = []
#     all_recall_at_5 = []
#     all_mrr = []
#     per_query = []

#     for item in query_relevance:
#         query = item["query"]
#         relevant_ids = set(item["relevant_chunks"])

#         # 跑搜索
#         results = search_fn(query=query, k=k, **search_kwargs)

#         # 取出 chunk_ids
#         retrieved_ids = [r["chunk"]["chunk_id"] for r in results]

#         # 計算指標
#         p3 = precision_at_k(retrieved_ids, relevant_ids, 3)
#         p5 = precision_at_k(retrieved_ids, relevant_ids, 5)
#         r5 = recall_at_k(retrieved_ids, relevant_ids, 5)
#         rr = mrr(retrieved_ids, relevant_ids)

#         all_p_at_3.append(p3)
#         all_p_at_5.append(p5)
#         all_recall_at_5.append(r5)
#         all_mrr.append(rr)

#         per_query.append({
#             "query_id": item.get("query_id", ""),
#             "query": query,
#             "category": item.get("category", ""),
#             "retrieved": retrieved_ids[:5],
#             "relevant": list(relevant_ids),
#             "precision@3": p3,
#             "precision@5": p5,
#             "recall@5": r5,
#             "mrr": rr,
#             # 記錄 raw scores（Day 3 會用到）
#             "raw_scores": [
#                 {"chunk_id": r["chunk"]["chunk_id"], "score": r["score"]}
#                 for r in results[:k]
#             ],
#         })

#     n = len(query_relevance)
#     return {
#         "search_method": search_fn_name,
#         "num_queries": n,
#         "aggregate": {
#             "mean_precision@3": sum(all_p_at_3) / n,
#             "mean_precision@5": sum(all_p_at_5) / n,
#             "mean_recall@5": sum(all_recall_at_5) / n,
#             "mean_mrr": sum(all_mrr) / n,
#         },
#         "per_query": per_query,
#     }

In [ ]:
# # === 三種搜索方法比較 ===

# # 載入標註資料
# with open("/content/drive/MyDrive/AML/eval/query_relevance_v1.json", "r") as f:
#     query_relevance = json.load(f)

# print(f"載入 {len(query_relevance)} 個標註 query\n")

# # 1. Dense Search
# dense_eval = evaluate_retrieval(
#     query_relevance,
#     search_fn=dense_search,
#     search_fn_name="dense",
#     k=5,
#     faiss_index=faiss_index,
#     chunks=chunks,
#     embedding_model=embedding_model,
# )

# # 2. BM25 Search
# bm25_eval = evaluate_retrieval(
#     query_relevance,
#     search_fn=bm25_search,
#     search_fn_name="bm25",
#     k=5,
#     bm25_index=bm25_index,
#     tokenized_corpus=tokenized_corpus,
#     chunks=chunks,
# )

# # 3. Hybrid Search (RRF)
# hybrid_eval = evaluate_retrieval(
#     query_relevance,
#     search_fn=hybrid_search,
#     search_fn_name="hybrid_rrf",
#     k=5,
#     faiss_index=faiss_index,
#     chunks=chunks,
#     embedding_model=embedding_model,
#     bm25_index=bm25_index,
#     tokenized_corpus=tokenized_corpus,
# )

# # === 印出比較表 ===
# print(f"{'Method':<15} {'P@3':>8} {'P@5':>8} {'R@5':>8} {'MRR':>8}")
# print("-" * 50)
# for eval_result in [dense_eval, bm25_eval, hybrid_eval]:
#     agg = eval_result["aggregate"]
#     print(f"{eval_result['search_method']:<15} "
#           f"{agg['mean_precision@3']:>8.3f} "
#           f"{agg['mean_precision@5']:>8.3f} "
#           f"{agg['mean_recall@5']:>8.3f} "
#           f"{agg['mean_mrr']:>8.3f}")

In [ ]:
# # === 儲存評估結果 ===

# eval_results = {
#     "experiment": "retrieval_only_eval_v1",
#     "date": datetime.now().isoformat(),
#     "num_queries": len(query_relevance),
#     "config": {
#         "embedding_model": EMBEDDING_MODEL_NAME,
#         "chunk_size": index_metadata.get("config", {}).get("chunk_size"),
#         "total_chunks": len(chunks),
#         "rrf_k": DEFAULT_RRF_K,
#         "priority_weighting": DEFAULT_USE_PRIORITY_WEIGHTING,
#     },
#     "results": {
#         "dense": dense_eval,
#         "bm25": bm25_eval,
#         "hybrid": hybrid_eval,
#     }
# }

# eval_output_path = "/content/drive/MyDrive/AML/eval/retrieval_eval_v1.json"
# with open(eval_output_path, "w", encoding="utf-8") as f:
#     json.dump(eval_results, f, ensure_ascii=False, indent=2)

# print(f"✅ 評估結果已儲存: {eval_output_path}")

# 🤖 PART 3: LLM 分析模組

這個部分負責「把檢索到的段落交給 LLM 分析」


職責：
- 接收 PART 2 檢索到的上下文
- 組裝 Prompt
- 呼叫 LLM 進行分析
- 格式化輸出結果

**模型區分**：
- `embedding_model`: SentenceTransformer → PART 2 向量檢索
- `llm_config` / `SELECTED_CONFIG`: Gemini / Groq → PART 3 文本生成

### 3.0 初始化 LLM Client

In [ ]:
llm_client = None

if SELECTED_CONFIG["provider"] == "groq":
    llm_client = Groq(api_key=SELECTED_CONFIG["api_key"])
    print(f"🚀 Groq 實體已就緒：{SELECTED_CONFIG['llm_model_name']}")

elif SELECTED_CONFIG["provider"] == "gemini":
    genai.configure(api_key=SELECTED_CONFIG["api_key"])
    llm_client = genai.GenerativeModel(SELECTED_CONFIG["llm_model_name"])
    print(f"🌟 Gemini 實體已就緒：{SELECTED_CONFIG['llm_model_name']}")

else:
    raise ValueError(f"❌ 未知的 Provider: {SELECTED_CONFIG['provider']}")

assert llm_client is not None, "❌ LLM Client 初始化失敗"
print(f"✅ LLM Client 就緒")

🚀 Groq 實體已就緒：llama-3.1-8b-instant
✅ LLM Client 就緒


In [ ]:
# ===== 推理函數 =====
def generate_response(prompt: str):
    """統一推理介面：支援 Groq 與 Gemini"""

    # 1. 處理 Groq 邏輯
    if SELECTED_CONFIG["provider"] == "groq":
        # Groq 使用的是類似 OpenAI 的 Chat Completion 格式
        chat_completion = llm_client.chat.completions.create(
            messages=[
                {"role": "user", "content": prompt}
            ],
            model=SELECTED_CONFIG["llm_model_name"],
            # 這裡可以加入 temperature 等參數
        )
        return chat_completion.choices[0].message.content.strip()

    # 2. 處理 Gemini 邏輯
    elif SELECTED_CONFIG["provider"] == "gemini":
        # Gemini 的 llm_client 是 genai.GenerativeModel 實例
        response = llm_client.generate_content(prompt)
        return response.text

    # 3. 處理舊的或未定義的邏輯 (安全網)
    else:
        return f"❌ 錯誤：未支援的 Provider ({SELECTED_CONFIG['provider']})"

# ===== 測試 =====
test_prompt = "請簡要說明什麼是 RAG 系統？"
print(f"✅ 使用模型：{SELECTED_CONFIG['llm_model_name']}")
response = generate_response(test_prompt)
print(f"回應：\n{response}")

✅ 使用模型：llama-3.1-8b-instant
回應：
RAG 系統（RAG是紅線-綠線-黃線的縮寫）是一種用於描述項目進展狀態的方法，最初是由阿爾伯特·帕金森博士在1940年代開發出的。

RAG 系統的主要目的是使您能夠快速和簡潔地評估項目的進展狀態，從而做出合理的決策。下面是 RAG 系統的基本原理：

- 红線（Red）：表示項目已經嚴重超時或是存在嚴重問題，這通常需要緊急處理。
- 綠線（Green）：表示項目正在順利進行，或是已經完成，這通常是正常狀態。
- 黄線（Amber）：表示項目可能會出現問題，或是存在一些阻礙，這通常需要關注和調整。

RAG 系統的應用包括項目管理、產品開發、運營維護等領域。它的優點包括簡單易使用、能夠快速評估項目狀態等。


### 3.1 System Prompt

In [ ]:
SYSTEM_PROMPT = """
你是一位「AML 紅旗助教」，專門協助金融從業人員辨識可疑交易情境中的洗錢防制紅旗指標。

## 你的角色
- 你是「教育輔助工具」，不是執法機關
- 你的輸出是「紅旗清單 + 證據引用」，不是定罪判決
- 每個判斷都要有文件依據

## 判定標準（重要！）
### CONFIRMED（確認紅旗）
- 情境中**明確描述**多個紅旗特徵
- 檢索到的文件中有**對應的定義或標準**
- 例：「連續 3 天，每次 49 萬」+ 文件提到「刻意低於申報門檻 50 萬」

### POSSIBLE（可能紅旗）
- 情境有可疑跡象，但**缺少關鍵細節**
- 例：「多筆小額交易」但沒說具體金額、頻率、是否接近門檻
- 需要追問才能確認

### UNLIKELY（不太可能）
- 情境不符合任何紅旗特徵

### REFUSE（拒絕判斷）
- 情境涉及的領域不在你的知識範圍
- 例：TBML（貿易型洗錢），但你只有虛擬資產的文件

## 你認識的紅旗類型
| ID | 名稱 | 關鍵特徵 |
|----|------|----------|
| RF-01 | 門檻拆分 | 多筆小額、接近申報門檻、規避申報 |
| RF-02 | 快速流轉 | 入帳即轉出、多對手方、過路帳戶 |
| RF-03 | 現金異常 | 現金與背景不符、突然大量現金 |
| RF-04 | 第三人代辦 | 他人操作、人頭帳戶 |
| RF-05 | 跨境高風險 | 高風險地區、無合理跨境目的 |
| RF-06 | 與身分不符 | 交易與職業/業務不符 |
| RF-07 | 虛擬資產匿名 | 混幣、非託管錢包、隱私幣 |
| RF-08 | 公司不透明 | 空殼公司、受益人不明 |

## 輸出格式（JSON）
{
  "scenario_summary": "一句話摘要",
  "assessment": "confirmed | possible | unlikely | refuse",
  "identified_flags": [
    {
      "flag_id": "RF-XX",
      "flag_name": "名稱",
      "confidence": "high | medium | low",
      "reasoning": "分析理由",
      "evidence": [{"source": "TW_GOV", "page": 8, "quote": "引用"}],
      "missing_info": ["缺少的資訊"]
    }
  ],
  "follow_up_questions": ["追問問題"],
  "disclaimer": "本分析僅供教育參考，不構成法律意見。"
}

## 判定標準
- confirmed: 明確紅旗特徵 + 證據充足
- possible: 可疑跡象 + 需要追問
- unlikely: 不符合紅旗特徵
- refuse: 超出範圍 / 無法判斷
"""

print(f"✅ SYSTEM_PROMPT 已定義 ({len(SYSTEM_PROMPT)} 字元)")

✅ SYSTEM_PROMPT 已定義 (1285 字元)


### 3.2 build_user_prompt()


In [ ]:
def build_user_prompt(scenario: str, contexts: list) -> str:
    """
    組裝 user prompt

    Args:
        scenario: 情境描述
        contexts: retrieve_contexts() 的輸出

    Returns:
        str: 完整的 user prompt
    """
    context_str = ""
    for i, ctx in enumerate(contexts, 1):
        style_tag = {
            "authoritative": "【權威來源】",
            "simplified": "【簡化說明】",
            "technical": "【技術細節】",
        }.get(ctx.get("explanation_style", "neutral"), "")

        context_str += f"""
### 段落 {i} {style_tag}
- 來源：{ctx['source']}，第 {ctx['page']} 頁
- 分類：{ctx.get('doc_category', 'unknown')}
- 內容：{ctx['text']}
---
"""

    return f"""
## 情境描述
{scenario}

## 檢索到的相關文件
{context_str}

請根據以上資訊，分析這個情境中可能存在的 AML 紅旗。

### 使用指引
- 優先引用【權威來源】的定義和標準
- 可用【簡化說明】輔助解釋
- 所有判斷必須有明確的文件依據

請以 JSON 格式輸出。
"""

### 3.3 call_llm() 選擇模型

In [ ]:
def call_llm(
    system_prompt: str,
    user_prompt: str,
    llm_config: dict,
    response_format: str = "json",  # ← 新增："json" | "text"
):
    """
    統一 LLM 呼叫介面（支援 Gemini, Groq）

    Args:
        system_prompt: 系統提示詞
        user_prompt: 使用者提示詞
        llm_config: LLM 配置 dict（provider, llm_model_name, api_key）
        response_format: 回應格式
            - "json": 強制 JSON 輸出，回傳 dict（原行為，向後相容）
            - "text": 純文字輸出，回傳 str

    Returns:
        dict (json mode) 或 str (text mode)

    History:
        - v3.0: JSON-only（寫死 response_format=json_object）
        - v3.1: 新增 text 模式，解決 rewrite_query 的 API contract mismatch
    """
    provider = llm_config["provider"]
    llm_model_name = llm_config["llm_model_name"]
    api_key = llm_config["api_key"]

    # === Groq ===
    if provider == "groq":
        client = Groq(api_key=api_key)

        # 根據模式決定 API 參數
        api_kwargs = {
            "model": llm_model_name,
            "messages": [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
            ],
            "temperature": 0.1,
        }

        if response_format == "json":
            api_kwargs["response_format"] = {"type": "json_object"}

        try:
            completion = client.chat.completions.create(**api_kwargs)
            response_text = completion.choices[0].message.content

            if response_format == "json":
                return json.loads(response_text)
            else:
                return response_text.strip()

        except Exception as e:
            print(f"❌ Groq API 錯誤: {e}")
            if response_format == "json":
                return {
                    "assessment": "error",
                    "scenario_summary": f"Groq 處理失敗: {str(e)}",
                    "identified_flags": [],
                    "follow_up_questions": [],
                    "disclaimer": "系統錯誤",
                }
            else:
                # text 模式：回傳 None，讓呼叫端決定 fallback
                return None

    # === Gemini ===
    elif provider == "gemini":
        genai.configure(api_key=api_key)

        gen_config = {}
        if response_format == "json":
            gen_config["response_mime_type"] = "application/json"

        model = genai.GenerativeModel(
            model_name=llm_model_name,
            system_instruction=system_prompt,
            generation_config=gen_config,
        )

        try:
            response = model.generate_content(user_prompt)

            if response_format == "json":
                return json.loads(response.text)
            else:
                return response.text.strip()

        except Exception as e:
            print(f"❌ Gemini API 錯誤: {e}")
            if response_format == "json":
                return {
                    "assessment": "error",
                    "scenario_summary": f"Gemini 處理失敗: {str(e)}",
                    "identified_flags": [],
                    "follow_up_questions": [],
                    "disclaimer": "系統錯誤",
                }
            else:
                return None

    else:
        raise ValueError(f"不支援的 LLM provider: {provider}")



### 3.4 analyze_scenario()（主函數）


In [ ]:
def analyze_scenario(
    scenario: str,
    faiss_index,
    chunks: list,
    embedding_model,
    bm25_index,
    tokenized_corpus: list,
    llm_config: dict = None,
    k: int = DEFAULT_TOP_K,
    retrieval_method: str = "hybrid",
    use_priority_weighting: bool = DEFAULT_USE_PRIORITY_WEIGHTING,
    enable_gate: bool = DEFAULT_ENABLE_GATE,
    verbose: bool = True,
) -> dict:
    """
    主函數：分析一個 AML 情境

    串接所有步驟：
    0. (可選) Pre-LLM Gate 檢查
    1. 檢索相關段落
    2. 組裝 Prompt
    3. 呼叫 LLM 進行分析

    Args:
        scenario: 情境描述
        llm_config: LLM 配置（None 時使用全域 SELECTED_CONFIG）
        use_priority_weighting: 是否使用 retrieval_priority 加權（原 v2 功能）
        enable_gate: 是否啟用 Pre-LLM Gate（原 v3 功能）

    Returns:
        dict: LLM 分析結果（JSON 格式）

    History:
        - v1 (2025-01): 基礎版本
        - v2 (2025-02-09): 加入 use_priority_weighting
        - v3 (2025-02-09): 整合 enable_gate，統一為單一函數
    """
    if llm_config is None:
        llm_config = SELECTED_CONFIG  # ← [BUG FIX] 原本 v2 誤用整個 LLM_CONFIG dict

    # ===== Step 0: Pre-LLM Gate =====
    if enable_gate:
        if verbose:
            print("=" * 60)
            print("🚦 Step 0: Pre-LLM Gate 檢查...")

        # gate_result = pre_llm_gate(scenario, verbose=verbose)
        gate_result = pre_llm_gate(
            scenario,
            scope_classifier=scope_classifier,
            verbose=verbose,
        )

        if gate_result.decision == GateDecision.REFUSE:
            if verbose:
                print(f"   🚫 決策: REFUSE")
                print(f"   原因: {gate_result.reason_message}")
                print("=" * 60)
            return gate_result.to_refuse_response()

        if verbose:
            print(f"   ✅ 決策: ALLOW")

    # ===== Step 1: 檢索 =====
    if verbose:
        if not enable_gate:
            print("=" * 60)
        print("🔍 Step 1: 檢索相關段落...")

    contexts = retrieve_contexts(
        query=scenario,
        faiss_index=faiss_index,
        chunks=chunks,
        embedding_model=embedding_model,
        bm25_index=bm25_index,
        tokenized_corpus=tokenized_corpus,
        k=k,
        method=retrieval_method,
        use_priority_weighting=use_priority_weighting,
    )

    if verbose:
        print(f"   找到 {len(contexts)} 個相關段落")
        for i, ctx in enumerate(contexts, 1):
            print(
                f"   [{i}] {ctx.get('doc_category', 'unknown'):15} "
                f"| {ctx['source']} p.{ctx['page']} "
                f"(score: {ctx['score']:.3f})"
            )

    # ===== Step 2: 組裝 Prompt =====
    if verbose:
        print("\n📝 Step 2: 組裝 Prompt...")

    user_prompt = build_user_prompt(scenario, contexts)

    if verbose:
        print(f"   Prompt 長度: {len(user_prompt)} 字元")
        print(f"\n🤖 Step 3: 呼叫 LLM ({llm_config['llm_model_name']})...")

    # ===== Step 3: 呼叫 LLM =====
    result = call_llm(SYSTEM_PROMPT, user_prompt, llm_config)  # ← [BUG FIX] 統一用 SYSTEM_PROMPT

    # 附加檢索資訊（供後續診斷）
    result["_retrieved_chunks"] = [
        {"chunk_id": ctx["chunk_id"], "source": ctx["source"],
         "page": ctx["page"], "score": ctx["score"]}
        for ctx in contexts
    ]

    if verbose:
        print("   ✅ 分析完成")
        print("=" * 60)

    return result

### 3.5 pretty_print_result()


In [ ]:
def pretty_print_result(result: dict):
    """美化輸出分析結果"""
    print("\n" + "=" * 60)
    print("📋 AML 紅旗分析報告")
    print("=" * 60)

    print(f"\n📌 情境摘要：{result.get('scenario_summary', 'N/A')}")

    assessment = result.get("assessment", "unknown")
    emoji_map = {
        "confirmed": "🚨", "possible": "⚠️", "unlikely": "✅",
        "refuse": "🚫", "error": "❌",
    }
    print(f"\n🎯 綜合評估：{emoji_map.get(assessment, '❓')} {assessment.upper()}")

    flags = result.get("identified_flags", [])
    if flags:
        print(f"\n🚩 識別到的紅旗 ({len(flags)} 個)：")
        print("-" * 40)
        for flag in flags:
            conf_emoji = {"high": "🔴", "medium": "🟡", "low": "🟢"}.get(
                flag.get("confidence"), "⚪"
            )
            print(f"\n  {conf_emoji} [{flag.get('flag_id')}] {flag.get('flag_name')}")
            print(f"     分析：{flag.get('reasoning', 'N/A')}")

            for ev in flag.get("evidence", []):
                quote = ev.get("quote", "")[:50]
                print(f"     📚 {ev.get('source')} p.{ev.get('page')}: 「{quote}...」")

    questions = result.get("follow_up_questions", [])
    if questions:
        print(f"\n❓ 建議追問：")
        for i, q in enumerate(questions, 1):
            print(f"   {i}. {q}")

    print(f"\n📜 {result.get('disclaimer', '本分析僅供教育參考，不構成法律意見。')}")
    print("=" * 60)

In [ ]:
print("\n✅ PART 3: LLM 分析模組已載入")
print(f"⚙️  使用 analyze_scenario() 統一入口")
print(f"   支援參數: use_priority_weighting / enable_gate")


✅ PART 3: LLM 分析模組已載入
⚙️  使用 analyze_scenario() 統一入口
   支援參數: use_priority_weighting / enable_gate


# 🧪 PART 4: 測試案例

這 16 題是測試「整體系統」的——LLM 能不能正確分析情境？

### 4.1 測試案例定義

In [ ]:
END_TO_END_TEST_CASES = {
    # ========== RF-01: 門檻拆分 (Structuring) ==========
    "1A": {
        "name": "門檻拆分 - confirmed",
        "scenario": """
某客戶在同一天內分 6 次臨櫃存入現金，每筆都略低於內部申報門檻。
存完後 30 分鐘內，將大部分金額分兩筆轉出到不同帳戶。
客戶被問到資金來源時，只回「朋友借我的」，無其他證明。
""",
        "expected_assessment": "confirmed",
        "expected_flags": ["RF-01", "RF-02"],
        "required_sources": ["TW_GOV", "FATF_VA"]
    },
    "1B": {
        "name": "門檻拆分 - possible",
        "scenario": """
某客戶最近開始頻繁存現，但每次金額都不大。
客戶說自己做小生意，現金多很正常。
你只知道「最近常存現」，不知道頻率與金額分布。
""",
        "expected_assessment": "possible",
        "expected_flags": ["RF-01"],
        "required_sources": ["TW_GOV"]
    },

    # ========== RF-02: 快速流轉 (Rapid Movement) ==========
    "2A": {
        "name": "快速流轉 - confirmed",
        "scenario": """
某帳戶凌晨收到一筆大額入帳，1 小時內分散轉出到 8 個不同對手方。
隔天又有另一筆大額入帳，重複相同模式。
帳戶持有人說「只是幫朋友周轉」，不清楚資金用途。
""",
        "expected_assessment": "confirmed",
        "expected_flags": ["RF-02"],
        "required_sources": ["FATF_VA", "FATF_TBML"]
    },
    "2B": {
        "name": "快速流轉 - possible",
        "scenario": """
某帳戶收到一筆入帳後很快轉出，但你不確定轉出對象有幾個。
客戶說轉出是「付貨款」，但沒提供發票或合約。
你也不確定這是否為該客戶常態行為。
""",
        "expected_assessment": "possible",
        "expected_flags": ["RF-02"],
        "required_sources": ["TW_GOV"]
    },

    # ========== RF-03: 現金密集且與客群不符 ==========
    "3A": {
        "name": "現金異常 - confirmed",
        "scenario": """
一名長期只用轉帳的小額上班族，突然開始每週大量存現。
存現後常在當日提領或立刻轉出，且拒絕說明現金來源。
其帳戶歷史與現金流量型態明顯不一致。
""",
        "expected_assessment": "confirmed",
        "expected_flags": ["RF-03"],
        "required_sources": ["TW_GOV"]
    },
    "3B": {
        "name": "現金異常 - possible",
        "scenario": """
某客戶常以現金交易，但他說自己是做餐飲小店。
你沒有他的營收規模，也沒有現金存入與營業型態的對照。
你只知道「現金很多」。
""",
        "expected_assessment": "possible",
        "expected_flags": ["RF-03"],
        "required_sources": ["TW_GOV"]
    },

    # ========== RF-04: 第三人代辦 / 人頭帳戶 ==========
    "4A": {
        "name": "第三人代辦 - confirmed",
        "scenario": """
某帳戶持有人是學生，但交易多由「叔叔」代為操作。
交易對象多為陌生公司帳戶，且資金常在入帳後迅速轉出。
當被問到受益人與資金用途時，持有人無法說明。
""",
        "expected_assessment": "confirmed",
        "expected_flags": ["RF-04", "RF-02"],
        "required_sources": ["TW_GOV", "FATF_VA"]
    },
    "4B": {
        "name": "第三人代辦 - possible",
        "scenario": """
客戶說是「幫家人收款」，但你不知道家人身分。
帳戶有轉入轉出，但金額大小與頻率未提供。
客戶不願意留下任何聯絡資訊。
""",
        "expected_assessment": "possible",
        "expected_flags": ["RF-04"],
        "required_sources": ["TW_GOV"]
    },

    # ========== RF-05: 跨境 / 高風險地區 ==========
    "5A": {
        "name": "跨境高風險 - confirmed",
        "scenario": """
某客戶在短時間內將資金分批匯往多個境外帳戶。
其中包含高風險或非合作地區，且客戶無法說明貿易或投資理由。
匯出後又很快有資金回流到其他帳戶。
""",
        "expected_assessment": "confirmed",
        "expected_flags": ["RF-05"],
        "required_sources": ["FATF_VA", "TW_GOV"]
    },
    "5B": {
        "name": "跨境高風險 - possible",
        "scenario": """
客戶說要匯款到海外親友，但沒有提供親友關係證明。
你不知道匯款頻率，也不知道收款地區風險屬性。
客戶表示「急用錢」要求立即處理。
""",
        "expected_assessment": "possible",
        "expected_flags": ["RF-05"],
        "required_sources": ["TW_GOV"]
    },

    # ========== RF-06: 與身分 / 商業模式不符 ==========
    "6A": {
        "name": "與身分不符 - confirmed",
        "scenario": """
某小型文創工作室帳戶，卻持續收取大量與其業務無關的款項。
收款後大多立即轉出至不同個人帳戶，且無合約或發票。
負責人無法說明交易對手與資金用途。
""",
        "expected_assessment": "confirmed",
        "expected_flags": ["RF-06", "RF-02"],
        "required_sources": ["FATF_TBML", "TW_GOV"]
    },
    "6B": {
        "name": "與身分不符 - possible",
        "scenario": """
客戶說自己是自由接案者，所以收款來源很多。
你不知道這些來源是平台撥款、還是陌生個人匯款。
你也沒有客戶過往交易型態可對照。
""",
        "expected_assessment": "possible",
        "expected_flags": ["RF-06"],
        "required_sources": ["TW_GOV"]
    },

    # ========== RF-07: 虛擬資產匿名性 ==========
    "7A": {
        "name": "虛擬資產匿名 - confirmed",
        "scenario": """
客戶將多筆資金轉入加密貨幣交易所後，立刻提到非託管錢包。
隔天又從不同地址把資金打回來，並說不清楚交易對象是誰。
客戶提到使用 OTC 私下交易且不留紀錄。
""",
        "expected_assessment": "confirmed",
        "expected_flags": ["RF-07"],
        "required_sources": ["FATF_VA"]
    },
    "7B": {
        "name": "虛擬資產匿名 - possible",
        "scenario": """
客戶說自己有在買比特幣，但只是長期投資。
你只知道他有出入金交易所，卻不知道頻率與金額。
你也不知道他是否提到外部錢包或私下交易。
""",
        "expected_assessment": "possible",
        "expected_flags": ["RF-07"],
        "required_sources": ["FATF_VA", "TW_GOV"]
    },

    # ========== RF-08: 公司 / 受益所有人不透明 ==========
    "8A": {
        "name": "公司不透明 - confirmed",
        "scenario": """
某新成立公司要求開戶並快速進行大額收付款，但無明確營業內容。
負責人無法說明實際受益人，股權結構多層且難以釐清。
交易對手遍布多地，且用途敘述反覆變更。
""",
        "expected_assessment": "confirmed",
        "expected_flags": ["RF-08"],
        "required_sources": ["FATF_TBML", "TW_GOV"]
    },
    "8B": {
        "name": "TBML 超出範圍 - refuse",
        "scenario": """
客戶從事「國際貿易」，交易涉及多種貨物與國家。
你懷疑可能有貿易型洗錢（TBML），但你手上的教材沒有任何 TBML 章節。
你也沒有報關單、發票、貨物流向等資料。
""",
        "expected_assessment": "refuse",
        "expected_flags": [],
        "required_sources": []
    }
}

### 4.2 單一情境測試


In [ ]:
# 測試 confirmed 案例
print("⚙️ 本次實驗設定:")
print(f"   LLM: {SELECTED_CONFIG['llm_model_name']}")
print(f"   Priority Weighting: {DEFAULT_USE_PRIORITY_WEIGHTING}")
print(f"   Gate: {DEFAULT_ENABLE_GATE}")

⚙️ 本次實驗設定:
   LLM: llama-3.1-8b-instant
   Priority Weighting: True
   Gate: False


In [ ]:
result = analyze_scenario(
    scenario=END_TO_END_TEST_CASES["2B"]["scenario"],
    faiss_index=faiss_index,
    chunks=chunks,
    embedding_model=embedding_model,
    bm25_index=bm25_index,
    tokenized_corpus=tokenized_corpus,
    llm_config=SELECTED_CONFIG,
    use_priority_weighting=True,
    enable_gate=False,       # ← 啟用 Gate
)
pretty_print_result(result)

🔍 Step 1: 檢索相關段落...
   找到 5 個相關段落
   [1] core            | FATF p.8 (score: 0.016)
   [2] core            | FATF p.8 (score: 0.016)
   [3] core            | FATF p.8 (score: 0.014)
   [4] sector_specific | FATF p.14 (score: 0.014)
   [5] sector_specific | FATF p.15 (score: 0.014)

📝 Step 2: 組裝 Prompt...
   Prompt 長度: 2308 字元

🤖 Step 3: 呼叫 LLM (llama-3.1-8b-instant)...
   ✅ 分析完成

📋 AML 紅旗分析報告

📌 情境摘要：客戶入帳後快速轉出，未提供發票或合約，轉出對象不明

🎯 綜合評估：⚠️ POSSIBLE

🚩 識別到的紅旗 (1 個)：
----------------------------------------

  🟡 [RF-02] 快速流轉
     分析：客戶入帳後快速轉出，可能與快速流轉的紅旗特徵相符
     📚 FATF p.8: 「An account displays an unexpectedly high number or...」

❓ 建議追問：
   1. 轉出對象的具體資訊是什麼?
   2. 客戶是否有提供合理的理由?

📜 本分析僅供教育參考，不構成法律意見。


In [ ]:
result = analyze_scenario(
    scenario=END_TO_END_TEST_CASES["6A"]["scenario"],
    faiss_index=faiss_index,
    chunks=chunks,
    embedding_model=embedding_model,
    bm25_index=bm25_index,
    tokenized_corpus=tokenized_corpus,
    llm_config=SELECTED_CONFIG,
    use_priority_weighting=True,
    enable_gate=False,       # ← 啟用 Gate
)
pretty_print_result(result)

🔍 Step 1: 檢索相關段落...
   找到 5 個相關段落
   [1] knowledge_bridge | TW_Gov p.6 (score: 0.025)
   [2] core            | FATF p.8 (score: 0.016)
   [3] core            | FATF p.8 (score: 0.016)
   [4] core            | FATF p.8 (score: 0.016)
   [5] sector_specific | FATF p.10 (score: 0.014)

📝 Step 2: 組裝 Prompt...
   Prompt 長度: 2090 字元

🤖 Step 3: 呼叫 LLM (llama-3.1-8b-instant)...
   ✅ 分析完成

📋 AML 紅旗分析報告

📌 情境摘要：小型文創工作室收取大量與業務無關的款項，迅速轉出至不同個人帳戶

🎯 綜合評估：🚨 CONFIRMED

🚩 識別到的紅旗 (2 個)：
----------------------------------------

  🔴 [RF-02] 快速流轉
     分析：收款後大多立即轉出至不同個人帳戶，且無合約或發票
     📚 FATF p.8: 「Incoming wire transfers to a trade-related account...」
     📚 FATF p.8: 「An account displays frequent deposits in cash whic...」

  🟡 [RF-06] 與身分不符
     分析：負責人無法說明交易對手與資金用途
     📚 TW_Gov p.6: 「疑似使⽤用假名、⼈人頭、虛設⾏行號或虛設法⼈人團體開設帳⼾戶...」

❓ 建議追問：
   1. 具體交易對手是誰?
   2. 交易對手與資金用途

📜 本分析僅供教育參考，不構成法律意見。


In [ ]:
# 測試 confirmed 案例
result = analyze_scenario(
    scenario=END_TO_END_TEST_CASES["1A"]["scenario"],
    faiss_index=faiss_index,
    chunks=chunks,
    embedding_model=embedding_model,
    bm25_index=bm25_index,
    tokenized_corpus=tokenized_corpus,
    llm_config=SELECTED_CONFIG,
    use_priority_weighting=True,
    enable_gate=False,       # ← 啟用 Gate
)
pretty_print_result(result)

🔍 Step 1: 檢索相關段落...
   找到 5 個相關段落
   [1] knowledge_bridge | TW_Gov p.8 (score: 0.026)
   [2] core            | FATF p.8 (score: 0.016)
   [3] sector_specific | FATF p.7 (score: 0.015)
   [4] core            | FATF p.8 (score: 0.014)
   [5] sector_specific | FATF p.9 (score: 0.014)

📝 Step 2: 組裝 Prompt...
   Prompt 長度: 2158 字元

🤖 Step 3: 呼叫 LLM (llama-3.1-8b-instant)...
   ✅ 分析完成

📋 AML 紅旗分析報告

📌 情境摘要：客戶在同一天內分 6 次臨櫃存入現金，每筆都略低於內部申報門檻，隨後轉出到不同帳戶，且無法提供資金來源證明。

🎯 綜合評估：🚨 CONFIRMED

🚩 識別到的紅旗 (2 個)：
----------------------------------------

  🔴 [RF-01] 門檻拆分
     分析：客戶在同一天內分 6 次臨櫃存入現金，每筆都略低於內部申報門檻，顯示刻意分批將金額拆解以規避洗錢防制程序。
     📚 TW_GOV p.8: 「若刻意分批將400萬元現金拆解為各筆50萬元以下交易，顯然為規避洗錢防制法第9條之規定...」
     📚 FATF p.8: 「An account displays frequent deposits in cash whic...」

  🟡 [RF-03] 現金異常
     分析：客戶存入大量現金後迅速轉出，可能與洗錢行為相關。
     📚 FATF p.7: 「Making multiple high-value transactions – in short...」

❓ 建議追問：
   1. 客戶的資金來源是什麼?
   2. 客戶的交易目的是什麼?

📜 本分析僅供教育參考，不構成法律意見。


In [ ]:
# 測試 refuse 案例（含 Gate）
result = analyze_scenario(
    scenario=END_TO_END_TEST_CASES["8B"]["scenario"],
    faiss_index=faiss_index,
    chunks=chunks,
    embedding_model=embedding_model,
    bm25_index=bm25_index,
    tokenized_corpus=tokenized_corpus,
    llm_config=SELECTED_CONFIG,
    use_priority_weighting=True,
    enable_gate=True,       # ← 啟用 Gate
)
pretty_print_result(result)

🚦 Step 0: Pre-LLM Gate 檢查...
🚫 Gate: 明確知識缺口 — ['TBML']
   🚫 決策: REFUSE
   原因: 情境明確表示缺乏 TBML 相關教材，無法進行分析

📋 AML 紅旗分析報告

📌 情境摘要：情境明確表示缺乏 TBML 相關教材，無法進行分析

🎯 綜合評估：🚫 REFUSE

📜 本分析僅供教育參考，不構成法律意見。


### 4.3 批次測試（可選）


In [ ]:
def run_all_tests(
    test_cases: dict,
    faiss_index,
    chunks: list,
    embedding_model,
    bm25_index,
    tokenized_corpus: list,
    llm_config: dict,
    use_priority_weighting: bool = DEFAULT_USE_PRIORITY_WEIGHTING,
    enable_gate: bool = DEFAULT_ENABLE_GATE,
    enable_logging: bool = True,
    version: str = "v2.0",
    experiment_type: str = "baseline",
):
    """
    批次執行所有測試案例

    Args:
        test_cases: END_TO_END_TEST_CASES 格式的 dict
        enable_gate: 是否啟用 Pre-LLM Gate
        enable_logging: 是否啟用實驗記錄
        version: 實驗版本號
        experiment_type: 實驗類型（baseline / fix / tuning / ab_test / debug）

    Returns:
        list[dict]: 每個 dict 包含 id, passed, expected, actual, gate_decision

    History:
        - v1: 基礎批次測試（run_all_tests）
        - v2: 整合 Gate 支援（合併 run_all_tests_with_gate）
    """
    results = []
    passed_count = 0
    total_count = len(test_cases)

    # Logger 初始化
    logger = None
    if enable_logging:
        config = {
            **llm_config,
            "use_priority_weighting": use_priority_weighting,
            "enable_gate": enable_gate,
            "k": DEFAULT_TOP_K,
            "retrieval_method": "hybrid",
            "rrf_k": DEFAULT_RRF_K,
            "index_version": index_metadata.get("version", "unknown"),
        }

        logger = ExperimentLogger(
            base_dir=EXPERIMENT_ROOT_DIR,
            version=version,
            experiment_type=experiment_type,
            # config=config,
            # auto_describe=True,
        )
        logger.log_config(config)

    for test_id, test in test_cases.items():
        print(f"\n🧪 測試 {test_id}: {test['name']}")

        try:
            result = analyze_scenario(
                scenario=test["scenario"],
                faiss_index=faiss_index,
                chunks=chunks,
                embedding_model=embedding_model,
                bm25_index=bm25_index,
                tokenized_corpus=tokenized_corpus,
                llm_config=llm_config,
                use_priority_weighting=use_priority_weighting,
                enable_gate=enable_gate,
                verbose=False,
            )

            actual = result.get("assessment", "unknown")
            expected = test["expected_assessment"]
            passed = actual == expected

            if passed:
                passed_count += 1

            gate_meta = result.get("_gate_metadata", {})
            gate_decision = gate_meta.get("decision", "LLM")

            status = "✅" if passed else "❌"
            print(f"   預期: {expected} | 實際: {actual} | 來源: {gate_decision} | {status}")

            results.append({
                "id": test_id,
                "passed": passed,
                "expected": expected,
                "actual": actual,
                "gate_decision": gate_decision,
            })

            if logger:
                logger.log_case(
                    case_id=test_id,
                    scenario=test["scenario"],
                    expected=expected,
                    actual=actual,
                    passed=passed,
                    retrieved_chunks=result.get("_retrieved_chunks", []),
                    llm_response={
                        "assessment": actual,
                        "flags": result.get("identified_flags", []),
                    },
                )

        except Exception as e:
            print(f"   ❌ 錯誤: {e}")
            results.append({"id": test_id, "passed": False, "error": str(e)})

            if logger:
                logger.log_case(
                    case_id=test_id,
                    scenario=test["scenario"],
                    expected=test["expected_assessment"],
                    actual="error",
                    passed=False,
                    error=str(e),
                )

    # 統計
    accuracy = passed_count / total_count if total_count > 0 else 0
    gate_refused = sum(1 for r in results if r.get("gate_decision") == "REFUSE")

    print(f"\n📊 總結: {passed_count}/{total_count} 通過 ({accuracy:.1%})")
    if enable_gate:
        print(f"🚦 Gate 攔截: {gate_refused} 個案例")

    if logger:
        metrics = {
            "total_cases": total_count,
            "passed": passed_count,
            "failed": total_count - passed_count,
            "accuracy": accuracy,
            "gate_refused": gate_refused,
        }
        logger.log_metrics(metrics)
        logger.log_batch_results(results)
        logger.finalize(summary=f"準確率: {accuracy:.1%} ({passed_count}/{total_count})")

    return results

In [ ]:
# 執行批次測試
print("\n" + "=" * 60)
print("🚀 PART 4: 批次測試所有情境...")
print("=" * 60)

all_test_results = run_all_tests(
    test_cases=END_TO_END_TEST_CASES,
    faiss_index=faiss_index,
    chunks=chunks,
    embedding_model=embedding_model,
    bm25_index=bm25_index,
    tokenized_corpus=tokenized_corpus,
    llm_config=SELECTED_CONFIG,
    use_priority_weighting=True,
    enable_gate=True,            # ← 啟用 Gate
    enable_logging=ENABLE_LOGGING
)

print("\n✅ 所有批次測試完成！")


🚀 PART 4: 批次測試所有情境...
📁 實驗目錄: v2.0_20260212_baseline_expand_KS
✅ 配置已保存

🧪 測試 1A: 門檻拆分 - confirmed
   預期: confirmed | 實際: confirmed | 來源: LLM | ✅
  ✅ 1A: confirmed → confirmed

🧪 測試 1B: 門檻拆分 - possible
   預期: possible | 實際: possible | 來源: LLM | ✅
  ✅ 1B: possible → possible

🧪 測試 2A: 快速流轉 - confirmed
   預期: confirmed | 實際: confirmed | 來源: LLM | ✅
  ✅ 2A: confirmed → confirmed

🧪 測試 2B: 快速流轉 - possible
   預期: possible | 實際: possible | 來源: LLM | ✅
  ✅ 2B: possible → possible

🧪 測試 3A: 現金異常 - confirmed
   預期: confirmed | 實際: confirmed | 來源: LLM | ✅
  ✅ 3A: confirmed → confirmed

🧪 測試 3B: 現金異常 - possible
   預期: possible | 實際: possible | 來源: LLM | ✅
  ✅ 3B: possible → possible

🧪 測試 4A: 第三人代辦 - confirmed
   預期: confirmed | 實際: confirmed | 來源: LLM | ✅
  ✅ 4A: confirmed → confirmed

🧪 測試 4B: 第三人代辦 - possible
   預期: possible | 實際: possible | 來源: LLM | ✅
  ✅ 4B: possible → possible

🧪 測試 5A: 跨境高風險 - confirmed
   預期: confirmed | 實際: confirmed | 來源: LLM | ✅
  ✅ 5A: confirmed → confirmed

🧪 測試 5B:

# PART 5: 診斷工具（可選執行）
這部分用於驗證「權重調整」是否真的改善了系統表現。
在確認有效後，可以不執行這部分。

以下情況發生時，必須進行診斷：

* 新增資料源：如果你引進了第三種資料（例如：銀行內部規章），你需要重跑診斷，確認新資料不會把 core 法規擠下去。

* 更換 Embedding 模型：如果你改用了不同廠牌的向量模型，空間分佈會變，權重可能需要微調。

* 發現新的「幻覺」：如果之後有題目該拒答卻沒拒答，就得回來這看是不是 knowledge_bridge 又爬到第一名了。

### 5.1 單一案例檢索診斷

In [ ]:
def diagnose_single_case(test_id: str, show_chunks: bool = False):
    """
    診斷單個測試案例的檢索結果
    對比有權重 vs 無權重的排序差異
    """
    test = END_TO_END_TEST_CASES[test_id]

    print(f"\n{'=' * 70}")
    print(f"🔍 診斷 {test_id}: {test['name']}")
    print(f"{'=' * 70}")

    print(f"\n📋 預期結果：")
    print(f"   Assessment: {test['expected_assessment']}")
    print(f"   Flags: {test['expected_flags']}")
    print(f"   Required Sources: {test.get('required_sources', [])}")

    # 無權重
    results_unweighted = hybrid_search(
        query=test["scenario"],
        faiss_index=faiss_index,
        chunks=chunks,
        embedding_model=embedding_model,
        bm25_index=bm25_index,
        tokenized_corpus=tokenized_corpus,
        k=5,
        use_priority_weighting=False,
    )

    print(f"\n  【無權重調整】")
    for i, r in enumerate(results_unweighted, 1):
        chunk = r["chunk"]
        cat = chunk.get("doc_category", "unknown")
        print(f"    [{i}] {cat:18} | Score: {r['score']:.4f} | {chunk.get('source')} p.{chunk.get('page')}")

    # 有權重
    results_weighted = hybrid_search(
        query=test["scenario"],
        faiss_index=faiss_index,
        chunks=chunks,
        embedding_model=embedding_model,
        bm25_index=bm25_index,
        tokenized_corpus=tokenized_corpus,
        k=5,
        use_priority_weighting=True,
    )

    print(f"\n  【有權重調整】")
    for i, r in enumerate(results_weighted, 1):
        chunk = r["chunk"]
        cat = chunk.get("doc_category", "unknown")
        priority = r.get("priority_weight", 1.0)
        raw = r.get("raw_rrf_score", 0)
        print(f"    [{i}] {cat:18} | Score: {r['score']:.4f} (raw: {raw:.4f} × {priority:.1f}) | {chunk.get('source')} p.{chunk.get('page')}")

    # 分析變化
    print(f"\n  📊 排序變化：")
    unweighted_cats = [r["chunk"].get("doc_category") for r in results_unweighted]
    weighted_cats = [r["chunk"].get("doc_category") for r in results_weighted]

    if unweighted_cats != weighted_cats:
        print(f"    ⚠️ 權重調整改變了排序")
        for i, (u_cat, w_cat) in enumerate(zip(unweighted_cats, weighted_cats), 1):
            if u_cat != w_cat:
                print(f"       排名{i}: {u_cat} → {w_cat}")
    else:
        print(f"    ✅ 權重調整沒有改變排序")

    if show_chunks:
        print(f"\n  📝 Chunk 內容預覽（加權後）：")
        for i, r in enumerate(results_weighted, 1):
            print(f"\n    【Chunk {i}】")
            print(f"    {r['chunk']['text'][:200]}...")

    return results_weighted

### 5.2 A/B 對比測試

In [ ]:
def ab_test_comparison(test_cases: dict):
    """對比有權重 vs 無權重的測試結果"""
    print(f"\n{'=' * 70}")
    print(f"⚖️ A/B 對比測試：權重調整的影響")
    print(f"{'=' * 70}")

    # 無權重版本
    print(f"\n【版本 A：無權重調整】")
    results_a = run_all_tests(
        test_cases=test_cases,
        faiss_index=faiss_index,
        chunks=chunks,
        embedding_model=embedding_model,
        bm25_index=bm25_index,
        tokenized_corpus=tokenized_corpus,
        llm_config=SELECTED_CONFIG,
        use_priority_weighting=False,
        enable_gate=False,
    )

    # 有權重版本
    print(f"\n{'=' * 70}")
    print(f"【版本 B：有權重調整】")
    results_b = run_all_tests(
        test_cases=test_cases,
        faiss_index=faiss_index,
        chunks=chunks,
        embedding_model=embedding_model,
        bm25_index=bm25_index,
        tokenized_corpus=tokenized_corpus,
        llm_config=SELECTED_CONFIG,
        use_priority_weighting=True,
        enable_gate=False,
    )

    # 對比分析
    print(f"\n{'=' * 70}")
    print(f"📊 對比分析")
    print(f"{'=' * 70}")

    pass_a = sum(1 for r in results_a if r.get("passed", False))
    pass_b = sum(1 for r in results_b if r.get("passed", False))
    total = len(test_cases)

    print(f"\n整體準確率：")
    print(f"  版本 A（無權重）: {pass_a}/{total} ({pass_a/total*100:.1f}%)")
    print(f"  版本 B（有權重）: {pass_b}/{total} ({pass_b/total*100:.1f}%)")
    print(f"  改善: {pass_b - pass_a:+d} 個案例")

    # 找出差異案例
    print(f"\n差異案例：")
    for test_id in test_cases.keys():
        result_a = next((r for r in results_a if r["id"] == test_id), None)
        result_b = next((r for r in results_b if r["id"] == test_id), None)

        if result_a and result_b:
            passed_a = result_a.get("passed", False)
            passed_b = result_b.get("passed", False)

            if passed_a != passed_b:
                status = "✅ 修正" if passed_b else "❌ 退化"
                print(f"  {test_id}: {status}")
                print(f"    版本 A: {result_a.get('actual', 'N/A')}")
                print(f"    版本 B: {result_b.get('actual', 'N/A')}")
                print(f"    預期: {test_cases[test_id]['expected_assessment']}")

    return results_a, results_b

### 5.3 執行診斷（取消註解即可執行）

In [ ]:
# diagnose_single_case("8A", show_chunks=False)
# diagnose_single_case("8B", show_chunks=False)
# ab_test_comparison(END_TO_END_TEST_CASES)


🔍 診斷 8A: 公司不透明 - confirmed

📋 預期結果：
   Assessment: confirmed
   Flags: ['RF-08']
   Required Sources: ['FATF_TBML', 'TW_GOV']

  【無權重調整】
    [1] core               | Score: 0.0164 | FATF p.8
    [2] knowledge_bridge   | Score: 0.0164 | TW_Gov p.8
    [3] knowledge_bridge   | Score: 0.0161 | TW_Gov p.5
    [4] core               | Score: 0.0161 | FATF p.5
    [5] sector_specific    | Score: 0.0159 | FATF p.5

  【有權重調整】
    [1] core               | Score: 0.0164 (raw: 0.0164 × 1.0) | FATF p.8
    [2] core               | Score: 0.0161 (raw: 0.0161 × 1.0) | FATF p.5
    [3] core               | Score: 0.0152 (raw: 0.0152 × 1.0) | FATF p.6
    [4] core               | Score: 0.0147 (raw: 0.0147 × 1.0) | FATF p.8
    [5] sector_specific    | Score: 0.0143 (raw: 0.0159 × 0.9) | FATF p.5

  📊 排序變化：
    ⚠️ 權重調整改變了排序
       排名2: knowledge_bridge → core
       排名3: knowledge_bridge → core

🔍 診斷 8B: TBML 超出範圍 - refuse

📋 預期結果：
   Assessment: refuse
   Flags: []
   Required Sources: []

  【無權重調整】

([{'id': '1A',
   'passed': True,
   'expected': 'confirmed',
   'actual': 'confirmed',
   'gate_decision': 'LLM'},
  {'id': '1B',
   'passed': True,
   'expected': 'possible',
   'actual': 'possible',
   'gate_decision': 'LLM'},
  {'id': '2A',
   'passed': True,
   'expected': 'confirmed',
   'actual': 'confirmed',
   'gate_decision': 'LLM'},
  {'id': '2B',
   'passed': True,
   'expected': 'possible',
   'actual': 'possible',
   'gate_decision': 'LLM'},
  {'id': '3A',
   'passed': True,
   'expected': 'confirmed',
   'actual': 'confirmed',
   'gate_decision': 'LLM'},
  {'id': '3B',
   'passed': True,
   'expected': 'possible',
   'actual': 'possible',
   'gate_decision': 'LLM'},
  {'id': '4A',
   'passed': True,
   'expected': 'confirmed',
   'actual': 'confirmed',
   'gate_decision': 'LLM'},
  {'id': '4B',
   'passed': True,
   'expected': 'possible',
   'actual': 'possible',
   'gate_decision': 'LLM'},
  {'id': '5A',
   'passed': True,
   'expected': 'confirmed',
   'actual': 'conf

In [ ]:
# # 執行 A/B 測試（可選，因為會跑兩次所有測試）
# ab_test_comparison(END_TO_END_TEST_CASES)


⚖️ A/B 對比測試：權重調整的影響

【版本 A：無權重調整】

🧪 測試 1A: 門檻拆分 - confirmed
   預期: confirmed | 實際: confirmed | 來源: LLM | ✅

🧪 測試 1B: 門檻拆分 - possible
   預期: possible | 實際: possible | 來源: LLM | ✅

🧪 測試 2A: 快速流轉 - confirmed
   預期: confirmed | 實際: confirmed | 來源: LLM | ✅

🧪 測試 2B: 快速流轉 - possible
   預期: possible | 實際: possible | 來源: LLM | ✅

🧪 測試 3A: 現金異常 - confirmed
   預期: confirmed | 實際: confirmed | 來源: LLM | ✅

🧪 測試 3B: 現金異常 - possible
   預期: possible | 實際: possible | 來源: LLM | ✅

🧪 測試 4A: 第三人代辦 - confirmed
   預期: confirmed | 實際: confirmed | 來源: LLM | ✅

🧪 測試 4B: 第三人代辦 - possible
   預期: possible | 實際: possible | 來源: LLM | ✅

🧪 測試 5A: 跨境高風險 - confirmed
   預期: confirmed | 實際: confirmed | 來源: LLM | ✅

🧪 測試 5B: 跨境高風險 - possible
   預期: possible | 實際: possible | 來源: LLM | ✅

🧪 測試 6A: 與身分不符 - confirmed
   預期: confirmed | 實際: confirmed | 來源: LLM | ✅

🧪 測試 6B: 與身分不符 - possible
   預期: possible | 實際: possible | 來源: LLM | ✅

🧪 測試 7A: 虛擬資產匿名 - confirmed
   預期: confirmed | 實際: confirmed | 來源: LLM | ✅

🧪 測試 7B: 虛擬資

([{'id': '1A',
   'passed': True,
   'expected': 'confirmed',
   'actual': 'confirmed',
   'gate_decision': 'LLM'},
  {'id': '1B',
   'passed': True,
   'expected': 'possible',
   'actual': 'possible',
   'gate_decision': 'LLM'},
  {'id': '2A',
   'passed': True,
   'expected': 'confirmed',
   'actual': 'confirmed',
   'gate_decision': 'LLM'},
  {'id': '2B',
   'passed': True,
   'expected': 'possible',
   'actual': 'possible',
   'gate_decision': 'LLM'},
  {'id': '3A',
   'passed': True,
   'expected': 'confirmed',
   'actual': 'confirmed',
   'gate_decision': 'LLM'},
  {'id': '3B',
   'passed': True,
   'expected': 'possible',
   'actual': 'possible',
   'gate_decision': 'LLM'},
  {'id': '4A',
   'passed': True,
   'expected': 'confirmed',
   'actual': 'confirmed',
   'gate_decision': 'LLM'},
  {'id': '4B',
   'passed': True,
   'expected': 'possible',
   'actual': 'possible',
   'gate_decision': 'LLM'},
  {'id': '5A',
   'passed': True,
   'expected': 'confirmed',
   'actual': 'conf

# 🔄 PART 6: Multi-Turn Conversation（多輪對話模組）

架構：
  history → query rewrite → hybrid retrieval → LLM answer

設計理由：
  追問（如「剛才提到的門檻是多少？」）如果直接拿去檢索，
  會因為缺乏上下文而搜到完全無關的 chunks。
  所以在檢索前先用 LLM 把追問「改寫」成獨立查詢，
  讓 retrieval 拿到的是一個語意完整的 query。

與既有模組的關係：
  - 複用 PART 1 的 pre_llm_gate()
  - 複用 PART 2 的 retrieve_contexts()
  - 複用 PART 3 的 build_user_prompt(), call_llm(), SYSTEM_PROMPT
  - 不修改任何既有函數，只新增

History:
  - v3.1 (2025-xx): 新增多輪對話支援

### 6.0 Config

In [ ]:
MAX_HISTORY_TURNS = 3  # 保留最近 N 輪（1 輪 = 1 user + 1 assistant）

# Query Rewrite 用的輕量 prompt（不需要完整的 AML SYSTEM_PROMPT）
REWRITE_SYSTEM_PROMPT = """你是一個查詢改寫助手。

你的唯一任務：根據對話歷史，把使用者的「追問」改寫成一個「獨立的查詢句」，
讓這個句子不需要任何上下文就能被搜尋引擎理解。

規則：
1. 如果使用者的問題已經是獨立的（不需要上下文就能理解），直接原樣回傳
2. 如果是追問，把代名詞、指示詞（「剛才那個」「上面提到的」）替換成具體內容
3. 只輸出改寫後的查詢句，不要加任何解釋
4. 保持原本的語言（中文問就中文答）

範例：
歷史：用戶問「某客戶連續三天分批存款，每次49萬」→ 助教分析了 RF-01 門檻拆分
追問：「台灣法規規定的門檻金額是多少？」
改寫：「台灣洗錢防制法規定的現金交易申報門檻金額是多少？」

歷史：用戶問了虛擬資產混幣的問題 → 助教提到了 RF-07
追問：「FATF 對這類行為還列了哪些紅旗指標？」
改寫：「FATF 對虛擬資產混幣服務相關的紅旗指標有哪些？」

歷史：（無）
問題：「什麼是洗錢？」
改寫：「什麼是洗錢？」
"""

print("✅ PART 6: Multi-Turn Config 已載入")

✅ PART 6: Multi-Turn Config 已載入


### 6.1 rewrite_query() — 查詢改寫

問題：
* chat() 把 assistant 回應存成 JSON blob（{"assessment":...}）
* rewrite_query() 把這個 JSON blob 當對話歷史餵給 8B
* 8B 要從 JSON 裡「理解之前聊了什麼」→ 語意理解品質很差
* → rewrite 丟失 entity（crypto 消失）、intent 被泛化

修正：
1.  chat() 同時存兩種格式：
```
content      → JSON（給 build_multiturn_user_prompt 用，原邏輯不變）
content_readable → 自然語言摘要（給 rewrite_query 用）
```
2.  
```
rewrite_query() 優先取 content_readable
```

影響範圍：
  - chat(): 第 361-370 行（history 存儲），新增一個欄位
  - rewrite_query(): 第 96-101 行（history 讀取），改取法
  - build_multiturn_user_prompt(): 不需要改（它讀 content，不受影響）
  - chat_loop(): 不需要改
  - run_e2e_tests / analyze_scenario: 不涉及多輪，不受影響

In [ ]:
def rewrite_query(
    user_input: str,
    history: list,
    llm_config: dict = None,
    verbose: bool = True,
) -> str:
    """
    根據對話歷史，將追問改寫為獨立查詢

    Pipeline 位置：history → [rewrite_query] → gate → retrieval → LLM

    History:
        - v3.1-fix-1: text 模式修復 Groq 400 error
        - v3.1-fix-2: content_readable 讓 8B 看到自然語言
        - v3.1-fix-3: 注入 conversation_state，8B 不用自己推斷上下文
    """
    if llm_config is None:
        llm_config = SELECTED_CONFIG

    # === 快速路徑：沒有歷史，不需要改寫 ===
    if not history:
        if verbose:
            print("   📝 Query Rewrite: 跳過（無歷史）")
        return user_input

    # === 從 history 取出 conversation_state ===
    state_block = ""
    last_assistant_turns = [t for t in history if t["role"] == "assistant"]
    if last_assistant_turns:
        state = last_assistant_turns[-1].get("conversation_state", {})

        parts = []
        if state.get("scenario_origin"):
            parts.append(f"核心情境：{state['scenario_origin'][:200]}")
        if state.get("active_flags"):
            parts.append(f"已識別紅旗：{', '.join(state['active_flags'])}")
        if state.get("scenario_summary"):
            parts.append(f"最新分析摘要：{state['scenario_summary'][:150]}")

        if parts:
            state_block = "【對話狀態】\n" + "\n".join(parts) + "\n\n"

    # === 組裝改寫用的 prompt ===
    recent = history[-(MAX_HISTORY_TURNS * 2):]

    history_text = ""
    for turn in recent:
        role = "用戶" if turn["role"] == "user" else "助教"
        content = turn.get("content_readable", turn["content"])[:300]
        history_text += f"【{role}】{content}\n"

    rewrite_user_prompt = f"""{state_block}對話歷史：
{history_text}

使用者現在的問題：
{user_input}

請改寫成獨立查詢："""

    # === 呼叫 LLM 進行改寫（text 模式）===
    try:
        rewritten = call_llm(
            system_prompt=REWRITE_SYSTEM_PROMPT,
            user_prompt=rewrite_user_prompt,
            llm_config=llm_config,
            response_format="text",
        )

        if rewritten is None:
            if verbose:
                print(f"   ⚠️ Query Rewrite API 失敗，使用原句")
            return user_input

        rewritten = rewritten.strip().strip('"').strip("'")

        # 8B 模型有時會把 prompt 範例的 label 帶進輸出
        if rewritten.startswith("改寫：") or rewritten.startswith("改寫:"):
            rewritten = rewritten.split("：", 1)[-1].split(":", 1)[-1].strip()

        if not rewritten:
            if verbose:
                print(f"   ⚠️ Query Rewrite 回傳空字串，使用原句")
            return user_input

        if verbose:
            if rewritten != user_input:
                print(f"   📝 Query Rewrite: 已改寫")
                print(f"      原句：{user_input}")
                print(f"      改寫：{rewritten}")
            else:
                print(f"   📝 Query Rewrite: 不需改寫（獨立查詢）")

        # 8B 有時把原句 + "改寫：" + 改寫結果串在一起
        if "改寫：" in rewritten or "改寫:" in rewritten:
        # 取「改寫：」之後的部分
            for delimiter in ["改寫：", "改寫:"]:
                if delimiter in rewritten:
                    rewritten = rewritten.split(delimiter)[-1].strip()
                    break

        return rewritten

    except Exception as e:
        if verbose:
            print(f"   ⚠️ Query Rewrite 失敗，使用原句: {e}")
        return user_input

### 6.2 build_multiturn_user_prompt() — 含歷史的 Prompt 組裝

In [ ]:
def build_multiturn_user_prompt(
    scenario: str,
    contexts: list,
    history: list = None,
    rewritten_query: str = None,
) -> str:
    """
    組裝多輪對話的 user prompt

    與 build_user_prompt() 的差異：
    - 注入對話歷史摘要（讓 LLM 知道之前聊了什麼）
    - 顯示改寫後的查詢（讓 LLM 知道檢索依據）
    - 指示 LLM 回應追問時不要重複分析整個情境

    Args:
        scenario: 使用者的原始輸入
        contexts: retrieve_contexts() 的輸出
        history: 對話歷史
        rewritten_query: 改寫後的查詢（用於透明度）

    Returns:
        str: 完整的 user prompt
    """
    # --- 組裝 context（複用既有邏輯）---
    context_str = ""
    for i, ctx in enumerate(contexts, 1):
        style_tag = {
            "authoritative": "【權威來源】",
            "simplified": "【簡化說明】",
            "technical": "【技術細節】",
        }.get(ctx.get("explanation_style", "neutral"), "")

        context_str += f"""
### 段落 {i} {style_tag}
- 來源：{ctx['source']}，第 {ctx['page']} 頁
- 分類：{ctx.get('doc_category', 'unknown')}
- 內容：{ctx['text']}
---
"""

    # --- 組裝歷史摘要 ---
    history_block = ""
    if history:
        recent = history[-(MAX_HISTORY_TURNS * 2):]
        history_block = "\n## 對話紀錄\n"
        for turn in recent:
            role = "使用者" if turn["role"] == "user" else "助教"
            content = turn["content"][:400]
            history_block += f"**{role}**：{content}\n\n"

    # --- 組裝改寫透明度 ---
    rewrite_note = ""
    if rewritten_query and rewritten_query != scenario:
        rewrite_note = f"\n> 系統將追問改寫為：「{rewritten_query}」，並以此進行檢索。\n"

    # --- 最終 prompt ---
    return f"""{history_block}{rewrite_note}
## 檢索到的相關文件
{context_str}

## 使用者當前問題
{scenario}

請根據以上資訊回應使用者。

### 回應指引
- 優先引用【權威來源】的定義和標準
- 可用【簡化說明】輔助解釋
- 所有判斷必須有明確的文件依據
- 若使用者在追問先前話題，請基於對話紀錄回應，避免重複分析
- 若使用者問了一個全新的情境，請進行完整的紅旗分析

請以 JSON 格式輸出。
"""

### Rule-based Baseline

用 LLM 做 intent classification 之前，先寫一個 rule-based 版本。
1. 需要一個不依賴 LLM 的 baseline 來比較
2. rule-based 版本可以處理 Turn 1（沒有歷史的情況），避免多餘的 LLM call。

定義標籤 (Enum)

In [ ]:
from enum import Enum

class TurnIntent(str, Enum):
    RETRIEVE = "RETRIEVE"
    ANSWER_FROM_HISTORY = "ANSWER_FROM_HISTORY"
    CLARIFICATION = "CLARIFICATION"
    OUT_OF_SCOPE = "OUT_OF_SCOPE"


# 偵測「追問前輪細節」的詞彙
CLARIFICATION_SIGNALS = [
    "什麼意思", "怎麼理解", "可以解釋", "剛才說的",
    "你提到的", "上面說的", "不太懂", "具體來說",
]

def classify_intent_rule_based(
    user_input: str,
    history: list,
    conversation_state: dict,
    verbose: bool = False,
) -> TurnIntent:
    """
    Rule-based intent classification（v4 baseline）

    Pipeline 位置：history → [classify_intent] → {路由決策}

    設計原則：
    - Turn 1（無歷史）一律走 RETRIEVE
    - 有歷史時，用訊號詞 + conversation_state 判斷
    - 寧可多走 RETRIEVE，不冒漏檢索的風險（保守策略）
    """
    # === 快速路徑：Turn 1 ===
    if not history:
        if verbose:
            print("   🧭 Intent: RETRIEVE（Turn 1，無歷史）")
        return TurnIntent.RETRIEVE

    # === 取得上一輪的 conversation_state ===
    if not conversation_state:
        # state 遺失，保守走 RETRIEVE
        if verbose:
            print("   🧭 Intent: RETRIEVE（state 遺失）")
        return TurnIntent.RETRIEVE

    # === Rule 1: 偵測 clarification 訊號 ===
    input_lower = user_input.lower()
    for signal in CLARIFICATION_SIGNALS:
        if signal in input_lower:
            if verbose:
                print(f"   🧭 Intent: CLARIFICATION（訊號詞：「{signal}」）")
            return TurnIntent.CLARIFICATION

    # === Rule 2: 偵測 out_of_scope（借用 SemanticScopeClassifier）===
    # 注意：這裡不做，留給後面的 gate。
    # classify_intent 的 OUT_OF_SCOPE 只處理「明顯離題」的快速路徑
    # 例如完全沒有 AML 相關詞彙的閒聊

    # === Rule 3: 判斷是否需要新資訊 ===
    # 如果 user_input 裡出現了 conversation_state.active_flags 裡
    # 沒有的新 entity，大概率需要新的 retrieval
    active_flags = conversation_state.get("active_flags", [])
    summary = conversation_state.get("scenario_summary", "")

    # 保守策略：預設走 RETRIEVE
    # 只有當問題「明確」是在問已有資訊時才走 ANSWER_FROM_HISTORY
    # 目前 rule-based 版本不嘗試判斷 ANSWER_FROM_HISTORY，
    # 因為誤判的代價太高（該檢索卻沒檢索 > 不該檢索卻多檢索一次）

    if verbose:
        print("   🧭 Intent: RETRIEVE（預設路徑）")
    return TurnIntent.RETRIEVE

In [ ]:
INTENT_CLASSIFY_PROMPT = """你是一個對話意圖分類器。

根據【對話狀態】和【使用者新輸入】，判斷這一輪應該走哪條路徑。

分類規則：
- RETRIEVE：使用者提出了新的情境、新的 entity、或需要查找知識庫才能回答的延伸問題
- ANSWER_FROM_HISTORY：使用者問的東西，在對話狀態的「分析摘要」或「已識別紅旗」裡已有答案
- CLARIFICATION：使用者在要求解釋或釐清前一輪的回答
- OUT_OF_SCOPE：使用者的問題明顯與洗錢防制/反洗錢無關

只輸出一個分類標籤，不要解釋。

範例：
狀態：已識別 RF-01（門檻拆分），摘要提到「50 萬申報門檻」
輸入：「那 RF-01 的申報門檻到底是多少？」
輸出：ANSWER_FROM_HISTORY

狀態：已識別 RF-01（門檻拆分）
輸入：「如果改用虛擬貨幣呢？」
輸出：RETRIEVE

狀態：已識別 RF-04（人頭帳戶）
輸入：「你說的第三方代辦是什麼意思？」
輸出：CLARIFICATION

狀態：已識別 RF-01
輸入：「今天台北天氣如何？」
輸出：OUT_OF_SCOPE
"""


def classify_intent_llm(
    user_input: str,
    conversation_state: dict,
    llm_config: dict = None,
    verbose: bool = False,
) -> TurnIntent:
    """
    LLM-based intent classification（v4）

    成本：1 次 LLM call（短 prompt，~200 tokens input）
    預期延遲：< 0.5s on Groq

    Fallback：如果 LLM 回傳無法解析的結果，降級為 RETRIEVE
    """
    if llm_config is None:
        llm_config = SELECTED_CONFIG

    # 組裝 state 摘要（跟 rewrite_query 類似，但更精簡）
    state_parts = []
    if conversation_state.get("active_flags"):
        state_parts.append(
            f"已識別紅旗：{', '.join(conversation_state['active_flags'])}"
        )
    if conversation_state.get("scenario_summary"):
        state_parts.append(
            f"分析摘要：{conversation_state['scenario_summary'][:150]}"
        )

    state_text = "\n".join(state_parts) if state_parts else "（無歷史狀態）"

    user_prompt = f"""【對話狀態】
{state_text}

【使用者新輸入】
{user_input}

分類："""

    try:
        raw = call_llm(
            system_prompt=INTENT_CLASSIFY_PROMPT,
            user_prompt=user_prompt,
            llm_config=llm_config,
            response_format="text",
        )

        if raw is None:
            if verbose:
                print("   🧭 Intent: RETRIEVE（LLM 回傳 None，fallback）")
            return TurnIntent.RETRIEVE

        label = raw.strip().upper()

        # 容錯：8B 有時候會輸出多餘的文字
        for intent in TurnIntent:
            if intent.value in label:
                if verbose:
                    print(f"   🧭 Intent: {intent.value}（LLM 判定）")
                return intent

        # 無法解析 → 保守走 RETRIEVE
        if verbose:
            print(f"   🧭 Intent: RETRIEVE（無法解析「{raw.strip()[:30]}」，fallback）")
        return TurnIntent.RETRIEVE

    except Exception as e:
        if verbose:
            print(f"   🧭 Intent: RETRIEVE（例外：{e}，fallback）")
        return TurnIntent.RETRIEVE

In [ ]:
def build_history_only_prompt(user_input, history, conversation_state):
    """
    不帶 retrieved context 的 prompt（路徑 B：ANSWER_FROM_HISTORY 用）
    設計：明確告訴 LLM 只要看「歷史」，不要編造。
    """
    # 從 history 組裝最近的對話脈絡（取 5 輪）
    MAX_HISTORY_TURNS = 5
    recent = history[-(MAX_HISTORY_TURNS * 2):]
    history_block = ""
    for turn in recent:
        role = "用戶" if turn["role"] == "user" else "助手"
        # 優先取人類可讀版本，避免 JSON 結構干擾 LLM 讀取歷史
        content = turn.get("content_readable", turn["content"])[:300]
        history_block += f"【{role}】{content}\n"

    return f"""## 對話歷史摘要 (Conversation State)
分析摘要：{conversation_state.get('scenario_summary', '無')}
已識別紅旗：{', '.join(conversation_state.get('active_flags', []))}

## 近期對話脈絡
{history_block}

## 使用者當前問題
{user_input}

### 回應指引
1. 本輪「未」進行知識庫檢索，請完全根據上述對話歷史與摘要回答。
2. 如果對話歷史中沒有提到相關資訊，請誠實告知使用者「根據之前的討論，我還沒有相關資訊，請問需要我針對特定主題進行檢索嗎？」。
3. 保持 JSON 輸出格式。
"""


def build_clarification_prompt(user_input, history, conversation_state):
    """
    專門用於釐清概念的 prompt（路徑 C：CLARIFICATION 用）
    設計：引導 LLM 扮演「教育者」角色解釋已有的回答。
    """
    MAX_HISTORY_TURNS = 3
    recent = history[-(MAX_HISTORY_TURNS * 2):]
    history_block = ""
    for turn in recent:
        role = "用戶" if turn["role"] == "user" else "助手"
        content = turn.get("content_readable", turn["content"])[:300]
        history_block += f"【{role}】{content}\n"

    return f"""## 對話歷史脈絡
{history_block}

## 使用者想釐清的問題
{user_input}

### 回應指引
1. 使用者目前對之前的回答內容有疑問或想深入了解定義。
2. 請針對上述歷史內容進行語義解釋、概念拆解或補充洗錢防制的核心邏輯。
3. 不需要檢索新法條，重點在於讓使用者「聽懂」剛才的分析結論。
4. 保持 JSON 輸出格式。
"""

### 6.3 chat() — 單輪對話處理

Step 0: classify_intent →
- if RETRIEVE:     Step 1: rewrite → Step 2: gate → Step 3: retrieve → Step 4: build_prompt → Step 5: LLM
- if ANSWER_FROM_HISTORY: Step 4': build_history_prompt → Step 5: LLM（跳過 rewrite + retrieve）
- if CLARIFICATION: Step 4'': build_clarification_prompt → Step 5: LLM（跳過 rewrite + retrieve）
- if OUT_OF_SCOPE:  直接回傳 REFUSE response（連 rewrite 的 LLM call 都省了）

In [ ]:
def chat(
    user_input: str,
    history: list,
    faiss_index,
    chunks: list,
    embedding_model,
    bm25_index,
    tokenized_corpus: list,
    llm_config: dict = None,
    k: int = DEFAULT_TOP_K,
    use_priority_weighting: bool = True,
    enable_gate: bool = True,
    intent_mode: str = "llm",   # "rule" | "llm" | "off"
    verbose: bool = True,
) -> dict:
    """
    v4: 加入 intent-routed pipeline

    Pipeline:
        Step 0: Intent Classification
        -> IF OUT_OF_SCOPE: 直接拒絕
        -> IF ANSWER_FROM_HISTORY / CLARIFICATION: 跳過檢索，用歷史回答
        -> IF RETRIEVE: 走 v3 完整 RAG 流程 (rewrite -> gate -> retrieve -> prompt -> LLM)

    History:
        - v3.1-fix-3: 加入 conversation_state 結構化狀態
        - v4.0: 插入意圖路由層，最佳化多輪對話效率與品質
    """
    if llm_config is None:
        llm_config = SELECTED_CONFIG

    if verbose:
        print("\n" + "=" * 60)
        print(f"💬 Multi-Turn Chat (v4: {intent_mode} mode)")
        print(f"   輸入：{user_input[:80]}{'...' if len(user_input) > 80 else ''}")
        print(f"   歷史：{len(history) // 2} 輪")
        print("=" * 60)

    # ===== Step 0: Intent Routing (意圖路由) =====
    # 1. 初始化狀態與預設意圖
    intent = TurnIntent.RETRIEVE
    prev_state = {}

    # 取得前一輪狀態
    prev_assistants = [t for t in history if t.get("role") == "assistant" and t.get("conversation_state")]
    if prev_assistants:
        prev_state = prev_assistants[-1]["conversation_state"]

    # 2. 判斷意圖 (當模式開啟且有歷史時才判斷)
    if intent_mode != "off" and history:
        if intent_mode == "llm":
            intent = classify_intent_llm(user_input, prev_state, llm_config, verbose)
        elif intent_mode == "rule":
            intent = classify_intent_rule_based(user_input, history, prev_state, verbose)

    if verbose:
        print(f"   🧭 路由決策: {intent}")

    # 3. 根據路由執行對應路徑
    result = None
    rewritten = None
    contexts = []
    gate_result = None

    # --- 路徑 A: OUT_OF_SCOPE (離題攔截) ---
    if intent == TurnIntent.OUT_OF_SCOPE:
        result = {
            "assessment": "refuse",
            "scenario_summary": "此問題與洗錢防制分析情境無關。",
            "identified_flags": [],
            "_intent": "OUT_OF_SCOPE",
            "_rewritten_query": user_input,
            "_retrieved_chunks": []
        }
        # 這種情況我們直接在下面 Step 6 處理歷史更新，不需要 return

    # --- 路徑 B: ANSWER_FROM_HISTORY (基於歷史) ---
    elif intent == TurnIntent.ANSWER_FROM_HISTORY:
        if verbose: print("   📚 執行路徑: ANSWER_FROM_HISTORY (跳過檢索)")
        user_prompt = build_history_only_prompt(user_input, history, prev_state)
        result = call_llm(SYSTEM_PROMPT, user_prompt, llm_config)
        result["_intent"] = "ANSWER_FROM_HISTORY"

    # --- 路徑 C: CLARIFICATION (解釋釐清) ---
    elif intent == TurnIntent.CLARIFICATION:
        if verbose: print("   🔍 執行路徑: CLARIFICATION (解釋說明)")
        user_prompt = build_clarification_prompt(user_input, history, prev_state)
        result = call_llm(SYSTEM_PROMPT, user_prompt, llm_config)
        result["_intent"] = "CLARIFICATION"

    # --- 路徑 D: RETRIEVE (標準 RAG 流程) ---
    else:
        if verbose: print("   🚀 執行路徑: RETRIEVE (完整 Pipeline)")

        # Step 1: Rewrite
        rewritten = rewrite_query(user_input, history, llm_config, verbose)

        # Step 2: Gate
        if enable_gate:
            gate_result = pre_llm_gate(rewritten, verbose=verbose)
            if gate_result.decision == GateDecision.REFUSE:
                # 若 Gate 攔截，直接準備結果並跳到 Step 6
                result = gate_result.to_refuse_response()
                result["_intent"] = "RETRIEVE_GATED"

        # Step 3~5 (只有在 Gate 沒攔截且還沒得到 result 時執行)
        if not result:
            contexts = retrieve_contexts(rewritten, faiss_index, chunks, embedding_model, bm25_index, tokenized_corpus, k=k)
            user_prompt = build_multiturn_user_prompt(user_input, contexts, history, rewritten)
            result = call_llm(SYSTEM_PROMPT, user_prompt, llm_config)
            result["_intent"] = "RETRIEVE"

        # Patch: Gate ALLOW 也記錄 metadata（讓 auto-save 能抓 covered_topics）
        if enable_gate and gate_result and gate_result.decision == GateDecision.ALLOW:
            result["_gate_metadata"] = {
                "decision": "ALLOW",
                "covered_topics": gate_result.metadata.get("covered_topics", []),
                "detected_topics": gate_result.metadata.get("detected_topics", []),
          }

        result["_rewritten_query"] = rewritten
        result["_retrieved_chunks"] = contexts

    # ===== Step 6: 統一更新歷史（匯流點） =====

    # 1. 確保基礎欄位存在 (預防路徑 A/B/C 漏掉 metadata)
    if "_intent" not in result: result["_intent"] = intent
    if "_rewritten_query" not in result: result["_rewritten_query"] = None
    if "_retrieved_chunks" not in result: result["_retrieved_chunks"] = []

    # 2. 更新 User 輸入
    history.append({"role": "user", "content": user_input})

    # 3. 準備 Assistant 的結構化儲存
    # 注意：如果是 CLARIFICATION 或 ANSWER_FROM_HISTORY，可能沒有 flags，給空 list
    current_flags = [f.get("flag_id", "?") for f in result.get("identified_flags", [])]

    # 狀態繼承與累加
    inherited_flags = prev_state.get("active_flags", [])
    all_flags = list(dict.fromkeys(inherited_flags + current_flags)) # 去重

    # 建立這一輪的狀態快照
    conversation_state = {
        "scenario_origin": prev_state.get("scenario_origin", user_input),
        "active_flags": all_flags,
        "scenario_summary": result.get("scenario_summary", "無摘要")[:200],
    }

    # 存入 History
    history.append({
        "role": "assistant",
        "content": json.dumps({
            "assessment": result.get("assessment", "N/A"),
            "summary": result.get("scenario_summary", "")[:200],
        }, ensure_ascii=False),
        "content_readable": result.get("scenario_summary", "請參考上述分析"),
        "conversation_state": conversation_state, # 這裡最重要，確保狀態傳遞下去
    })

    if verbose:
        print(f"   ✅ 歷史更新完成 (Intent: {result['_intent']})")

    return result

    # # ===== Step 6: 統一更新歷史（關鍵：確保所有路徑都走這裡） =====
    # # 如果前面沒有定義這些 metadata 欄位，補上預設值防止報錯
    # if "_rewritten_query" not in result: result["_rewritten_query"] = rewritten
    # if "_retrieved_chunks" not in result: result["_retrieved_chunks"] = contexts

    # # 更新歷史紀錄
    # history.append({"role": "user", "content": user_input})

    # # 建立結構化輸出內容
    # assistant_structured = json.dumps({
    #     "assessment": result.get("assessment", "unknown"),
    #     "flags": [f.get("flag_id", "?") for f in result.get("identified_flags", [])],
    #     "summary": result.get("scenario_summary", "")[:200],
    # }, ensure_ascii=False)

    # # 更新狀態 (State Management)
    # current_flags = [f.get("flag_id", "?") for f in result.get("identified_flags", [])]
    # all_flags = list(dict.fromkeys(prev_state.get("active_flags", []) + current_flags))

    # conversation_state = {
    #     "scenario_origin": prev_state.get("scenario_origin", user_input),
    #     "active_flags": all_flags,
    #     "scenario_summary": result.get("scenario_summary", "")[:200],
    # }

    # history.append({
    #     "role": "assistant",
    #     "content": assistant_structured,
    #     "content_readable": f"分析：{result.get('assessment')}。{result.get('scenario_summary')[:200]}",
    #     "conversation_state": conversation_state,
    # })

    # if verbose:
    #     print(f"   ✅ 完成 - Intent: {result['_intent']} | 累積 Flags: {all_flags}")
    #     print("=" * 60)

    # return result



    # # ===== Step 0-1: 取得前一輪狀態（為分類做準備） =====
    # prev_state = {}
    # prev_assistants = [t for t in history if t.get("role") == "assistant" and t.get("conversation_state")]
    # if prev_assistants:
    #     prev_state = prev_assistants[-1]["conversation_state"]

    # # ===== Step 0-2: Intent Classification =====
    # if intent_mode == "off" or not history:
    #     intent = TurnIntent.RETRIEVE
    # elif intent_mode == "llm":
    #     intent = classify_intent_llm(user_input, prev_state, llm_config, verbose)
    # else:  # "rule"
    #     intent = classify_intent_rule_based(user_input, history, prev_state, verbose)

    # # 初始化一些追蹤欄位
    # rewritten = None
    # contexts = []

    # # ===== 路由決策：根據 intent 走不同路徑 =====

    # # --- 路徑 A: OUT_OF_SCOPE (離題直接拒絕) ---
    # if intent == TurnIntent.OUT_OF_SCOPE:
    #     if verbose:
    #         print("   🚫 Intent OUT_OF_SCOPE -> 直接快速拒絕")

    #     refuse_response = {
    #         "assessment": "refuse",
    #         "scenario_summary": "此問題明顯與洗錢防制或目前的 AML 分析情境無關。",
    #         "identified_flags": [],
    #         "risk_level": "N/A",
    #         "_intent": "OUT_OF_SCOPE",
    #         "_rewritten_query": user_input,
    #         "_retrieved_chunks": []
    #     }
    #     # 更新歷史並直接回傳
    #     history.append({"role": "user", "content": user_input})
    #     history.append({
    #         "role": "assistant",
    #         "content": json.dumps({"assessment": "refuse", "summary": "離題"}, ensure_ascii=False),
    #         "content_readable": "[OUT_OF_SCOPE] 您好，我是 AML 輔助助手，目前僅能回答與洗錢防制相關的問題。",
    #         "conversation_state": prev_state # 繼承舊狀態
    #     })
    #     return refuse_response

    # # --- 路徑 B: ANSWER_FROM_HISTORY (基於歷史回答) ---
    # elif intent == TurnIntent.ANSWER_FROM_HISTORY:
    #     if verbose:
    #         print("   📚 Intent ANSWER_FROM_HISTORY -> 跳過檢索，從對話歷史尋找答案")

    #     user_prompt = build_history_only_prompt(
    #         user_input=user_input,
    #         history=history,
    #         conversation_state=prev_state,
    #     )
    #     result = call_llm(SYSTEM_PROMPT, user_prompt, llm_config)
    #     result["_intent"] = "ANSWER_FROM_HISTORY"
    #     result["_rewritten_query"] = None
    #     result["_retrieved_chunks"] = []

    # # --- 路徑 C: CLARIFICATION (解釋或釐清) ---
    # elif intent == TurnIntent.CLARIFICATION:
    #     if verbose:
    #         print("   🔍 Intent CLARIFICATION -> 用歷史資料進行解釋/釐清")

    #     user_prompt = build_clarification_prompt(
    #         user_input=user_input,
    #         history=history,
    #         conversation_state=prev_state,
    #     )
    #     result = call_llm(SYSTEM_PROMPT, user_prompt, llm_config)
    #     result["_intent"] = "CLARIFICATION"
    #     result["_rewritten_query"] = None
    #     result["_retrieved_chunks"] = []

    # # --- 路徑 D: RETRIEVE (標準 RAG 流程 - 預設路徑) ---
    # else:
    #     if verbose:
    #         print("   🚀 Intent RETRIEVE -> 執行完整 RAG Pipeline")

    #     # 原有 Step 1: Query Rewrite
    #     if verbose: print("\n🔄 Step 1: Query Rewrite...")
    #     rewritten = rewrite_query(user_input, history, llm_config, verbose)

    #     # 原有 Step 2: Pre-LLM Gate
    #     if enable_gate:
    #         if verbose: print("\n🚦 Step 2: Pre-LLM Gate...")
    #         gate_result = pre_llm_gate(rewritten, verbose=verbose)
    #         if gate_result.decision == GateDecision.REFUSE:
    #             refuse_res = gate_result.to_refuse_response()
    #             refuse_res["_rewritten_query"] = rewritten
    #             history.append({"role": "user", "content": user_input})
    #             history.append({"role": "assistant", "content": f"[REFUSE] {gate_result.reason_message}"})
    #             return refuse_res

    #     # 原有 Step 3: Retrieval
    #     if verbose: print(f"\n🔍 Step 3: Retrieval (query: {rewritten[:60]}...)")
    #     contexts = retrieve_contexts(
    #         query=rewritten, faiss_index=faiss_index, chunks=chunks,
    #         embedding_model=embedding_model, bm25_index=bm25_index,
    #         tokenized_corpus=tokenized_corpus, k=k, method="hybrid",
    #         use_priority_weighting=use_priority_weighting,
    #     )

    #     # 原有 Step 4: Build Prompt
    #     if verbose: print(f"\n📝 Step 4: Build Prompt (含 {len(history)//2} 輪歷史)")
    #     user_prompt = build_multiturn_user_prompt(
    #         scenario=user_input, contexts=contexts, history=history, rewritten_query=rewritten
    #     )

    #     # 原有 Step 5: LLM Answer
    #     if verbose: print(f"\n🤖 Step 5: LLM 回應 ({llm_config['llm_model_name']})...")
    #     result = call_llm(SYSTEM_PROMPT, user_prompt, llm_config)

    #     # 標記 Metadata
    #     result["_intent"] = "RETRIEVE"
    #     result["_rewritten_query"] = rewritten
    #     result["_retrieved_chunks"] = [
    #         {"chunk_id": ctx["chunk_id"], "source": ctx["source"], "page": ctx["page"], "score": ctx["score"]}
    #         for ctx in contexts
    #     ]

    # # ===== Step 6: 統一更新歷史（含 conversation_state）=====
    # # 不論走 B, C, D 路徑，最後都會執行到這裡

    # history.append({"role": "user", "content": user_input})

    # # 準備儲存用的格式
    # assistant_structured = json.dumps({
    #     "assessment": result.get("assessment", "unknown"),
    #     "flags": [f.get("flag_id", "?") for f in result.get("identified_flags", [])],
    #     "summary": result.get("scenario_summary", "")[:200],
    # }, ensure_ascii=False)

    # assistant_readable = (
    #     f"分析結果：{result.get('assessment', 'unknown')}。"
    #     f"{result.get('scenario_summary', '')[:200]}"
    # )

    # # 狀態管理
    # current_flags = [f.get("flag_id", "?") for f in result.get("identified_flags", [])]
    # inherited_flags = prev_state.get("active_flags", [])
    # all_flags = list(dict.fromkeys(inherited_flags + current_flags)) # 去重

    # conversation_state = {
    #     "scenario_origin": prev_state.get("scenario_origin", user_input),
    #     "active_flags": all_flags,
    #     "scenario_summary": result.get("scenario_summary", "")[:200],
    # }

    # history.append({
    #     "role": "assistant",
    #     "content": assistant_structured,
    #     "content_readable": assistant_readable,
    #     "conversation_state": conversation_state,
    # })

    # if verbose:
    #     print(f"   🧭 路由結果: {result['_intent']}")
    #     if conversation_state["active_flags"]:
    #         print(f"   📌 累積 flags: {conversation_state['active_flags']}")
    #     print(f"   ✅ 對話完成（歷史已更新至 {len(history)//2} 輪）")
    #     print("=" * 60)

    # return result

### 6.4 chat_loop() — 互動式對話迴圈

In [ ]:
def chat_loop(
    faiss_index=faiss_index,
    chunks=chunks,
    embedding_model=embedding_model,
    bm25_index=bm25_index,
    tokenized_corpus=tokenized_corpus,
    llm_config=None,
    enable_gate=True,
    verbose=True,
):
    """
    互動式多輪對話（Colab 環境可用）

    指令：
        quit   — 結束對話
        reset  — 清除歷史，重新開始
        history — 顯示目前的對話歷史

    Demo 流程建議：
        Turn 1: 「某客戶連續三天每次存 49 萬現金」
        Turn 2: 「台灣法規的申報門檻是多少？」
        Turn 3: 「如果用虛擬貨幣做類似的事呢？」
    """
    if llm_config is None:
        llm_config = SELECTED_CONFIG

    conversation_history = []

    print("\n" + "=" * 60)
    print("🔄 AML 紅旗助教 — 多輪對話模式")
    print("=" * 60)
    print(f"   模型：{llm_config['llm_model_name']}")
    print(f"   Gate：{'ON' if enable_gate else 'OFF'}")
    print(f"   輸入 'quit' 結束 | 'reset' 清除歷史 | 'history' 查看歷史")
    print("=" * 60)

    while True:
        try:
            user_input = input("\n🧑 你的問題：").strip()
        except (EOFError, KeyboardInterrupt):
            print("\n👋 對話結束")
            break

        if not user_input:
            continue

        if user_input.lower() == "quit":
            print("\n👋 對話結束")
            break

        if user_input.lower() == "reset":
            conversation_history.clear()
            print("🔄 對話歷史已清除，重新開始")
            continue

        if user_input.lower() == "history":
            if not conversation_history:
                print("   （空，尚未開始對話）")
            else:
                for i, turn in enumerate(conversation_history):
                    role = "🧑 用戶" if turn["role"] == "user" else "🤖 助教"
                    print(f"   [{i//2 + 1}] {role}：{turn['content'][:100]}")
            continue

        # === 執行對話 ===
        result = chat(
            user_input=user_input,
            history=conversation_history,
            faiss_index=faiss_index,
            chunks=chunks,
            embedding_model=embedding_model,
            bm25_index=bm25_index,
            tokenized_corpus=tokenized_corpus,
            llm_config=llm_config,
            enable_gate=enable_gate,
            verbose=verbose,
        )

        # === 輸出結果 ===
        pretty_print_result(result)

        # 顯示改寫資訊（如果有改寫的話）
        rewritten = result.get("_rewritten_query", "")
        if rewritten and rewritten != user_input:
            print(f"\n💡 本輪檢索使用的改寫查詢：「{rewritten}」")

    # === 對話結束，回傳完整歷史 ===
    return conversation_history


print("\n✅ PART 6: Multi-Turn Conversation 模組已載入")
print("   使用 chat_loop() 啟動互動式對話")
print("   使用 chat() 進行單輪對話（供程式化呼叫）")


✅ PART 6: Multi-Turn Conversation 模組已載入
   使用 chat_loop() 啟動互動式對話
   使用 chat() 進行單輪對話（供程式化呼叫）


### 🚀 啟動多輪對話（取消下面這行的註解即可）

In [ ]:
# chat_loop()

### TEST: Multi-Turn 行為測試

目的：驗證 rewrite → gate pipeline 在多輪情境下的行為
這不是 retrieval eval，沒有 gold chunk，不計算 MRR

觀察點：
  [R] rewrite 有沒有在正確時機觸發
  [Q] 改寫後的 query 語意是否正確（最重要）
  [G] gate 有沒有在應該攔的時候攔

執行方式：
  直接執行這個 cell，三個 session 會依序跑完
  每個 session 之間有分隔線，方便截圖

工具函數：執行單一 session 並列印結構化結果 (v4 路由測試版)

In [ ]:
MULTITURN_LOG_DIR = "/content/drive/MyDrive/AML/eval/multiturn"

In [ ]:
import json

def run_multiturn_session(
    session_id: str,
    session_name: str,
    turns: list,
    observe: list,
    faiss_index=faiss_index,
    chunks=chunks,
    embedding_model=embedding_model,
    bm25_index=bm25_index,
    tokenized_corpus=tokenized_corpus,
    llm_config=None,
    enable_gate: bool = True,
    intent_mode: str = "off",
    auto_save: bool = True,       # ← 新增：要不要自動存 JSON
):
    """
    執行一個多輪測試 session (支援 v4 Intent Routing)

    auto_save=True 時，跑完會自動呼叫 log_multiturn_session() 存檔。
    人工標註（rewrite_verdict, intent_notes 等）事後再補。
    """
    if llm_config is None:
        llm_config = SELECTED_CONFIG

    # ====== 以下是你原本的 header 印刷，一字不改 ======
    print("\n" + "=" * 70)
    print(f"🧪 Session {session_id}：{session_name}")
    print(f"   模式：{intent_mode} | 模型：{llm_config['llm_model_name']}")
    print(f"   Gate：{'ON' if enable_gate else 'OFF'}")
    print("─" * 70)
    print("   觀察點：")
    for obs in observe:
        print(f"   → {obs}")
    print("=" * 70)

    history = []
    session_log = []       # 給螢幕摘要表格用（你原本的）
    logged_turns = []      # ← 新增：給 JSON 存檔用

    for i, turn in enumerate(turns, 1):
        user_input = turn["input"]
        note = turn.get("note", "")

        # ====== 你原本的 turn header ======
        print(f"\n{'─' * 70}")
        print(f"   Turn {i}  {'【' + note + '】' if note else ''}")
        print(f"   輸入：{user_input}")
        print(f"{'─' * 70}")

        # ====== 你原本的 chat() 呼叫 ======
        result = chat(
            user_input=user_input,
            history=history,
            faiss_index=faiss_index,
            chunks=chunks,
            embedding_model=embedding_model,
            bm25_index=bm25_index,
            tokenized_corpus=tokenized_corpus,
            llm_config=llm_config,
            enable_gate=enable_gate,
            intent_mode=intent_mode,
            verbose=True,
        )

        # ====== 你原本的欄位擷取（給螢幕表格用）======
        intent = result.get("_intent", "N/A")
        rewritten = result.get("_rewritten_query", "—")
        assessment = result.get("assessment", "—")
        flags = [f.get("flag_id", "?") for f in result.get("identified_flags", [])]

        session_log.append({
            "turn": i,
            "input": user_input,
            "intent": intent,
            "rewritten": rewritten,
            "assessment": assessment,
            "flags": flags,
        })

        # ====== 新增：自動組裝 log_turn() ======
        if auto_save:
            expected_intent = turn.get("expected_intent")

            # 自動判定 intent verdict
            if not history or expected_intent is None:
                # Turn 1 或沒標預期 → 不評估
                intent_verdict = "N/A"
            elif intent == expected_intent:
                intent_verdict = "PASS"
            else:
                intent_verdict = "WRONG"

            logged = log_turn(
                turn_number=i,
                user_input=user_input,
                expected_behavior=note,

                # intent（自動從 result 取）
                intent=intent,
                expected_intent=expected_intent,
                intent_verdict=intent_verdict,

                # rewrite（自動從 result 取）
                rewrite_triggered=(
                    rewritten is not None
                    and rewritten != user_input
                    and rewritten != "—"
                ),
                original_query=user_input,
                rewritten_query=rewritten if rewritten != "—" else None,
                # rewrite_verdict 留空 → 事後人工標註
                rewrite_verdict=None,

                # gate（自動從 result 取）
                actual_gate=result.get("_gate_metadata", {}).get("decision", "ALLOW"),
                gate_covered_topics=result.get("_gate_metadata", {}).get("covered_topics", []),

                # retrieval（自動從 result 取）
                retrieved_chunks=result.get("_retrieved_chunks", []),

                # LLM（自動從 result 取）
                assessment=assessment,
                flags=flags,
            )
            logged_turns.append(logged)

    # ====== 你原本的摘要表格，加了 Intent 欄位 ======
    print(f"\n{'=' * 75}")
    print(f"📋 Session {session_id} 摘要")
    print(f"{'─' * 75}")
    print(f"  {'Turn':<5} {'Intent':<20} {'Assessment':<12} {'Flags':<15} 改寫/狀態")
    print(f"{'─' * 75}")
    for row in session_log:
        intent_str = row["intent"]
        flags_str = ", ".join(row["flags"]) if row["flags"] else "—"
        display_query = row["rewritten"] if row["rewritten"] else "(跳過檢索)"
        short_query = (display_query[:30] + "…") if len(str(display_query)) > 30 else display_query
        print(f"  {row['turn']:<5} {intent_str:<20} {row['assessment']:<12} {flags_str:<15} {short_query}")
    print(f"{'=' * 75}\n")

    # ====== 新增：自動存檔 ======
    saved_path = None
    if auto_save and logged_turns:
        saved_path = log_multiturn_session(
            session_id=session_id,
            session_purpose=session_name,
            turns=logged_turns,
            llm_config=llm_config,
            intent_mode=intent_mode,
            enable_gate=enable_gate,
            note=f"auto | observe: {'; '.join(observe[:2])}",
        )

    return session_log, saved_path

In [ ]:
def log_turn(
    turn_number: int,
    user_input: str,
    expected_behavior: str,
    # ===== v4 新增：intent =====
    intent: str = None,                # LLM/rule 判定的意圖標籤
    expected_intent: str = None,       # 人工標註的「應該是什麼」
    intent_verdict: str = None,        # PASS / WRONG / N/A
    intent_notes: str = None,
    # ===== 原有：rewrite =====
    rewrite_triggered: bool = False,
    original_query: str = None,
    rewritten_query: str = None,
    expected_rewrite: str = None,
    rewrite_verdict: str = None,       # PASS / DEGRADED / SKIP
    rewrite_notes: str = None,
    # ===== 原有：gate =====
    expected_gate: str = "ALLOW",
    actual_gate: str = "ALLOW",
    gate_covered_topics: list = None,
    # ===== 原有：retrieval =====
    retrieved_chunks: list = None,
    # ===== 原有：LLM =====
    assessment: str = None,
    flags: list = None,
) -> dict:
    """
    組裝單一 Turn 的記錄

    v4 變更：
    - 新增 intent 區塊（intent / expected_intent / intent_verdict）
    - 當 intent 為 ANSWER_FROM_HISTORY 或 CLARIFICATION 時，
      rewrite 和 retrieval 區塊自動標記為「不適用」
    """

    # --- 根據 intent 自動調整 rewrite / retrieval 的記錄 ---
    skip_retrieval = intent in ("ANSWER_FROM_HISTORY", "CLARIFICATION", "OUT_OF_SCOPE")

    turn = {
        "turn": turn_number,
        "user_input": user_input,
        "expected_behavior": expected_behavior,

        # v4 新增
        "intent": {
            "classified": intent,          # 系統判定
            "expected": expected_intent,    # 人工標註
            "verdict": intent_verdict,      # PASS / WRONG / N/A
            "notes": intent_notes,
        },

        "rewrite": {
            "triggered": rewrite_triggered if not skip_retrieval else False,
            "original": original_query or user_input,
            "rewritten": rewritten_query if not skip_retrieval else None,
            "expected": expected_rewrite,
            "verdict": rewrite_verdict if not skip_retrieval else "SKIPPED_BY_INTENT",
            "notes": rewrite_notes or (
                f"Intent={intent}，跳過 rewrite" if skip_retrieval else None
            ),
        },

        "gate": {
            "expected": expected_gate if not skip_retrieval else "SKIPPED_BY_INTENT",
            "actual": actual_gate if not skip_retrieval else "SKIPPED_BY_INTENT",
            "covered_topics": gate_covered_topics or [],
        },

        "retrieval": {
            "chunks": retrieved_chunks or [],
            "skipped": skip_retrieval,     # ← 新欄位：明確標記是否跳過檢索
        },

        "llm_output": {
            "assessment": assessment,
            "flags": flags or [],
        },
    }

    # === flat fields（給 session summary 統計用）===
    turn["_flat"] = {
        "intent": intent,
        "rewrite_triggered": rewrite_triggered and not skip_retrieval,
        "rewrite_verdict": turn["rewrite"]["verdict"],
        "expected_gate": turn["gate"]["expected"],
        "actual_gate": turn["gate"]["actual"],
        "retrieval_skipped": skip_retrieval,
    }

    return turn

In [ ]:
def log_multiturn_session(
    session_id: str,
    session_purpose: str,
    turns: list,
    llm_config: dict,
    intent_mode: str = "off",     # v4 新增："off" / "rule" / "llm"
    enable_gate: bool = True,
    note: str = "",
) -> str:
    """
    v4 變更：
    - 檔名加入 intent_mode 標籤
    - 自動遞增序號避免覆蓋
    - summary 新增 intent_distribution 和 retrieval_skipped_count
    """
    os.makedirs(MULTITURN_LOG_DIR, exist_ok=True)

    date_str = datetime.now().strftime("%Y%m%d")
    model_tag = llm_config["llm_model_name"].split("/")[-1].replace("-", "_")

    # --- 檔名：session_{id}_{date}_{model}_{mode}.json ---
    base = f"session_{session_id}_{date_str}_{model_tag}_{intent_mode}"
    path = os.path.join(MULTITURN_LOG_DIR, f"{base}.json")

    counter = 1
    while os.path.exists(path):
        path = os.path.join(MULTITURN_LOG_DIR, f"{base}_{counter:02d}.json")
        counter += 1

    filename = os.path.basename(path)

    # === Summary 計算 ===
    flat = [t.get("_flat", {}) for t in turns]

    # intent 分布
    intent_counts = {}
    for f in flat:
        it = f.get("intent", "N/A")
        intent_counts[it] = intent_counts.get(it, 0) + 1

    # intent 判定準確率
    intent_verdicts = [t.get("intent", {}).get("verdict") for t in turns]
    intent_pass = sum(1 for v in intent_verdicts if v == "PASS")
    intent_wrong = sum(1 for v in intent_verdicts if v == "WRONG")

    # rewrite（只統計走 RETRIEVE 路徑的 turn）
    retrieve_turns = [f for f in flat if f.get("intent") == "RETRIEVE"]
    rewrites_triggered = sum(1 for f in retrieve_turns if f.get("rewrite_triggered"))
    rewrite_pass = sum(
        1 for t in turns
        if t.get("_flat", {}).get("intent") == "RETRIEVE"
        and t.get("rewrite", {}).get("verdict") == "PASS"
    )
    rewrite_degraded = sum(
        1 for t in turns
        if t.get("_flat", {}).get("intent") == "RETRIEVE"
        and t.get("rewrite", {}).get("verdict") == "DEGRADED"
    )

    # gate
    gate_expected_refuse = sum(1 for f in flat if f.get("expected_gate") == "REFUSE")
    gate_actual_refuse = sum(1 for f in flat if f.get("actual_gate") == "REFUSE")

    # 檢索跳過次數（v4 核心指標）
    retrieval_skipped = sum(1 for f in flat if f.get("retrieval_skipped"))

    session_data = {
        "session_id": session_id,
        "purpose": session_purpose,
        "date": datetime.now().isoformat(),
        "note": note,

        "config": {
            "model": llm_config["llm_model_name"],
            "provider": llm_config.get("provider", "unknown"),
            "gate_enabled": enable_gate,
            "intent_mode": intent_mode,
            "max_history_turns": MAX_HISTORY_TURNS,
        },

        "turns": turns,

        "summary": {
            "total_turns": len(turns),

            # v4 新增
            "intent_distribution": intent_counts,
            "intent_pass": intent_pass,
            "intent_wrong": intent_wrong,
            "retrieval_skipped": retrieval_skipped,

            # 原有（現在只統計 RETRIEVE 路徑）
            "rewrites_triggered": rewrites_triggered,
            "rewrite_pass": rewrite_pass,
            "rewrite_degraded": rewrite_degraded,
            "gate_expected_refuse": gate_expected_refuse,
            "gate_actual_refuse": gate_actual_refuse,
        },
    }

    with open(path, "w", encoding="utf-8") as f:
        json.dump(session_data, f, ensure_ascii=False, indent=2)

    print(f"💾 Session {session_id} 已儲存: {filename}")
    return path

無AB測試

In [ ]:
# # ----------------------------------------------------------
# # Session A：代名詞消解 + 條件延伸
# # ----------------------------------------------------------
# # 預期行為：
# #   Turn 1 → Intent: RETRIEVE (無歷史)
# #   Turn 2 → Intent: ANSWER_FROM_HISTORY (回答門檻無需檢索)
# #   Turn 3 → Intent: RETRIEVE (引入虛擬貨幣新實體，需檢索)

# session_a_log = run_multiturn_session(
#     session_id="A",
#     session_name="代名詞消解 + 條件延伸",
#     intent_mode="llm",  # 啟動意圖分流
#     turns=[
#         {
#             "input": "某客戶連續三天分批存款，每次金額都在 49 萬左右，存完後隔天立即轉出。這筆有問題嗎？",
#             "note": "建立歷史 / 應觸發 RF-01",
#             "expected_intent": None,                  # ← Turn 1 不評估
#         },
#         {
#             "input": "它的申報門檻是多少？",
#             "note": "代名詞追問 → 預期跳過檢索",
#             "expected_intent": "ANSWER_FROM_HISTORY",  # ← 你的預期
#         },
#         {
#             "input": "如果改用虛擬貨幣做類似的事，還算同一個紅旗嗎？",
#             "note": "條件延伸 → 預期重啟檢索",
#             "expected_intent": "RETRIEVE",             # ← 你的預期
#         },
#     ],
#     observe=[
#         "[I] Turn 2 是否正確判斷為 ANSWER_FROM_HISTORY 以節省 token",
#         "[R] Turn 3 的 rewrite 是否同時保留 structuring + 加入虛擬資產語意",
#         "[Q] 改寫後的 query 夠不夠讓 retrieval 找到正確段落",
#     ],
# )


# # ----------------------------------------------------------
# # Session B：Gate 在多輪中的攔截行為
# # ----------------------------------------------------------
# # 預期行為：
# #   Turn 1, 2 → 正常 AML 問答
# #   Turn 3    → Intent: OUT_OF_SCOPE (直接攔截)

# session_b_log = run_multiturn_session(
#     session_id="B",
#     session_name="Gate 攔截：話題跳離 AML",
#     intent_mode="llm",
#     turns=[
#         {
#             "input": "某小型文創公司帳戶突然收到多筆大額款項，負責人說不清楚交易對手，資金進來後馬上轉出。",
#             "note": "正常 AML 情境 / 應觸發 RF-06、RF-02",
#             "expected_intent": None,                  # ← Turn 1 不評估
#         },
#         {
#             "input": "FATF 對空殼公司的紅旗有哪些具體描述？",
#             "note": "合理追問 / 不應被攔",
#             "expected_intent": "RETRIEVE",  # ← 你的預期
#         },
#         {
#             "input": "順便問一下，台灣現在的央行利率是多少？",
#             "note": "話題跳離 → 預期 Intent 直接判為 OUT_OF_SCOPE",
#             "expected_intent": "OUT_OF_SCOPE",  # ← 你的預期
#         },
#     ],
#     observe=[
#         "[I] Turn 3 是否被意圖分類器判定為 OUT_OF_SCOPE",
#         "[G] Turn 1、2 是否沒有被誤攔（false REFUSE）",
#         "[R] 觀察離題問題是否在進入 Rewrite 前就已被攔截",
#     ],
# )


# # ----------------------------------------------------------
# # Session C：跨 RF 複合追問（壓力測試）
# # ----------------------------------------------------------
# # 預期行為：
# #   Turn 2 → Intent: CLARIFICATION (要求解釋法規差異)
# #   Turn 3 → Intent: RETRIEVE (複合情境需新檢索)

# session_c_log = run_multiturn_session(
#     session_id="C",
#     session_name="跨 RF 複合追問（壓力測試）",
#     intent_mode="llm",
#     turns=[
#         {
#             "input": "某學生帳戶由叔叔代辦所有操作，交易對象多為陌生公司，資金入帳後立刻轉出，學生本人無法說明用途。",
#             "note": "RF-04 人頭帳戶情境",
#             "expected_intent": None,                  # ← Turn 1 不評估
#         },
#         {
#             "input": "人頭帳戶跟第三方代辦，法規上怎麼區分？",
#             "note": "細節釐清 → 預期判為 CLARIFICATION",
#             "expected_intent": "CLARIFICATION",       # ← 你的預期
#         },
#         {
#             "input": "如果這個叔叔又用加密貨幣把錢轉走，這樣算幾個紅旗？",
#             "note": "跨 RF 複合 → 預期判為 RETRIEVE",
#             "expected_intent": "RETRIEVE",           # ← 你的預期
#         },
#     ],
#     observe=[
#         "[I] Turn 2 是否判為 CLARIFICATION 並提供結構化解釋",
#         "[R] Turn 3 能否同時帶入 RF-04 和虛擬資產兩個脈絡進行檢索",
#         "[Q] 若 Turn 3 改寫品質下降 → 記錄為系統限制",
#     ],
# )

全部 session 完成

In [ ]:
# print("\n" + "=" * 70)
# print("✅ 三個 Session 全部完成")
# print("   截圖重點：")
# print("   1. 每個 Turn 的「📝 Query Rewrite」那兩行（原句 vs 改寫）")
# print("   2. Session B Turn 3 的「🚫 Gate REFUSE」")
# print("   3. 每個 Session 結尾的摘要表格")
# print("=" * 70)

ab_test_multiturn() — 多輪 A/B 對比測試

In [ ]:
def ab_test_multiturn(
    session_id: str,
    session_name: str,
    turns: list,
    observe: list,
    # --- A/B 差異參數 ---
    config_a: dict = None,       # {"intent_mode": "off", "label": "無意圖分流"}
    config_b: dict = None,       # {"intent_mode": "llm", "label": "LLM 意圖分流"}
    # --- 共用參數 ---
    faiss_index=None,
    chunks=None,
    embedding_model=None,
    bm25_index=None,
    tokenized_corpus=None,
    llm_config=None,
    enable_gate: bool = True,
):
    """
    對同一組 session turns，跑兩個 config（A/B），自動存檔 + 輸出行為差異報告。

    用途：驗證「intent routing 開 vs 關」對多輪行為的影響
    不做的事：品質判斷（rewrite_verdict 等仍需人工標註）

    Args:
        config_a / config_b: 格式 {"intent_mode": str, "label": str, ...}
            除了 intent_mode 和 label，其餘 key 會被傳進 run_multiturn_session()
        其餘參數同 run_multiturn_session()

    Returns:
        dict: {
            "session_id": str,
            "config_a": {...}, "config_b": {...},
            "result_a": (session_log, saved_path),
            "result_b": (session_log, saved_path),
            "diff": [per-turn diff list],
        }

    使用範例：
        ab_result = ab_test_multiturn(
            session_id="A",
            session_name="代名詞消解 + 條件延伸",
            turns=[
                {"input": "某客戶連續三天分批存款...", "note": "RF-01", "expected_intent": None},
                {"input": "它的申報門檻是多少？", "note": "代名詞追問", "expected_intent": "ANSWER_FROM_HISTORY"},
                {"input": "如果改用虛擬貨幣...", "note": "條件延伸", "expected_intent": "RETRIEVE"},
            ],
            observe=["[I] Turn 2 意圖分流", "[R] Turn 3 rewrite 品質"],
            config_a={"intent_mode": "off",  "label": "Baseline（無意圖分流）"},
            config_b={"intent_mode": "llm",  "label": "LLM 意圖分流"},
        )
    """
    # --- 預設 config ---
    if config_a is None:
        config_a = {"intent_mode": "off", "label": "Baseline（無意圖分流）"}
    if config_b is None:
        config_b = {"intent_mode": "llm", "label": "LLM 意圖分流"}

    label_a = config_a.pop("label", "Config A")
    label_b = config_b.pop("label", "Config B")

    # --- Header ---
    print("\n" + "=" * 75)
    print(f"⚖️  Multi-Turn A/B Test: Session {session_id}")
    print(f"    {label_a}  vs  {label_b}")
    print(f"    Session: {session_name}")
    print("=" * 75)

    # --- Run A ---
    print(f"\n{'━' * 75}")
    print(f"  🅰️  {label_a}")
    print(f"{'━' * 75}")

    result_a = run_multiturn_session(
        session_id=f"{session_id}_A",
        session_name=f"{session_name} [{label_a}]",
        turns=deepcopy(turns),    # deepcopy：避免 history 互相汙染
        observe=observe,
        faiss_index=faiss_index,
        chunks=chunks,
        embedding_model=embedding_model,
        bm25_index=bm25_index,
        tokenized_corpus=tokenized_corpus,
        llm_config=llm_config,
        enable_gate=enable_gate,
        auto_save=True,
        **config_a,               # intent_mode 等
    )

    # --- Run B ---
    print(f"\n{'━' * 75}")
    print(f"  🅱️  {label_b}")
    print(f"{'━' * 75}")

    result_b = run_multiturn_session(
        session_id=f"{session_id}_B",
        session_name=f"{session_name} [{label_b}]",
        turns=deepcopy(turns),
        observe=observe,
        faiss_index=faiss_index,
        chunks=chunks,
        embedding_model=embedding_model,
        bm25_index=bm25_index,
        tokenized_corpus=tokenized_corpus,
        llm_config=llm_config,
        enable_gate=enable_gate,
        auto_save=True,
        **config_b,
    )

    # --- Diff Report ---
    log_a, path_a = result_a
    log_b, path_b = result_b

    print(f"\n{'=' * 75}")
    print(f"📊 A/B 行為差異報告：Session {session_id}")
    print(f"{'─' * 75}")
    print(f"  {'Turn':<5} {'欄位':<18} {'🅰️ ' + label_a:<28} {'🅱️ ' + label_b:<28}")
    print(f"{'─' * 75}")

    diffs = []
    for i in range(len(turns)):
        a = log_a[i] if i < len(log_a) else {}
        b = log_b[i] if i < len(log_b) else {}

        turn_diffs = {}
        turn_num = i + 1

        # 比較 intent
        if a.get("intent") != b.get("intent"):
            turn_diffs["intent"] = (a.get("intent"), b.get("intent"))
            print(f"  T{turn_num:<4} {'intent':<18} {str(a.get('intent')):<28} {str(b.get('intent')):<28}")

        # 比較 assessment
        if a.get("assessment") != b.get("assessment"):
            turn_diffs["assessment"] = (a.get("assessment"), b.get("assessment"))
            print(f"  T{turn_num:<4} {'assessment':<18} {str(a.get('assessment')):<28} {str(b.get('assessment')):<28}")

        # 比較 flags
        if a.get("flags") != b.get("flags"):
            fa = ", ".join(a.get("flags", [])) or "—"
            fb = ", ".join(b.get("flags", [])) or "—"
            turn_diffs["flags"] = (a.get("flags"), b.get("flags"))
            print(f"  T{turn_num:<4} {'flags':<18} {fa:<28} {fb:<28}")

        # 比較 rewritten query（只比是否觸發，不比內容——內容需人工判斷）
        a_rewrite = a.get("rewritten", a.get("input", ""))
        b_rewrite = b.get("rewritten", b.get("input", ""))
        a_triggered = a_rewrite != a.get("input", "")
        b_triggered = b_rewrite != b.get("input", "")
        if a_triggered != b_triggered:
            turn_diffs["rewrite_triggered"] = (a_triggered, b_triggered)
            print(f"  T{turn_num:<4} {'rewrite_triggered':<18} {str(a_triggered):<28} {str(b_triggered):<28}")

        if turn_diffs:
            diffs.append({"turn": turn_num, "diffs": turn_diffs})

    if not diffs:
        print(f"  （所有 Turn 的 intent / assessment / flags 完全一致）")

    # --- Summary ---
    print(f"\n{'─' * 75}")

    # 統計 retrieval 跳過次數
    a_skip = sum(1 for r in log_a if r.get("intent") in ("ANSWER_FROM_HISTORY", "CLARIFICATION", "OUT_OF_SCOPE"))
    b_skip = sum(1 for r in log_b if r.get("intent") in ("ANSWER_FROM_HISTORY", "CLARIFICATION", "OUT_OF_SCOPE"))

    print(f"  Retrieval 跳過次數:  🅰️ {a_skip}  vs  🅱️ {b_skip}")
    print(f"  行為差異 Turn 數:    {len(diffs)} / {len(turns)}")
    print(f"{'─' * 75}")
    print(f"  存檔:")
    print(f"    🅰️  {path_a}")
    print(f"    🅱️  {path_b}")
    print(f"{'=' * 75}")
    print()
    print(f"  ⚠️  以上是「行為差異」，品質判斷（rewrite 語意保留、答案正確性）需手動標註。")
    print(f"     → 使用 annotate_multiturn_session('{path_a or path_b}') 進行標註")
    print()

    return {
        "session_id": session_id,
        "config_a": {"label": label_a, **config_a},
        "config_b": {"label": label_b, **config_b},
        "result_a": result_a,
        "result_b": result_b,
        "diff": diffs,
    }

In [ ]:
from copy import deepcopy

# ==============================================================
# 🚀 執行多輪 A/B 測試：Session A, B, C
# ==============================================================

# 1. 準備通用的 A/B 配置
config_baseline = {"intent_mode": "off", "label": "Baseline（無意圖分流）"}
config_intent = {"intent_mode": "llm", "label": "LLM 意圖分流"}

# ----------------------------------------------------------
# ⚖️ Session A：代名詞消解 + 條件延伸
# ----------------------------------------------------------
ab_result_a = ab_test_multiturn(
    session_id="A",
    session_name="代名詞消解 + 條件延伸",
    turns=[
        {
            "input": "某客戶連續三天分批存款，每次金額都在 49 萬左右，存完後隔天立即轉出。這筆有問題嗎？",
            "note": "建立歷史 / 應觸發 RF-01",
            "expected_intent": None,
        },
        {
            "input": "它的申報門檻是多少？",
            "note": "代名詞追問 → 預期跳過檢索",
            "expected_intent": "ANSWER_FROM_HISTORY",
        },
        {
            "input": "如果改用虛擬貨幣做類似的事，還算同一個紅旗嗎？",
            "note": "條件延伸 → 預期重啟檢索",
            "expected_intent": "RETRIEVE",
        },
    ],
    observe=[
        "[I] Turn 2 是否正確判斷為 ANSWER_FROM_HISTORY 以節省 token",
        "[R] Turn 3 的 rewrite 是否同時保留 structuring + 加入虛擬資產語意",
        "[Q] 改寫後的 query 夠不夠讓 retrieval 找到正確段落",
    ],
    config_a=config_baseline,
    config_b=config_intent,
    # 傳入全域變數
    faiss_index=faiss_index,
    chunks=chunks,
    embedding_model=embedding_model,
    bm25_index=bm25_index,
    tokenized_corpus=tokenized_corpus,
    llm_config=SELECTED_CONFIG,
    enable_gate=True,
)

# ----------------------------------------------------------
# ⚖️ Session B：Gate 在多輪中的攔截行為
# ----------------------------------------------------------
ab_result_b = ab_test_multiturn(
    session_id="B",
    session_name="Gate 攔截：話題跳離 AML",
    turns=[
        {
            "input": "某小型文創公司帳戶突然收到多筆大額款項，負責人說不清楚交易對手，資金進來後馬上轉出。",
            "note": "正常 AML 情境 / 應觸發 RF-06、RF-02",
            "expected_intent": None,
        },
        {
            "input": "FATF 對空殼公司的紅旗有哪些具體描述？",
            "note": "合理追問 / 不應被攔",
            "expected_intent": "RETRIEVE",
        },
        {
            "input": "順便問一下，台灣現在的央行利率是多少？",
            "note": "話題跳離 → 預期 Intent 直接判為 OUT_OF_SCOPE",
            "expected_intent": "OUT_OF_SCOPE",
        },
    ],
    observe=[
        "[I] Turn 3 是否被意圖分類器判定為 OUT_OF_SCOPE",
        "[G] Turn 1、2 是否沒有被誤攔（false REFUSE）",
        "[R] 觀察離題問題是否在進入 Rewrite 前就已被攔截",
    ],
    config_a=config_baseline,
    config_b=config_intent,
    # 傳入全域變數
    faiss_index=faiss_index,
    chunks=chunks,
    embedding_model=embedding_model,
    bm25_index=bm25_index,
    tokenized_corpus=tokenized_corpus,
    llm_config=SELECTED_CONFIG,
    enable_gate=True,
)

# ----------------------------------------------------------
# ⚖️ Session C：跨 RF 複合追問（壓力測試）
# ----------------------------------------------------------
ab_result_c = ab_test_multiturn(
    session_id="C",
    session_name="跨 RF 複合追問（壓力測試）",
    turns=[
        {
            "input": "某學生帳戶由叔叔代辦所有操作，交易對象多為陌生公司，資金入帳後立刻轉出，學生本人無法說明用途。",
            "note": "RF-04 人頭帳戶情境",
            "expected_intent": None,
        },
        {
            "input": "人頭帳戶跟第三方代辦，法規上怎麼區分？",
            "note": "細節釐清 → 預期判為 CLARIFICATION",
            "expected_intent": "CLARIFICATION",
        },
        {
            "input": "如果這個叔叔又用加密貨幣把錢轉走，這樣算幾個紅旗？",
            "note": "跨 RF 複合 → 預期判為 RETRIEVE",
            "expected_intent": "RETRIEVE",
        },
    ],
    observe=[
        "[I] Turn 2 是否判為 CLARIFICATION 並提供結構化解釋",
        "[R] Turn 3 能否同時帶入 RF-04 和虛擬資產兩個脈絡進行檢索",
        "[Q] 若 Turn 3 改寫品質下降 → 記錄為系統限制",
    ],
    config_a=config_baseline,
    config_b=config_intent,
    # 傳入全域變數
    faiss_index=faiss_index,
    chunks=chunks,
    embedding_model=embedding_model,
    bm25_index=bm25_index,
    tokenized_corpus=tokenized_corpus,
    llm_config=SELECTED_CONFIG,
    enable_gate=True,
)

print("\n✅ 所有 A/B 測試執行完畢，請查看上方差異報告。")


⚖️  Multi-Turn A/B Test: Session A
    Baseline（無意圖分流）  vs  LLM 意圖分流
    Session: 代名詞消解 + 條件延伸

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  🅰️  Baseline（無意圖分流）
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

🧪 Session A_A：代名詞消解 + 條件延伸 [Baseline（無意圖分流）]
   模式：off | 模型：llama-3.1-8b-instant
   Gate：ON
──────────────────────────────────────────────────────────────────────
   觀察點：
   → [I] Turn 2 是否正確判斷為 ANSWER_FROM_HISTORY 以節省 token
   → [R] Turn 3 的 rewrite 是否同時保留 structuring + 加入虛擬資產語意
   → [Q] 改寫後的 query 夠不夠讓 retrieval 找到正確段落

──────────────────────────────────────────────────────────────────────
   Turn 1  【建立歷史 / 應觸發 RF-01】
   輸入：某客戶連續三天分批存款，每次金額都在 49 萬左右，存完後隔天立即轉出。這筆有問題嗎？
──────────────────────────────────────────────────────────────────────

💬 Multi-Turn Chat (v4: off mode)
   輸入：某客戶連續三天分批存款，每次金額都在 49 萬左右，存完後隔天立即轉出。這筆有問題嗎？
   歷史：0 輪
   🧭 路由決策: TurnIntent.RETRIEVE
   🚀 執行路徑: RETRIEVE (完整 Pipeline)
   📝 Query Rewrite: 跳過（

Building prefix dict from the default dictionary ...
DEBUG:jieba:Building prefix dict from the default dictionary ...
Dumping model to file cache /tmp/jieba.cache
DEBUG:jieba:Dumping model to file cache /tmp/jieba.cache
Loading model cost 1.253 seconds.
DEBUG:jieba:Loading model cost 1.253 seconds.
Prefix dict has been built successfully.
DEBUG:jieba:Prefix dict has been built successfully.


   ✅ 歷史更新完成 (Intent: RETRIEVE)

──────────────────────────────────────────────────────────────────────
   Turn 2  【代名詞追問 → 預期跳過檢索】
   輸入：它的申報門檻是多少？
──────────────────────────────────────────────────────────────────────

💬 Multi-Turn Chat (v4: off mode)
   輸入：它的申報門檻是多少？
   歷史：1 輪
   🧭 路由決策: TurnIntent.RETRIEVE
   🚀 執行路徑: RETRIEVE (完整 Pipeline)
   📝 Query Rewrite: 已改寫
      原句：它的申報門檻是多少？
      改寫：台灣洗錢防制法規定的現金交易申報門檻金額是多少？
✅ Gate: ALLOW — Covered topics: ['cash_structuring']
   ✅ 歷史更新完成 (Intent: RETRIEVE)

──────────────────────────────────────────────────────────────────────
   Turn 3  【條件延伸 → 預期重啟檢索】
   輸入：如果改用虛擬貨幣做類似的事，還算同一個紅旗嗎？
──────────────────────────────────────────────────────────────────────

💬 Multi-Turn Chat (v4: off mode)
   輸入：如果改用虛擬貨幣做類似的事，還算同一個紅旗嗎？
   歷史：2 輪
   🧭 路由決策: TurnIntent.RETRIEVE
   🚀 執行路徑: RETRIEVE (完整 Pipeline)
   📝 Query Rewrite: 已改寫
      原句：如果改用虛擬貨幣做類似的事，還算同一個紅旗嗎？
      改寫：虛擬貨幣類似洗錢行為是否屬於同一紅旗指標？
✅ Gate: ALLOW — Covered topics: []
   ✅ 歷史更新完成 (Intent: RETRIEVE)


### 6.5: Multi-Turn Session Logger

與既有儲存模式的關係：
  - E2E tests → ExperimentLogger → experiments/runs/{version}_{date}_{type}/
  - Retrieval eval → eval/{test_set}_annotated.json + _eval_result.json
  - Multi-turn → multiturn/{session_id}_{date}_{model}.json  ← 這個

設計選擇：
  多輪不用 ExperimentLogger，因為：
  1. ExperimentLogger 是 one-case-one-file，多輪是 one-session-one-file
  2. 多輪的 "case" 是一個 Session（含多個 Turn），不是單一查詢
  3. 多輪需要記錄 Turn 之間的依賴關係（rewrite 品質受前一輪影響）

  所以用獨立的 logger，但遵循相同的慣例：
  - JSON 格式
  - 存到 Google Drive
  - config / turns / summary 三層結構

In [ ]:
MULTITURN_LOG_DIR = "/content/drive/MyDrive/AML/eval/multiturn"

annotate_multiturn_session() — 手動標註回寫

In [ ]:
def annotate_multiturn_session(
    json_path: str,
    annotations: list,
    save_mode: str = "overwrite",
) -> str:
    """
    載入已存的 multiturn session JSON，回寫人工標註，存檔。

    設計原則：
      - 自動存的是「骨架」（系統輸出）
      - 人工標註是「血肉」（語意判斷）
      - 兩者分離、事後合併，不破壞原始資料

    Args:
        json_path: 要標註的 JSON 檔案路徑
        annotations: 每個 turn 的標註 dict 組成的 list，格式：
            [
                {   # Turn 1
                    "rewrite_verdict": "SKIP",
                    "rewrite_notes": "Turn 1 無歷史，不觸發 rewrite",
                },
                {   # Turn 2
                    "rewrite_verdict": "PASS",
                    "rewrite_notes": "代名詞消解成功，domain anchoring 正確",
                    "expected_rewrite": "台灣洗錢防制法規定的現金交易申報門檻金額是多少？",
                    "expected_gate": "ALLOW",
                    "gate_covered_topics": ["cash_structuring"],
                    "intent_notes": "正確判斷為歷史可答",
                },
                {   # Turn 3 — 不需要標註的 turn 傳 None 或 {}
                    "rewrite_verdict": "DEGRADED",
                    "rewrite_notes": "structuring 語意流失...",
                },
            ]
        save_mode:
            "overwrite" — 直接覆蓋原檔（適合反覆修改）
            "copy"      — 存為 _annotated.json 副本（保留原始自動存檔）

    Returns:
        str: 存檔路徑

    使用範例：
        annotate_multiturn_session(
            json_path="/content/drive/MyDrive/AML/eval/multiturn/session_A_20250317_llama3.json",
            annotations=[
                {"rewrite_verdict": "SKIP", "rewrite_notes": "Turn 1 無歷史"},
                {
                    "rewrite_verdict": "PASS",
                    "rewrite_notes": "代名詞「它」成功消解",
                    "expected_rewrite": "台灣洗錢防制法規定的現金交易申報門檻金額是多少？",
                },
                {
                    "rewrite_verdict": "DEGRADED",
                    "rewrite_notes": "structuring 語意在改寫中流失",
                    "expected_rewrite": "使用虛擬貨幣進行類似分批存款（structuring）的行為，是否仍屬於同一個 AML 紅旗指標？",
                },
            ],
        )
    """
    # --- 載入 ---
    with open(json_path, "r", encoding="utf-8") as f:
        session = json.load(f)

    turns = session.get("turns", [])

    if len(annotations) > len(turns):
        print(f"⚠️  annotations ({len(annotations)}) 多於 turns ({len(turns)})，多的會被忽略")

    # --- 合併標註 ---
    annotated_count = 0
    for i, anno in enumerate(annotations):
        if i >= len(turns):
            break
        if not anno:  # None 或 {} → 跳過
            continue

        turn = turns[i]
        annotated_count += 1

        # rewrite 區塊
        if "rewrite_verdict" in anno:
            turn.setdefault("rewrite", {})["verdict"] = anno["rewrite_verdict"]
        if "rewrite_notes" in anno:
            turn.setdefault("rewrite", {})["notes"] = anno["rewrite_notes"]
        if "expected_rewrite" in anno:
            turn.setdefault("rewrite", {})["expected"] = anno["expected_rewrite"]

        # gate 區塊
        if "expected_gate" in anno:
            turn.setdefault("gate", {})["expected"] = anno["expected_gate"]
        if "gate_covered_topics" in anno:
            turn.setdefault("gate", {})["covered_topics"] = anno["gate_covered_topics"]

        # intent 區塊
        if "intent_notes" in anno:
            turn.setdefault("intent", {})["notes"] = anno["intent_notes"]
        if "intent_verdict" in anno:
            turn.setdefault("intent", {})["verdict"] = anno["intent_verdict"]

        # _flat 也同步更新（給 summary 重算用）
        flat = turn.get("_flat", {})
        if "rewrite_verdict" in anno:
            flat["rewrite_verdict"] = anno["rewrite_verdict"]
        if "expected_gate" in anno:
            flat["expected_gate"] = anno["expected_gate"]

    # --- 標記為已標註 ---
    session["_annotation"] = {
        "annotated_at": datetime.now().isoformat(),
        "turns_annotated": annotated_count,
        "total_turns": len(turns),
    }

    # --- 重算 summary（因為 rewrite_verdict 等已更新）---
    flat_list = [t.get("_flat", {}) for t in turns]

    rewrite_pass = sum(1 for t in turns if t.get("rewrite", {}).get("verdict") == "PASS")
    rewrite_degraded = sum(1 for t in turns if t.get("rewrite", {}).get("verdict") == "DEGRADED")

    session["summary"]["rewrite_pass"] = rewrite_pass
    session["summary"]["rewrite_degraded"] = rewrite_degraded

    gate_expected_refuse = sum(1 for f in flat_list if f.get("expected_gate") == "REFUSE")
    session["summary"]["gate_expected_refuse"] = gate_expected_refuse

    # --- 存檔 ---
    if save_mode == "copy":
        base, ext = os.path.splitext(json_path)
        out_path = f"{base}_annotated{ext}"
    else:
        out_path = json_path

    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(session, f, ensure_ascii=False, indent=2)

    print(f"✅ 標註完成: {annotated_count}/{len(turns)} turns")
    print(f"   存檔: {out_path}")

    return out_path

batch_annotate_turns() — 互動式逐 turn 標註

In [ ]:
def batch_annotate_turns(json_path: str, save_mode: str = "overwrite") -> str:
    """
    載入 session JSON，逐 turn 印出系統輸出，讓你在 Colab 裡直接 input() 標註。

    適合場景：剛跑完測試，想邊看螢幕輸出邊標註。
    如果你已經整理好標註 dict，直接用 annotate_multiturn_session() 更快。

    互動流程（每個 Turn）：
      1. 印出 user_input + rewritten_query + assessment + flags
      2. 問你 rewrite_verdict (PASS/DEGRADED/SKIP/Enter跳過)
      3. 問你 rewrite_notes (Enter跳過)
      4. 問你 expected_rewrite (Enter跳過)

    Returns:
        str: 存檔路徑
    """
    with open(json_path, "r", encoding="utf-8") as f:
        session = json.load(f)

    turns = session.get("turns", [])
    annotations = []

    print(f"\n{'=' * 60}")
    print(f"📝 標註 Session: {session.get('session_id', '?')} — {session.get('purpose', '?')}")
    print(f"   共 {len(turns)} turns，Enter 跳過 = 不標註")
    print(f"{'=' * 60}")

    for i, turn in enumerate(turns):
        print(f"\n{'─' * 60}")
        print(f"  Turn {turn.get('turn', i+1)}")
        print(f"  輸入:   {turn.get('user_input', '?')}")

        rewrite = turn.get("rewrite", {})
        print(f"  改寫:   {rewrite.get('rewritten', '—')}")

        intent = turn.get("intent", {})
        print(f"  Intent: {intent.get('classified', '—')}")

        llm = turn.get("llm_output", {})
        print(f"  判定:   {llm.get('assessment', '—')}")
        print(f"  Flags:  {llm.get('flags', [])}")
        print(f"{'─' * 60}")

        anno = {}

        # rewrite verdict
        v = input("  rewrite_verdict (PASS/DEGRADED/SKIP) [Enter=跳過]: ").strip().upper()
        if v in ("PASS", "DEGRADED", "SKIP"):
            anno["rewrite_verdict"] = v

        # rewrite notes
        notes = input("  rewrite_notes [Enter=跳過]: ").strip()
        if notes:
            anno["rewrite_notes"] = notes

        # expected rewrite
        exp = input("  expected_rewrite [Enter=跳過]: ").strip()
        if exp:
            anno["expected_rewrite"] = exp

        # intent notes (只在 intent 不是 N/A 時問)
        if intent.get("classified") not in (None, "N/A"):
            inote = input("  intent_notes [Enter=跳過]: ").strip()
            if inote:
                anno["intent_notes"] = inote

        annotations.append(anno if anno else None)

    # 呼叫 annotate 寫回
    return annotate_multiturn_session(json_path, annotations, save_mode=save_mode)

In [ ]:
# ✅ Print
# ==============================================================
print("✅ PART 6.6: Multi-Turn A/B Test + Annotation 已載入")
print("   ab_test_multiturn()            — 同組 session 跑兩個 config")
print("   annotate_multiturn_session()   — dict 式批次標註")
print("   batch_annotate_turns()         — 互動式逐 turn 標註")

✅ PART 6.6: Multi-Turn A/B Test + Annotation 已載入
   ab_test_multiturn()            — 同組 session 跑兩個 config
   annotate_multiturn_session()   — dict 式批次標註
   batch_annotate_turns()         — 互動式逐 turn 標註


In [ ]:
# # ============================================================
# # 📋 Multi-Turn 測試記錄：Session A / B / C（第二次執行）
# # ============================================================
# # 測試日期：2025-03
# # 模型：llama-3.1-8b-instant
# # Gate：ON（balanced_v3）
# # SemanticScopeClassifier：已插入 Rule 4（fallback-only）
# # 涵蓋面向：代名詞消解、話題跳離攔截、跨 RF 複合追問
# # ============================================================


# # ============================================================
# # 🧪 Session A：代名詞消解 + 條件延伸
# # ============================================================

# session_a_turns = [

#     log_turn(
#         turn_number=1,
#         user_input="某客戶連續三天分批存款，每次金額都在 49 萬左右，存完後隔天立即轉出。這筆有問題嗎？",
#         expected_behavior="建立歷史 / 應觸發 RF-01",
#         # rewrite
#         rewrite_triggered=False,
#         rewrite_verdict="SKIP",
#         rewrite_notes="Turn 1 無歷史，不觸發 rewrite",
#         # gate
#         expected_gate="ALLOW",
#         actual_gate="ALLOW",
#         gate_covered_topics=["rapid_movement"],
#         # retrieval
#         retrieved_chunks=[
#             {"rank": 1, "doc_category": "knowledge_bridge", "source": "TW_Gov", "page": 8,  "score": 0.025},
#             {"rank": 2, "doc_category": "core",             "source": "FATF",   "page": 8,  "score": 0.016},
#             {"rank": 3, "doc_category": "core",             "source": "FATF",   "page": 8,  "score": 0.015},
#             {"rank": 4, "doc_category": "sector_specific",  "source": "FATF",   "page": 7,  "score": 0.015},
#             {"rank": 5, "doc_category": "sector_specific",  "source": "FATF",   "page": 11, "score": 0.015},
#         ],
#         # LLM
#         assessment="confirmed",
#         flags=["RF-01"],
#     ),

#     log_turn(
#         turn_number=2,
#         user_input="它的申報門檻是多少？",
#         expected_behavior="代名詞追問 → 應觸發 rewrite，將「它」還原為具體 AML 脈絡",
#         # rewrite
#         rewrite_triggered=True,
#         original_query="它的申報門檻是多少？",
#         rewritten_query="台灣洗錢防制法規定的現金交易申報門檻金額是多少？",
#         expected_rewrite="台灣洗錢防制法規定的現金交易申報門檻金額是多少？",
#         rewrite_verdict="PASS",
#         rewrite_notes=(
#             "代名詞「它」成功消解。"
#             "domain anchoring 正確補入台灣洗錢防制法脈絡，查詢意圖保留完整。"
#             "TW_Gov 文件上升至前兩名，顯示改寫有效改善 retrieval 精準度。"
#         ),
#         # gate
#         expected_gate="ALLOW",
#         actual_gate="ALLOW",
#         gate_covered_topics=["cash_structuring"],
#         # retrieval
#         retrieved_chunks=[
#             {"rank": 1, "doc_category": "knowledge_bridge", "source": "TW_Gov", "page": 8,  "score": 0.026},
#             {"rank": 2, "doc_category": "knowledge_bridge", "source": "TW_Gov", "page": 7,  "score": 0.026},
#             {"rank": 3, "doc_category": "core",             "source": "FATF",   "page": 1,  "score": 0.016},
#             {"rank": 4, "doc_category": "sector_specific",  "source": "FATF",   "page": 15, "score": 0.015},
#             {"rank": 5, "doc_category": "sector_specific",  "source": "FATF",   "page": 11, "score": 0.014},
#         ],
#         # LLM
#         assessment="confirmed",
#         flags=["RF-01"],
#     ),

#     log_turn(
#         turn_number=3,
#         user_input="如果改用虛擬貨幣做類似的事，還算同一個紅旗嗎？",
#         expected_behavior="條件延伸 → rewrite 應同時保留 structuring + 補入虛擬資產語意",
#         # rewrite
#         rewrite_triggered=True,
#         original_query="如果改用虛擬貨幣做類似的事，還算同一個紅旗嗎？",
#         rewritten_query="虛擬貨幣類似洗錢行為是否屬於同一紅旗指標？",
#         expected_rewrite="使用虛擬貨幣進行類似分批存款（structuring）的行為，是否仍屬於同一個 AML 紅旗指標？",
#         rewrite_verdict="DEGRADED",
#         rewrite_notes=(
#             "entity: structuring 脈絡被丟失，「類似的事」未被還原為具體行為描述。"
#             "intent: 從比較型（還算同一個紅旗嗎）退化為判斷型（是否屬於同一指標），語意仍部分保留。"
#             "anchoring: 虛擬貨幣有補入，但 structuring 關鍵詞消失導致 retrieval 轉向 sector_specific FATF，"
#             "未能拉回 TW_Gov 台灣法規脈絡。"
#             "記錄為 limitation，非 bug：歷史越長改寫難度越高，此為已知多輪 context degradation 問題。"
#         ),
#         # gate
#         expected_gate="ALLOW",
#         actual_gate="ALLOW",
#         gate_covered_topics=[],  # 改寫後未觸發任何 covered topic，gate 以空集合 ALLOW
#         # retrieval
#         retrieved_chunks=[
#             {"rank": 1, "doc_category": "sector_specific", "source": "FATF", "page": 19, "score": 0.015},
#             {"rank": 2, "doc_category": "sector_specific", "source": "FATF", "page": 12, "score": 0.015},
#             {"rank": 3, "doc_category": "sector_specific", "source": "FATF", "page": 5,  "score": 0.014},
#             {"rank": 4, "doc_category": "sector_specific", "source": "FATF", "page": 17, "score": 0.014},
#             {"rank": 5, "doc_category": "sector_specific", "source": "FATF", "page": 10, "score": 0.014},
#         ],
#         # LLM
#         assessment="unlikely",
#         flags=[],
#     ),

# ]

# log_multiturn_session(
#     session_id="A",
#     session_purpose="代名詞消解 + 條件延伸",
#     turns=session_a_turns,
#     llm_config=SELECTED_CONFIG,
#     enable_gate=True,
#     note=(
#         "Turn 2 rewrite PASS：代名詞消解正確，TW_Gov 文件成功被檢索到前兩名，"
#         "相較 Turn 1 的 retrieval 分佈有明顯改善。"
#         "Turn 3 rewrite DEGRADED：structuring 語意在改寫中流失，"
#         "為多輪 context degradation 的典型案例，記錄為系統限制。"
#         "assessment=unlikely 的原因尚未確認是 retrieval 偏移還是 LLM 本身的判斷。"
#     ),
# )


# # ============================================================
# # 🧪 Session B：Gate 攔截：話題跳離 AML
# # ============================================================

# session_b_turns = [

#     log_turn(
#         turn_number=1,
#         user_input="某小型文創公司帳戶突然收到多筆大額款項，負責人說不清楚交易對手，資金進來後馬上轉出。",
#         expected_behavior="正常 AML 情境 / 應觸發 RF-06、RF-02，不應被 Gate 攔截",
#         # rewrite
#         rewrite_triggered=False,
#         rewrite_verdict="SKIP",
#         rewrite_notes="Turn 1 無歷史，不觸發 rewrite",
#         # gate
#         expected_gate="ALLOW",
#         actual_gate="ALLOW",
#         gate_covered_topics=["rapid_movement", "cash_structuring"],
#         # retrieval
#         retrieved_chunks=[
#             {"rank": 1, "doc_category": "core",            "source": "FATF", "page": 8,  "score": 0.016},
#             {"rank": 2, "doc_category": "sector_specific", "source": "FATF", "page": 10, "score": 0.015},
#             {"rank": 3, "doc_category": "sector_specific", "source": "FATF", "page": 15, "score": 0.015},
#             {"rank": 4, "doc_category": "sector_specific", "source": "FATF", "page": 17, "score": 0.014},
#             {"rank": 5, "doc_category": "sector_specific", "source": "FATF", "page": 18, "score": 0.014},
#         ],
#         # LLM — 觸發 RF-02, RF-06，與預期一致
#         assessment="confirmed",
#         flags=["RF-02", "RF-06"],
#     ),

#     log_turn(
#         turn_number=2,
#         user_input="FATF 對空殼公司的紅旗有哪些具體描述？",
#         expected_behavior="合理追問 / 不應被 Gate 攔截",
#         # rewrite
#         rewrite_triggered=True,
#         original_query="FATF 對空殼公司的紅旗有哪些具體描述？",
#         rewritten_query="FATF 對空殼公司的紅旗有哪些具體描述？",  # 實際使用的 query（前綴已被系統截取）
#         expected_rewrite="（獨立完整查詢，理想應為 SKIP）",
#         rewrite_verdict="NOISE",
#         rewrite_notes=(
#             "rewrite 被觸發但輸出含格式雜訊：「改寫後的查詢句：FATF 對空殼公司的紅旗有哪些具體描述？」。"
#             "語意無改變，前綴字串「改寫後的查詢句：」是 rewrite_query() prompt 輸出格式控制不穩定的已知問題。"
#             "實際傳入 retrieval 的 query 為原句，功能上未造成影響，但輸出格式需修正。"
#             "建議修正方向：在 rewrite_query() 加入後處理，過濾常見格式前綴字串。"
#         ),
#         # gate
#         expected_gate="ALLOW",
#         actual_gate="ALLOW",
#         gate_covered_topics=["shell_company"],
#         # retrieval
#         retrieved_chunks=[
#             {"rank": 1, "doc_category": "sector_specific", "source": "FATF", "page": 2,  "score": 0.015},
#             {"rank": 2, "doc_category": "sector_specific", "source": "FATF", "page": 6,  "score": 0.015},
#             {"rank": 3, "doc_category": "sector_specific", "source": "FATF", "page": 24, "score": 0.014},
#             {"rank": 4, "doc_category": "sector_specific", "source": "FATF", "page": 21, "score": 0.014},
#             {"rank": 5, "doc_category": "sector_specific", "source": "FATF", "page": 3,  "score": 0.014},
#         ],
#         # LLM
#         assessment="possible",
#         flags=["RF-08"],
#     ),

#     log_turn(
#         turn_number=3,
#         user_input="順便問一下，台灣現在的央行利率是多少？",
#         expected_behavior="話題跳離 AML → 應觸發 Gate REFUSE",
#         # rewrite
#         rewrite_triggered=True,
#         original_query="順便問一下，台灣現在的央行利率是多少？",
#         rewritten_query="台灣現在的央行利率是多少？",
#         expected_rewrite="（任何改寫結果均不應改變話題性質，Gate 應仍攔截）",
#         rewrite_verdict="NEUTRAL",
#         rewrite_notes=(
#             "改寫僅移除「順便問一下」前綴，語意未變，屬正常輸出。"
#             "核心問題：rewrite 後 Gate 仍未攔截（actual_gate=ALLOW），為 Gate false ALLOW。"
#             "原因分析：TopicDetector 以關鍵字比對為主；「央行利率」不在 covered / not_covered topic "
#             "關鍵字集合內，導致以空集合通過 balanced_v3 邏輯。"
#             "balanced_v3 在此場景下退化為 no-op：covered=[] 且 not_covered=[] 時直接 ALLOW。"
#             "SemanticScopeClassifier（Rule 4）為此類 false ALLOW 的架構回應，"
#             "本次測試確認其必要性。"
#         ),
#         # gate
#         expected_gate="REFUSE",
#         actual_gate="ALLOW",   # ← Gate miss，為本 session 的核心觀察點
#         gate_covered_topics=[],
#         # retrieval — Gate 未攔截，流入 retrieval（以 AML 文件硬回答利率問題）
#         retrieved_chunks=[
#             {"rank": 1, "doc_category": "sector_specific", "source": "FATF", "page": 15, "score": 0.015},
#             {"rank": 2, "doc_category": "sector_specific", "source": "FATF", "page": 15, "score": 0.015},
#             {"rank": 3, "doc_category": "sector_specific", "source": "FATF", "page": 11, "score": 0.014},
#             {"rank": 4, "doc_category": "sector_specific", "source": "FATF", "page": 20, "score": 0.014},
#             {"rank": 5, "doc_category": "sector_specific", "source": "FATF", "page": 10, "score": 0.014},
#         ],
#         # LLM — flags 繼承自歷史，非當前 turn 新觸發
#         assessment="unlikely",
#         flags=[],
#     ),

# ]

# log_multiturn_session(
#     session_id="B",
#     session_purpose="Gate 攔截：話題跳離 AML",
#     turns=session_b_turns,
#     llm_config=SELECTED_CONFIG,
#     enable_gate=True,
#     note=(
#         "Turn 1 無誤攔，flags 觸發正確（RF-02, RF-06）。"
#         "Turn 2 rewrite NOISE：格式雜訊問題確認，語意功能未受損，但需修正輸出格式控制。"
#         "Turn 3 Gate false ALLOW：「央行利率」不觸發任何 topic keyword，"
#         "TopicDetector 以空集合通過，為已知限制。"
#         "Session B 的核心實驗價值：確認 SemanticScopeClassifier 的架構必要性，"
#         "純 keyword gate 無法處理 vocabulary 外的 out-of-domain 查詢。"
#     ),
# )


# # ============================================================
# # 🧪 Session C：跨 RF 複合追問（壓力測試）
# # ============================================================

# session_c_turns = [

#     log_turn(
#         turn_number=1,
#         user_input="某學生帳戶由叔叔代辦所有操作，交易對象多為陌生公司，資金入帳後立刻轉出，學生本人無法說明用途。",
#         expected_behavior="RF-04 人頭帳戶情境 / 應觸發 RF-04，Gate 不攔截",
#         # rewrite
#         rewrite_triggered=False,
#         rewrite_verdict="SKIP",
#         rewrite_notes="Turn 1 無歷史，不觸發 rewrite",
#         # gate
#         expected_gate="ALLOW",
#         actual_gate="ALLOW",
#         gate_covered_topics=["third_party", "rapid_movement", "identity_mismatch"],
#         # retrieval
#         retrieved_chunks=[
#             {"rank": 1, "doc_category": "knowledge_bridge", "source": "TW_Gov", "page": 6,  "score": 0.024},
#             {"rank": 2, "doc_category": "core",             "source": "FATF",   "page": 8,  "score": 0.016},
#             {"rank": 3, "doc_category": "core",             "source": "FATF",   "page": 8,  "score": 0.015},
#             {"rank": 4, "doc_category": "sector_specific",  "source": "FATF",   "page": 10, "score": 0.015},
#             {"rank": 5, "doc_category": "sector_specific",  "source": "FATF",   "page": 15, "score": 0.014},
#         ],
#         # LLM
#         assessment="confirmed",
#         flags=["RF-04", "RF-02"],
#     ),

#     log_turn(
#         turn_number=2,
#         user_input="人頭帳戶跟第三方代辦，法規上怎麼區分？",
#         expected_behavior="RF-04 細節追問 → rewrite 應保留人頭帳戶語意",
#         # rewrite
#         rewrite_triggered=True,
#         original_query="人頭帳戶跟第三方代辦，法規上怎麼區分？",
#         rewritten_query="人頭帳戶和第三方代辦在法律規範上的區分是什麼？",  # 系統截取後半段使用
#         expected_rewrite="在洗錢防制法規下，人頭帳戶與合法第三方代辦帳戶的區分標準是什麼？",
#         rewrite_verdict="PARTIAL",
#         rewrite_notes=(
#             "輸出有格式雜訊：原句與改寫句串接輸出（「人頭帳戶跟第三方代辦，法規上怎麼區分？改寫：…」），"
#             "系統實際使用串接後的後半段作為 query，功能上未失效。"
#             "語意核心保留（人頭帳戶 / 第三方代辦 / 法律規範區分）。"
#             "改寫效果正面：TW_Gov 文件上升至前兩名（p.6, p.3），顯示 domain anchoring 改善了 retrieval。"
#             "未補入 AML 或洗錢防制法等 domain anchoring 詞彙，為輕微 limitation。"
#             "格式雜訊問題同 Session B Turn 2，確認為系統性 rewrite prompt 輸出控制問題。"
#         ),
#         # gate
#         expected_gate="ALLOW",
#         actual_gate="ALLOW",
#         gate_covered_topics=["third_party"],
#         # retrieval — TW_Gov 上升，顯示改寫有效
#         retrieved_chunks=[
#             {"rank": 1, "doc_category": "knowledge_bridge", "source": "TW_Gov", "page": 6,  "score": 0.026},
#             {"rank": 2, "doc_category": "knowledge_bridge", "source": "TW_Gov", "page": 3,  "score": 0.024},
#             {"rank": 3, "doc_category": "sector_specific",  "source": "FATF",   "page": 14, "score": 0.015},
#             {"rank": 4, "doc_category": "core",             "source": "FATF",   "page": 8,  "score": 0.015},
#             {"rank": 5, "doc_category": "sector_specific",  "source": "FATF",   "page": 11, "score": 0.015},
#         ],
#         # LLM — flags 未新增，維持 RF-04, RF-02（本 turn 為定義釐清，非新情境觸發）
#         assessment="confirmed",
#         flags=[],
#     ),

#     log_turn(
#         turn_number=3,
#         user_input="如果這個叔叔又用加密貨幣把錢轉走，這樣算幾個紅旗？",
#         expected_behavior="跨 RF 複合 → rewrite 應同時帶入 RF-04（第三方代辦）+ 虛擬資產兩個脈絡",
#         # rewrite
#         rewrite_triggered=True,
#         original_query="如果這個叔叔又用加密貨幣把錢轉走，這樣算幾個紅旗？",
#         rewritten_query="如果第三方代辦者使用加密貨幣將錢轉走，該行為涉及哪些紅旗指標？",
#         expected_rewrite="若第三方代辦帳戶（涉及人頭帳戶疑慮）再以加密貨幣轉移資金，依 AML 規範可觸發哪些紅旗指標？",
#         rewrite_verdict="PASS",
#         rewrite_notes=(
#             "第三方代辦脈絡（RF-04）成功保留並轉化為通用語意。"
#             "加密貨幣語意補入正確，gate 觸發 virtual_assets + third_party 雙 topic。"
#             "輕微 limitation：「人頭帳戶」具體語意略微抽象化為「第三方代辦者」，"
#             "但未造成 retrieval 偏移；TW_Gov p.4 仍出現在 rank 1。"
#             "LLM 成功新增 RF-07，表明多脈絡 retrieval 有效傳遞歷史語意。"
#             "本 turn 為 Session C 壓力測試的核心驗證點：2-turn 歷史下跨 RF 組合查詢可正常運作。"
#         ),
#         # gate
#         expected_gate="ALLOW",
#         actual_gate="ALLOW",
#         gate_covered_topics=["virtual_assets", "third_party"],
#         # retrieval
#         retrieved_chunks=[
#             {"rank": 1, "doc_category": "knowledge_bridge", "source": "TW_Gov", "page": 4,  "score": 0.024},
#             {"rank": 2, "doc_category": "sector_specific",  "source": "FATF",   "page": 12, "score": 0.015},
#             {"rank": 3, "doc_category": "sector_specific",  "source": "FATF",   "page": 14, "score": 0.015},
#             {"rank": 4, "doc_category": "sector_specific",  "source": "FATF",   "page": 7,  "score": 0.014},
#             {"rank": 5, "doc_category": "sector_specific",  "source": "FATF",   "page": 11, "score": 0.014},
#         ],
#         # LLM
#         assessment="confirmed",
#         flags=["RF-07", "RF-02"],
#     ),

# ]

# log_multiturn_session(
#     session_id="C",
#     session_purpose="跨 RF 複合追問（壓力測試）",
#     turns=session_c_turns,
#     llm_config=SELECTED_CONFIG,
#     enable_gate=True,
#     note=(
#         "Turn 2 rewrite PARTIAL：格式雜訊再次出現（原句 + 改寫句串接），"
#         "確認為系統性問題，需在 rewrite_query() 加後處理修正。"
#         "Turn 2 retrieval 正面結果：TW_Gov 上升至前兩名，改寫對 retrieval 有實質改善。"
#         "Turn 3 rewrite PASS：跨脈絡組合成功，gate 雙 topic 觸發，RF-07 新觸發。"
#         "整體壓力測試結論：2-turn 歷史下跨 RF 複合查詢可正常運作；"
#         "更深歷史的 context degradation 需進一步設計實驗觀察。"
#     ),
# )

In [ ]:
print("✅ PART 6.5: Multi-Turn Session Logger 已載入")
print(f"   儲存路徑: {MULTITURN_LOG_DIR}")

✅ PART 6.5: Multi-Turn Session Logger 已載入
   儲存路徑: /content/drive/MyDrive/AML/eval/multiturn


In [ ]:
# ==============================================================
# 📝 手動標註：將工程判斷寫回 JSON
# ==============================================================

# --- 標註 Session A (聰明版) ---
annotate_multiturn_session(
    json_path="/content/drive/MyDrive/AML/eval/multiturn/session_A_B_20260317_llama_3.1_8b_instant_llm.json",
    annotations=[
        {"rewrite_verdict": "SKIP", "rewrite_notes": "第一輪無歷史，正常跳過"},
        {
            "rewrite_verdict": "PASS",
            "rewrite_notes": "「它」正確消解為「大額現金交易」，且判斷由歷史回答正確。",
            "intent_notes": "節省了一次檢索費用"
        },
        {
            "rewrite_verdict": "PASS",
            "rewrite_notes": "成功將「虛擬貨幣」與前文的「分批存款」結合進行檢索。"
        }
    ],
    save_mode="overwrite"
)

# --- 標註 Session B (Gate 攔截測試) ---
annotate_multiturn_session(
    json_path="/content/drive/MyDrive/AML/eval/multiturn/session_B_B_20260317_llama_3.1_8b_instant_llm.json",
    annotations=[
        {"rewrite_verdict": "SKIP", "rewrite_notes": "正常"},
        {"rewrite_verdict": "PASS", "rewrite_notes": "專業問題正常放行"},
        {
            "rewrite_verdict": "PASS",
            "rewrite_notes": "正確攔截非 AML 領域問題（央行利率）。",
            "intent_notes": "Gate 運作符合安全規格"
        }
    ],
    save_mode="overwrite"
)

# --- 標註 Session C (複合壓力測試) ---
annotate_multiturn_session(
    json_path="/content/drive/MyDrive/AML/eval/multiturn/session_C_B_20260317_llama_3.1_8b_instant_llm.json",
    annotations=[
        {"rewrite_verdict": "SKIP", "rewrite_notes": "正常"},
        {
            "rewrite_verdict": "PASS",
            "rewrite_notes": "正確識別為法律概念釐清（CLARIFICATION）而非單純搜索。"
        },
        {
            "rewrite_verdict": "PASS",
            "rewrite_notes": "複雜脈絡（人頭+代理+加密貨幣）完整保留於 rewrite 中。"
        }
    ],
    save_mode="overwrite"
)

print("✅ 手動標註已完成！現在你的 JSON 檔案已經包含了『人類評分』。")

✅ 標註完成: 3/3 turns
   存檔: /content/drive/MyDrive/AML/eval/multiturn/session_A_B_20260317_llama_3.1_8b_instant_llm.json
✅ 標註完成: 3/3 turns
   存檔: /content/drive/MyDrive/AML/eval/multiturn/session_B_B_20260317_llama_3.1_8b_instant_llm.json
✅ 標註完成: 3/3 turns
   存檔: /content/drive/MyDrive/AML/eval/multiturn/session_C_B_20260317_llama_3.1_8b_instant_llm.json
✅ 手動標註已完成！現在你的 JSON 檔案已經包含了『人類評分』。
